# Annotated notebook: Phase 1 decoder (GQA) and the toolkits for Phases 2 to 7

This is the same notebook as the working one, with a short explanation under every code cell: what the cell does, what each part means, and why it was written that way. Read it top to bottom. Notes marked *hypothesis* are explanations of why a choice was made that we did not test directly.

How to replicate: run the cells in order. Each phase section starts by downloading its data and ends by writing the prediction file for that phase; later phases reuse the functions defined earlier (`RRModel`, `make_backbone`, `inject`, the toolkits).

## 1. Data Access Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [ ]:
import os
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from your Colab Secrets (make sure a secret named 'HF' is saved in your notebook settings)
HF_TOKEN = userdata.get('HF')
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful on Google Colab.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful on Google Colab.


**Code cell 1. What this cell does.** Reads the access token from Colab's secret store and logs in to the dataset host, so the course data files can be downloaded later.

**Line by line**
- `import os` / `from huggingface_hub import login` / `from google.colab import userdata`: `os` to set an environment variable, `login` to register the token with the download library, `userdata` to read Colab Secrets (so the token is never typed into the notebook).
- `HF_TOKEN = userdata.get('HF')`: fetches the secret named `HF`. It must exist in the notebook's Secrets panel with notebook access switched on, otherwise this raises an error.
- `os.environ["HF_TOKEN"] = HF_TOKEN`: makes the token visible to any library that looks for it in the environment.
- `login(token=HF_TOKEN)`: stores the token for this session. The later `hf_hub_download` calls pass the token explicitly as well.
- The `print` confirms the login ran.

**Why.** The Phase-1 dataset repository is gated, so every download needs this token. Keeping it in Colab Secrets means the notebook can be shared without leaking it.

**To replicate.** Create a read token on the dataset host, add it as a Colab secret called `HF`, and run this cell first.

## 2. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 1


**Code cell 2. What this cell does.** Checks that a GPU is attached to the Colab runtime.

**Line by line**
- `torch.cuda.is_available()` returns `True` if PyTorch can see a CUDA GPU.
- `torch.cuda.device_count()` prints how many GPUs are visible (1 on Colab).

**Why.** Training an 87M-parameter model on a CPU would take weeks. If this prints `False`, change the runtime type to a GPU (Runtime > Change runtime type) before continuing.

## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [ ]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Updated training files list (Supreme Court, Acts, and Tribunals)
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json"
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 1_500_000_000
VAL_TOKENS = 10_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 4096

# Model architecture configured to match the previous run (n_layer = 8)
N_EMBD = 768
N_LAYER = 8
N_HEAD = 12

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.6
TOP_P = 0.9

print("Configuration updated successfully.")

Configuration updated successfully.


**Code cell 3. What this cell does.** Holds the global settings for Phase 1 in one place: dataset location, training budget, block size and the first model shape.

**Line by line**
- The imports (`os, random, json, time, math, zipfile, shutil`) are the standard-library helpers used by later cells.
- `HF_DATASET_REPO`: the id of the course dataset on the data host.
- `TRAIN_FILES`: the three pretraining files (Supreme Court judgments, Acts, Tribunal decisions). Later cells use a subset of these (see the packing cell).
- `TEST_FILE`: the test prompts file used to produce the generation output.
- `TRAIN_TOKENS = 1_500_000_000` and `VAL_TOKENS = 10_000_000`: the planned token budgets for training and validation (the actual packed data is whatever the packing cell produced; these were the initial targets).
- `TRAIN_BLOCK_SIZE = 2048`: each training example is a block of 2,048 token ids.
- `EPOCHS = 1`, `SEED = 42`, `random.seed(SEED)`: initial epoch count and a fixed seed so shuffles are repeatable. The real training cell sets its own `EPOCHS`.
- `MODEL_CONTEXT_LENGTH = 4096`: the longest context the early design aimed for; the final GQA model was built with `max_seq_len=2048`.
- `N_EMBD = 768`, `N_LAYER = 8`, `N_HEAD = 12`: the first-draft model shape. The model cell later uses 10 layers.
- `TARGET_GENERATION_WORDS`, `MAX_GENERATION_TOKENS`, `TEMPERATURE`, `TOP_P`: sampling settings for text generation.

**Note.** These are starting values from the assignment template. Where a later cell sets its own value (layers, epochs), the later cell is the one that counts.

## 4. Load Tokenizer

### Tokenizer implementation (from scratch)

Byte-level BPE written by hand: word splitting, merge training, encoding and decoding. The cells below load the saved merge table with it.

In [ ]:
# ============================================================
# Byte-level BPE tokenizer, implemented from scratch
#   - pretokenize : split text into words with a regular expression, map every byte to a printable symbol
#   - train_bpe   : learn the merge table (counts word frequencies, then repeatedly merges the most frequent pair)
#   - ByteBPE     : encode / decode with a learned merge table
# ============================================================
import json, heapq, time
from collections import Counter, defaultdict
try:
    import regex as re          # needed for the Unicode classes \p{L} and \p{N} (installed on Colab)
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
except ImportError:
    import re
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
PAT = re.compile(PAT)

def bytes_to_unicode():
    """Map each of the 256 byte values to one printable character (so every token is a readable string)."""
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}
B2U = bytes_to_unicode()
U2B = {v: k for k, v in B2U.items()}
SPECIALS = ["<|endoftext|>", "[PAD]"]
SPECIAL_SPLIT = re.compile("(" + "|".join(re.escape(s) for s in SPECIALS) + ")")

def pretokenize(text):
    return ["".join(B2U[b] for b in w.encode("utf-8")) for w in PAT.findall(text)]

def train_bpe(texts, vocab_size=32000, min_frequency=2):
    wc = Counter()
    for t in texts:
        wc.update(pretokenize(t))
    alphabet = sorted({c for w in wc for c in w})
    vocab = {s: i for i, s in enumerate(SPECIALS)}
    for c in alphabet: vocab[c] = len(vocab)
    words = [[vocab[c] for c in w] for w in wc]
    freqs = list(wc.values())
    pair_cnt = defaultdict(int); where = defaultdict(set)
    for wi, w in enumerate(words):
        for p in zip(w, w[1:]):
            pair_cnt[p] += freqs[wi]; where[p].add(wi)
    heap = [(-c, p) for p, c in pair_cnt.items()]; heapq.heapify(heap)
    id2tok = {i: s for s, i in vocab.items()}; merges = []
    while len(vocab) < vocab_size and heap:
        negc, pair = heapq.heappop(heap)
        if pair_cnt.get(pair, 0) != -negc:
            if pair_cnt.get(pair, 0) > 0: heapq.heappush(heap, (-pair_cnt[pair], pair))
            continue
        if -negc < min_frequency: break
        a, b = pair; new_tok = id2tok[a] + id2tok[b]
        new_id = len(vocab); vocab[new_tok] = new_id; id2tok[new_id] = new_tok
        merges.append((id2tok[a], id2tok[b]))
        touched = set()
        for wi in list(where[pair]):
            w, f = words[wi], freqs[wi]
            for p in zip(w, w[1:]):
                pair_cnt[p] -= f
                if p != pair: touched.add(p)
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: out.append(new_id); i += 2
                else: out.append(w[i]); i += 1
            words[wi] = out
            for p in zip(out, out[1:]):
                pair_cnt[p] += f; where[p].add(wi); touched.add(p)
        pair_cnt[pair] = 0; del where[pair]
        for p in touched:
            if pair_cnt[p] > 0: heapq.heappush(heap, (-pair_cnt[p], p))
    return vocab, merges

class _Enc:                                   # small result object so callers can write  tokenizer.encode(text).ids
    def __init__(self, ids): self.ids = ids

class ByteBPE:
    def __init__(self, vocab, merges):
        self.vocab, self.id2tok = vocab, {i: t for t, i in vocab.items()}
        self.merges = [tuple(m) for m in merges]
        self.rank = {m: r for r, m in enumerate(self.merges)}; self.cache = {}
    def _parts(self, w):
        parts = list(w)
        while len(parts) > 1:
            best = min((self.rank.get(p, 1 << 30), i) for i, p in enumerate(zip(parts, parts[1:])))
            if best[0] == 1 << 30: break
            i = best[1]; parts[i:i + 2] = [parts[i] + parts[i + 1]]
        return parts
    def _bpe(self, w):
        if w not in self.cache:
            self.cache[w] = [self.vocab.get(p, 0) for p in self._parts(w)]   # a symbol never seen in training maps to id 0
        return self.cache[w]
    def get_vocab_size(self): return len(self.vocab)
    def token_to_id(self, t): return self.vocab.get(t)
    def encode_ids(self, text):
        ids = []
        for piece in SPECIAL_SPLIT.split(text):
            if piece in self.vocab and piece in SPECIALS: ids.append(self.vocab[piece])
            elif piece:
                for w in pretokenize(piece): ids += self._bpe(w)
        return ids
    def encode(self, text):
        """Like encode_ids, but also returns the character span (start, end) of every token in `text`.
        Leading and trailing spaces are trimmed from a token's span, so the span covers only visible characters."""
        ids, toks, offs = [], [], []
        pos = 0
        for piece in SPECIAL_SPLIT.split(text):
            if piece in SPECIALS:
                ids.append(self.vocab[piece]); toks.append(piece); offs.append((pos, pos + len(piece)))
            elif piece:
                for m in PAT.finditer(piece):
                    word = m.group(0); cstart = pos + m.start()
                    b2c = [k for k, ch in enumerate(word) for _ in ch.encode("utf-8")]   # byte index -> character index
                    w = "".join(B2U[b] for b in word.encode("utf-8")); b = 0
                    for p in self._parts(w):
                        s0, e0 = cstart + b2c[b], cstart + b2c[b + len(p) - 1] + 1
                        lead = len(p) - len(p.lstrip("Ġ")); trail = 0   # Ġ is the symbol for a space byte
                        s1 = min(s0 + lead, e0)
                        ids.append(self.vocab.get(p, 0)); toks.append(p); offs.append((s1, e0)); b += len(p)
            pos += len(piece)
        e = _Enc(ids); e.tokens, e.offsets = toks, offs
        return e
    def encode_batch(self, texts):
        out = []
        for t in texts:
            e = _Enc(self.encode_ids(t)); out.append(e)      # batch use only needs ids; use encode() when offsets are needed
        return out
    def decode(self, ids):
        out = bytearray()
        for i in ids:
            t = self.id2tok[int(i)]
            out += t.encode() if t in SPECIALS else bytes(U2B[c] for c in t)
        return out.decode("utf-8", errors="replace")
    @classmethod
    def from_json(cls, path):
        d = json.load(open(path)); m = d.get("model", d)       # accepts the saved merge-table file
        return cls(m["vocab"], [tuple(x.split(" ")) if isinstance(x, str) else tuple(x) for x in m["merges"]])
    def save(self, path):
        json.dump({"vocab": self.vocab, "merges": [list(m) for m in self.merges]}, open(path, "w"))


**Code cell 4. What this cell does.** Implements the byte-level BPE tokenizer by hand: it can learn a merge table from text (`train_bpe`) and it can encode and decode text with a merge table (`ByteBPE`). Nothing in it depends on a tokenizer library.

**Background.** BPE (byte-pair encoding) starts with single symbols and repeatedly merges the most frequent adjacent pair into a new symbol, until the vocabulary has the wanted size. Byte-level means the starting symbols are the 256 byte values, so any text, including accents and rare characters, can be encoded.

**Pieces, in order**
- `PAT` (the regular expression): splits text into "words" the way GPT-2 does: contractions (`'s`, `'t`...), runs of letters with an optional leading space, runs of digits, runs of punctuation, and whitespace. Merges never cross these boundaries. `\p{L}` and `\p{N}` mean any Unicode letter or digit, which needs the `regex` module (the `except ImportError` branch is a plainer fallback).
- `bytes_to_unicode()`: gives every byte value a printable character (printable bytes map to themselves; the others, such as space and newline, are shifted to unused code points; this is why a leading space appears as `Ġ`). `B2U` maps byte to character, `U2B` is the reverse.
- `SPECIALS = ["<|endoftext|>", "[PAD]"]`: two reserved tokens that take ids 0 and 1. `SPECIAL_SPLIT` is a regex that finds them in text so they are kept whole.
- `pretokenize(text)`: splits text with `PAT`, converts each word to UTF-8 bytes, maps each byte to its character, and returns the list of words.

**`train_bpe(texts, vocab_size, min_frequency)`**
1. Count how often each distinct word occurs (`wc`), so each distinct word is processed once with its count instead of once per occurrence.
2. The starting vocabulary is the two special tokens, then every character that occurs, in sorted order. Each word becomes a list of ids.
3. Count every adjacent id pair across all words (`pair_cnt`, weighted by word count) and remember which words contain each pair (`where`).
4. Put the pair counts in a max-heap (counts are negated because Python's heap is a min-heap; ties are broken by the smaller pair, which makes the result deterministic).
5. Loop until the vocabulary reaches `vocab_size`: pop the best pair; if its stored count is stale (counts change as merges happen) skip or re-push it; stop if its count is below `min_frequency`; otherwise create the merged token (concatenation of the two strings), give it the next id, and record the merge.
6. Rewrite only the words that contained the pair: subtract their old pair counts, replace each occurrence of the pair by the new id (left to right), add the new pair counts, and push the changed pairs onto the heap.
7. Return `vocab` (token string to id) and `merges` (ordered list; the order is the priority used when encoding).

**`ByteBPE`**
- `__init__` stores the vocabulary, the reverse map `id2tok`, the merge list, and `rank` (merge to its position; a lower rank means it was learned earlier and applies first). `cache` remembers the result for each word already seen, which speeds up encoding a lot because legal text repeats words.
- `_bpe(w)`: starts from single characters and repeatedly merges the adjacent pair with the lowest rank until no known pair is left; any symbol missing from the vocabulary maps to id 0.
- `encode_ids(text)`: splits around the special tokens, then pre-tokenizes the rest and applies `_bpe` to each word.
- `encode(text)` returns a small object with `.ids`, `.tokens` and `.offsets`. The offsets give, for every token, the character span `(start, end)` in the original text; they are what the NER task (Phase 2) needs to turn token labels into character spans. It works out which character each byte belongs to, so a token that covers part of an accented letter still maps to the right character, and it trims the leading space symbol from the span so a span covers only visible characters. `encode_batch` returns the ids only and is used where offsets are not needed.
- `decode(ids)`: looks up each token, converts its characters back to bytes (`U2B`), and decodes the bytes as UTF-8.
- `from_json(path)`: loads a saved merge table (accepts both a flat file and one with a `model` section). `save(path)` writes `vocab` and `merges`.

**Why this design.** Counting distinct words first and updating pair counts incrementally avoids re-scanning the corpus after each merge, so 30,000 merges take minutes, not days. The result is deterministic, so the same text and settings always give the same merge table.

In [ ]:
import os, json, glob, shutil, torch
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shutil.copy(f"{DRIVE_PROJECT_DIR}/legal_bpe_32k_FINAL.json", "legal_bpe_32k_FINAL.json")
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
assert tokenizer.get_vocab_size() == 32000
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# HF_TOKEN = "..."   # set it as before (the Phase-2 setup cell used it)
print("device:", device, "| vocab:", tokenizer.get_vocab_size())
for p in sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True)): print(" ", p)

Mounted at /content/drive
device: cuda | vocab: 32000
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt
  /content/drive/M

**Code cell 5. What this cell does.** Mounts Google Drive, picks the device, and loads the saved tokenizer. Every later cell relies on the variables created here (`tokenizer`, `device`, `DRIVE_PROJECT_DIR`).

**Line by line**
- Imports: `os, json, glob, shutil, torch`, plus `drive` from Colab.
- `if not os.path.isdir("/content/drive/MyDrive"): drive.mount(...)`: mounts Drive only if it is not already mounted, so re-running the cell does not prompt again.
- `DRIVE_PROJECT_DIR`: the Phase-1 folder on Drive where tokenizer, packed data and checkpoints live.
- `device = torch.device("cuda" if torch.cuda.is_available() else "cpu")`: use the GPU when present.
- `shutil.copy(...)`: copies the tokenizer file from Drive to the local disk, which is faster to read.
- `tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")`: builds the tokenizer from the saved vocabulary and merge table with the class defined in the previous cell.
- `assert tokenizer.get_vocab_size() == 32000`: fails immediately if a wrong tokenizer file was picked up.
- `HF_DATASET_REPO`: the dataset id again, so this cell can be used on its own after a restart.
- The `print` lines show the device, the vocabulary size and every `.pt` checkpoint found on Drive, which tells you at a glance what has already been trained.

**Why.** Colab runtimes reset, so the first thing after a restart is to re-create these three variables from Drive.

In [ ]:
for name in ["tokenizer", "train_dataset", "val_dataset", "vocab_sz", "device"]:
    print(name, "->", "defined" if name in globals() else "MISSING")

tokenizer -> defined
train_dataset -> defined
val_dataset -> defined
vocab_sz -> defined
device -> defined


**Code cell 6. What this cell does.** A safety check: confirms that the main objects exist before moving on.

**Line by line.** It loops over the names `tokenizer`, `train_dataset`, `val_dataset`, `vocab_sz`, `device` and prints `defined` or `MISSING` for each by looking them up in `globals()`.

**Why.** In a long notebook it is easy to skip a cell. Seeing `MISSING` here tells you which earlier cell to run, instead of getting a confusing error later. (At this point in the notebook `train_dataset`, `val_dataset` and `vocab_sz` are created by later cells, so those lines are expected to say `MISSING` on a fresh run.)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Pick a clear destination folder - adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
os.makedirs(DEST_DIR, exist_ok=True)

src_path = "legal_bpe_32k_FINAL.json"
dest_path = os.path.join(DEST_DIR, "legal_bpe_32k_FINAL.json")

shutil.copy(src_path, dest_path)
print(f"Saved to: {dest_path}")

# Quick confirmation it actually copied correctly
check = ByteBPE.from_json(dest_path)
print(f"Verified — loads fine from Drive, vocab size: {check.get_vocab_size()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/legal_bpe_32k_FINAL.json
Verified — loads fine from Drive, vocab size: 32000


**Code cell 7. What this cell does.** Saves the tokenizer file to Google Drive so it survives a runtime reset, and checks that it loads from there.

**Line by line**
- `drive.mount(...)`: mounts Drive.
- `DEST_DIR`: the Phase-1 project folder; `os.makedirs(..., exist_ok=True)` creates it if it does not exist and does nothing otherwise.
- `shutil.copy(src_path, dest_path)`: copies `legal_bpe_32k_FINAL.json` from the local disk to Drive.
- `check = ByteBPE.from_json(dest_path)`: loads the copy to prove it is readable, and the `print` shows its vocabulary size.

**Why.** Local files in Colab disappear when the runtime ends; this file is needed by every later phase, so it is kept on Drive.

## 5. Download Training Data

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = []

for file_path in TRAIN_FILES:
    print(f"Downloading '{file_path}' from repository '{HF_DATASET_REPO}'...")
    downloaded_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    train_file_paths.append(downloaded_path)
    print(f"✅ Downloaded to: {downloaded_path}\n")

print("All selected training files downloaded successfully!")
print(train_file_paths)

Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json



Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json



Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json

All selected training files downloaded successfully!
['/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_tribunals.json']


**Code cell 8. What this cell does.** Downloads the pretraining files from the dataset repository.

**Line by line**
- `train_file_paths = []` collects the local paths.
- The loop goes over `TRAIN_FILES` (defined in the configuration cell). For each file it prints a message, then calls `hf_hub_download(repo_id=..., filename=..., repo_type="dataset", token=HF_TOKEN)`, which downloads the file into the local cache and returns its path.
- The returned path is appended to `train_file_paths` and printed.

**Why.** `repo_type="dataset"` is required because the data lives in a dataset repository, not a model repository. The token comes from the login cell. Files are cached, so re-running does not download again.

## 6. Create Train and Validation Tokens Set

In [ ]:
import json
from huggingface_hub import hf_hub_download
from datasets import Dataset
import numpy as np
import os
import time
from tqdm import tqdm

# 1. Load your verified tokenizer — MAKE SURE THIS IS THE CORRECT FILE
tokenizer_path = "legal_bpe_32k_FINAL.json"
tokenizer = ByteBPE.from_json(tokenizer_path)
print(f"✅ Loaded tokenizer from {tokenizer_path} (Vocab size: {tokenizer.get_vocab_size()})")

_test = tokenizer.encode("the court held that")
assert tokenizer.decode(_test.ids).strip() != "", "Tokenizer decode failed sanity check — STOP."
print("Tokenizer sanity check passed.")

# 2. Gather raw text — everything except high courts
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
]

all_texts = []
for file_path in tqdm(TRAIN_FILES, desc="Downloading & Reading Dataset Files"):
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
    with open(local_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        records = list(data.values()) if isinstance(data, dict) else data
        for record in records:
            text = record.get("text") if isinstance(record, dict) else record
            if text and isinstance(text, str):
                all_texts.append(text)

print(f"Total raw texts collected: {len(all_texts)}")

# 3. Train/Val split
raw_dataset = Dataset.from_dict({"text": all_texts})
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train samples: {len(split_dataset['train'])} | Val samples: {len(split_dataset['test'])}")

# 4. Tokenize + pack — SMALL batch_size and writer_batch_size to avoid memory pressure/swapping
BLOCK_SIZE = 2048
endoftext_id = tokenizer.token_to_id("<|endoftext|>")
if endoftext_id is None:
    endoftext_id = 256

def tokenize_and_pack(examples):
    batch_encodings = tokenizer.encode_batch(examples["text"])
    all_token_ids = []
    for encoding in batch_encodings:
        all_token_ids.extend(encoding.ids)
        all_token_ids.append(endoftext_id)

    total_length = (len(all_token_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_token_ids = all_token_ids[:total_length]
    chunked_ids = np.array(all_token_ids, dtype=np.int32).reshape(-1, BLOCK_SIZE)

    return {"input_ids": chunked_ids.tolist(), "labels": chunked_ids.tolist()}

PACK_BATCH_SIZE = 500      # small, avoids huge in-RAM lists per batch
WRITER_BATCH_SIZE = 500    # flush to disk often, don't hoard memory
NUM_PROC = 2                # safe default for Colab; don't blindly trust os.cpu_count()

print(f"Detected CPU count: {os.cpu_count()} (using NUM_PROC={NUM_PROC} regardless)")

print(f"⚙️ Tokenizing and packing training set...")
t0 = time.time()
tokenized_train = split_dataset["train"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Train Split"
)
print(f"Train packing took {time.time() - t0:.1f}s")

print("⚙️ Tokenizing and packing validation set...")
t0 = time.time()
tokenized_val = split_dataset["test"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Val Split"
)
print(f"Val packing took {time.time() - t0:.1f}s")

# 5. Save
output_dir = "packed_pretrain_data"
os.makedirs(output_dir, exist_ok=True)
tokenized_train.save_to_disk(os.path.join(output_dir, "train"))
tokenized_val.save_to_disk(os.path.join(output_dir, "val"))

print(f"\n🎉 Saved to '{output_dir}/'")
print(f"   - Train blocks: {len(tokenized_train)} (~{len(tokenized_train) * BLOCK_SIZE / 1e6:.2f}M tokens)")
print(f"   - Val blocks:   {len(tokenized_val)} (~{len(tokenized_val) * BLOCK_SIZE / 1e6:.2f}M tokens)")

# 6. Verify packed data decodes correctly
sample_ids = tokenized_train[0]["input_ids"]
decoded_sample = tokenizer.decode(sample_ids[:50])
print(f"\n--- Packed block decode check ---\n{decoded_sample}")
assert decoded_sample.strip() != "", "Packed data decodes to empty — tokenizer/data mismatch!"
print("Packed data verified readable.")

**Code cell 9. What this cell does.** Turns the raw judgments into fixed-length blocks of token ids, ready for language-model training, and saves them to disk. This is the data-preparation step of Phase 1.

**Line by line**
1. **Load the tokenizer** from `legal_bpe_32k_FINAL.json` and run a quick encode/decode sanity check (`"the court held that"` must decode to non-empty text).
2. **Gather raw text.** `TRAIN_FILES` here is the Acts file plus the Supreme Court file (the tribunal file is left out for speed). Each file is downloaded, read as JSON, and every record's `text` field is appended to `all_texts`.
3. **Split.** `Dataset.from_dict({"text": all_texts})` makes a table; `train_test_split(test_size=0.05, seed=42)` holds out 5% of documents as validation, reproducibly.
4. **Tokenize and pack** (`tokenize_and_pack`):
   - `tokenizer.encode_batch(...)` tokenizes a batch of documents.
   - After each document's ids, the end-of-text id is appended, so the model can see where one document ends and the next begins.
   - All ids in the batch are joined into one long list, cut to a multiple of `BLOCK_SIZE` (2,048), and reshaped into rows of 2,048. Any leftover tail shorter than a block is dropped.
   - The function returns `input_ids` and an identical `labels` copy; the model shifts labels by one position itself when it computes the loss.
5. **Settings.** `PACK_BATCH_SIZE = 500` documents per batch and `WRITER_BATCH_SIZE = 500` rows per write keep memory low; `NUM_PROC = 2` uses two worker processes.
6. `.map(..., batched=True, remove_columns=["text"])` applies the function to the train and validation splits and drops the raw text.
7. **Save** both splits to `packed_pretrain_data/train` and `/val` with `save_to_disk`, and print block counts and approximate token counts.
8. **Verify** by decoding the first 50 ids of the first block and asserting the text is non-empty.

**Why.** Packing removes padding entirely: every block is full, so no compute is wasted. Splitting by document (not by block) keeps validation text unseen during training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Destination folder on Drive — adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data"
os.makedirs(DEST_DIR, exist_ok=True)

src_dir = "packed_pretrain_data"  # the local folder your script already saved to

for split in ["train", "val"]:
    src_path = os.path.join(src_dir, split)
    dest_path = os.path.join(DEST_DIR, split)

    if os.path.exists(dest_path):
        shutil.rmtree(dest_path)  # avoid stale leftovers from a previous run

    shutil.copytree(src_path, dest_path)
    print(f"Copied {src_path} -> {dest_path}")

print("\nDone. Verifying by reloading from Drive...")

# Verify both splits reload correctly straight from Drive
from datasets import load_from_disk

train_check = load_from_disk(os.path.join(DEST_DIR, "train"))
val_check = load_from_disk(os.path.join(DEST_DIR, "val"))

print(f"Train blocks on Drive: {len(train_check)}")
print(f"Val blocks on Drive:   {len(val_check)}")
print(f"Sample input_ids length: {len(train_check[0]['input_ids'])}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied packed_pretrain_data/train -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data/train
Copied packed_pretrain_data/val -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data/val

Done. Verifying by reloading from Drive...
Train blocks on Drive: 190794
Val blocks on Drive:   10207
Sample input_ids length: 2048


**Code cell 10. What this cell does.** Copies the packed training and validation data from the Colab disk to Drive and verifies the copy.

**Line by line**
- Mounts Drive and creates `.../packed_pretrain_data` on it.
- For `train` and `val`: if a copy already exists on Drive it is deleted first (so no stale files remain), then the local folder is copied over with `shutil.copytree`.
- It reloads both splits from Drive with `load_from_disk` and prints the block counts and the length of the first block.

**Why.** Packing the data takes a while; the packed folder on Drive can be reused by every Phase-1 notebook and after any restart.

## 7. Create Token Blocks

In [ ]:
from datasets import load_from_disk

# Load your pre-packed training and validation blocks from disk
train_blocks = load_from_disk("packed_pretrain_data/train")
val_blocks = load_from_disk("packed_pretrain_data/val")

print(f"✅ Loaded train_blocks: {len(train_blocks):,} sequences")
print(f"✅ Loaded val_blocks:   {len(val_blocks):,} sequences")

✅ Loaded train_blocks: 190,794 sequences
✅ Loaded val_blocks:   10,207 sequences


**Code cell 11. What this cell does.** Loads the packed blocks from disk and reports how many there are.

**Line by line.** `load_from_disk` reads the saved `train` and `val` tables; the two `print` lines show the number of sequences (each one 2,048 tokens) with thousands separators.




## 8. Create Datasets

In [ ]:
# train_blocks and val_blocks are already Hugging Face Datasets!
print(train_blocks)
print(val_blocks)

# You can inspect columns directly:
print(train_blocks[0])

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 190794
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 10207
})
{'input_ids': [1443, 484, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 5512, 2173, 1934, 2900, 1753, 13, 235, 1589, 492, 2899, 13, 3714, 4064, 241, 2073, 2183, 2368, 308, 19616, 1703, 10, 372, 15, 280, 15, 342, 15, 24340, 308, 38, 922, 655, 325, 1050, 1616, 521, 292, 235, 2266, 294, 2401, 295, 1672, 1633, 25828, 36, 15, 395, 15, 14177, 3, 1443, 484, 4385, 3520, 308, 9017, 19669, 241, 2632, 1318, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 440, 1365, 241, 235, 1301, 2226, 292, 1110, 308, 18, 10, 241, 2749, 14869, 43, 241, 235, 991, 13, 235, 2266, 2458, 3056, 13, 343, 803, 259, 235, 484, 241, 3520, 13, 235, 1050, 1616, 13, 1618, 3549, 180, 18, 15, 4477, 2312, 268, 2083, 15, 180, 9, 18, 10, 948, 1616, 414, 274, 1847, 235, 1008, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 161

**Code cell 12. What this cell does.** Shows the structure of the loaded datasets.

**Line by line.** Printing a dataset shows its column names (`input_ids`, `labels`) and row count; `train_blocks[0]` shows the first example, a dictionary of two lists of 2,048 ids. This is just a look at the data.

## 9. Define The Model

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. CORE BUILDING BLOCKS & GQA
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


def precompute_rope_params(head_dim: int, max_seq_len: int = 4096, theta: float = 10000.0):
    assert head_dim % 2 == 0, "Head dimension must be even for RoPE"
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cos = torch.cos(freqs).repeat_interleave(2, dim=-1)
    freqs_sin = torch.sin(freqs).repeat_interleave(2, dim=-1)
    return freqs_cos, freqs_sin


def apply_rotary_emb(x, cos, sin):
    # x: [B, heads, T, head_dim]; cos/sin: [T, head_dim]
    def rotate_half(tensor):
        x1 = tensor[..., 0::2]
        x2 = tensor[..., 1::2]
        return torch.stack([-x2, x1], dim=-1).flatten(-2)
    cos = cos.to(x.dtype).unsqueeze(0).unsqueeze(1)   # cast so bf16 stays bf16
    sin = sin.to(x.dtype).unsqueeze(0).unsqueeze(1)
    return (x * cos) + (rotate_half(x) * sin)


class SwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden_dim = int(2.5 * d_model)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class GroupedQueryAttention(nn.Module):
    """GQA with RoPE. Attention is explicit matmul + softmax (no flash / no SDPA)."""
    def __init__(self, d_model: int, n_head: int, n_kv_head: int = 4):
        super().__init__()
        assert n_head % n_kv_head == 0, "n_head must be divisible by n_kv_head"
        self.d_model = d_model
        self.n_head = n_head
        self.n_kv_head = n_kv_head
        self.n_rep = n_head // n_kv_head
        self.head_dim = d_model // n_head

        self.q_proj = nn.Linear(d_model, n_head * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_head * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_head * self.head_dim, d_model, bias=False)

    def forward(self, x, cos, sin, past_kv=None):
        B, T, C = x.shape

        q = self.q_proj(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)      # [B, H, T, hd]
        k = self.k_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, T, hd]
        v = self.v_proj(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)   # [B, Hkv, T, hd]

        past_len = past_kv[0].shape[2] if past_kv is not None else 0
        total_len = past_len + T

        q = apply_rotary_emb(q, cos[past_len:total_len], sin[past_len:total_len])
        k = apply_rotary_emb(k, cos[past_len:total_len], sin[past_len:total_len])

        if past_kv is not None:
            k = torch.cat([past_kv[0], k], dim=2)
            v = torch.cat([past_kv[1], v], dim=2)
        present_kv = (k, v)   # cache holds only the 4 KV heads, 3x smaller than full MHA

        # Group query heads under the KV head they share: [B, Hkv, n_rep, T, hd]
        q = q.reshape(B, self.n_kv_head, self.n_rep, T, self.head_dim)
        # K broadcasts across the n_rep dimension, so no expanded copy is made
        scores = torch.matmul(q, k.unsqueeze(2).transpose(-2, -1)) / math.sqrt(self.head_dim)

        if T > 1:   # a single new token (T == 1) may attend to everything, so no mask needed
            allowed = torch.ones(T, total_len, dtype=torch.bool, device=x.device).tril(diagonal=past_len)
            scores = scores.masked_fill(~allowed, float("-inf"))

        attn_weights = F.softmax(scores, dim=-1)
        y = torch.matmul(attn_weights, v.unsqueeze(2))                    # [B, Hkv, n_rep, T, hd]
        y = y.reshape(B, self.n_head, T, self.head_dim).transpose(1, 2).reshape(B, T, C)
        return self.out_proj(y), present_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, n_kv_head: int = 4):
        super().__init__()
        self.attention_norm = RMSNorm(d_model)
        self.attn = GroupedQueryAttention(d_model, n_head, n_kv_head)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, cos, sin, past_kv=None):
        attn_out, present_kv = self.attn(self.attention_norm(x), cos, sin, past_kv=past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, present_kv


# ==========================================
# 2. MODEL WRAPPER & INITIALIZATION
# ==========================================
class CustomDecoderLM(nn.Module):
    """Full Decoder-Only Language Model from Scratch with GQA + KV Caching."""
    def __init__(self, vocab_size=32000, d_model=768, n_layer=10, n_head=12,
                 n_kv_head=4, max_seq_len=2048, tie_weights=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_head, n_kv_head) for _ in range(n_layer)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_weights:
            self.token_embedding.weight = self.lm_head.weight

        # RoPE is applied to the full head dimension in GQA
        cos, sin = precompute_rope_params(d_model // n_head, max_seq_len)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None, past_key_values=None):
        B, T = input_ids.shape
        x = self.token_embedding(input_ids)

        new_present_key_values = []
        for idx, layer in enumerate(self.layers):
            layer_past = past_key_values[idx] if past_key_values is not None else None
            x, present_kv = layer(x, self.cos, self.sin, past_kv=layer_past)
            new_present_key_values.append(present_kv)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, self.vocab_size), shift_labels.view(-1))

        return {
            "loss": loss,
            "logits": logits,
            "past_key_values": tuple(new_present_key_values)
        }


# ==========================================
# 3. DETERMINE VOCAB SIZE — explicit, no silent fallback
# ==========================================
assert 'tokenizer' in globals(), "Load your verified tokenizer (e.g. legal_bpe_32k_FINAL.json) before building the model."
vocab_sz = tokenizer.get_vocab_size()
print(f"Using tokenizer vocab size: {vocab_sz}")

target_device = DEVICE if 'DEVICE' in globals() else (device if 'device' in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# ==========================================
# 4. BUILD MODEL — ~87.5M parameters
# ==========================================
model = CustomDecoderLM(
    vocab_size=vocab_sz,
    d_model=768,
    n_layer=10,
    n_head=12,
    n_kv_head=4,
    max_seq_len=2048,
    tie_weights=True
).to(target_device)

model_size = sum(p.numel() for p in model.parameters())
print(f"🚀 Custom GQA Decoder Model Size: {model_size / 1e6:.2f}M parameters")

# ==========================================
# 5. QUICK SHAPE + CACHE CHECK
# ==========================================
with torch.no_grad():
    _ids = torch.randint(0, vocab_sz, (2, 16), device=target_device)
    _full = model(_ids)["logits"]
    _o1 = model(_ids[:, :8])
    _o2 = model(_ids[:, 8:], past_key_values=_o1["past_key_values"])
    _diff = (_full[:, 8:] - _o2["logits"]).abs().max().item()
print(f"Cached vs uncached max logit diff: {_diff:.2e} (should be ~1e-5 or smaller)")

Using tokenizer vocab size: 32000
🚀 Custom GQA Decoder Model Size: 87.51M parameters
Cached vs uncached max logit diff: 1.91e-06 (should be ~1e-5 or smaller)


**Code cell 13. What this cell does.** Defines the whole Phase-1 GQA language model from scratch, builds it, and checks that its cache works. Nothing here uses a ready-made transformer layer.

**Building blocks**
- `RMSNorm`: normalises each token's vector by its root-mean-square (no mean subtraction, no bias), then scales by a learned weight. `torch.rsqrt(variance + eps)` is one over the root; `eps` avoids dividing by zero. It is cheaper than LayerNorm and works as well here.
- `precompute_rope_params`: builds the angles for rotary position embeddings (RoPE). For each pair of dimensions it computes a frequency `1 / theta^(2i/d)`, multiplies by every position, and stores cosine and sine tables of shape `[max_seq_len, head_dim]`.
- `apply_rotary_emb`: rotates each pair of channels in q and k by the angle of its position, so the dot product of two tokens depends on their distance. `rotate_half` builds the (-x2, x1) partner vector. Casting `cos`/`sin` to the tensor's dtype keeps bf16 as bf16.
- `SwiGLU`: the feed-forward network `w2(silu(w1 x) * w3 x)`; the hidden width is 2.5 times the model width, rounded up to a multiple of 256 for hardware efficiency.
- `GroupedQueryAttention` (GQA): 12 query heads but only 4 key/value heads; each group of 3 query heads shares one key/value head. This makes the KV cache 3 times smaller than full multi-head attention while keeping most of the quality.
  - `q_proj`, `k_proj`, `v_proj`, `out_proj` are the four projections, without bias.
  - `forward` projects, reshapes to `[batch, heads, time, head_dim]`, applies RoPE using the correct positions (`past_len` is how many tokens are already cached), appends the new k and v to the cache (`past_kv`) and returns the cache as `present_kv`.
  - Queries are regrouped to `[B, Hkv, n_rep, T, hd]` so that keys broadcast over the `n_rep` axis without copying.
  - `scores = q k^T / sqrt(head_dim)`; for more than one new token a lower-triangular mask sets future positions to minus infinity so a token cannot see later tokens (`tril(diagonal=past_len)` accounts for cached tokens).
  - softmax gives attention weights, which multiply v; the heads are merged and `out_proj` mixes them.
- `TransformerBlock`: pre-norm residual block: `x = x + attn(norm(x))`, then `x = x + ffn(norm(x))`.

**`CustomDecoderLM`**
- `token_embedding` maps ids to vectors; a stack of `n_layer` blocks; a final `RMSNorm`; `lm_head` maps back to vocabulary logits.
- `tie_weights=True` shares the embedding matrix with `lm_head`, saving 24.6M parameters and usually helping small models.
- `cos`/`sin` tables are registered as non-persistent buffers: they move with the model but are not saved in checkpoints.
- `_init_weights`: normal(0, 0.02) for all linear and embedding weights, the standard small initialisation.
- `forward`: embeds, runs the layers while collecting each layer's cache, normalises, computes logits. If `labels` are given, the loss shifts by one (the logits at position t predict the label at t+1) and takes cross-entropy.

**Building and checking**
- The vocabulary size comes from the tokenizer (32,000). The model is `d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048`, about 87.5M parameters; the cell prints the exact count.
- The cache check runs a 16-token input in one pass, and again as 8 tokens followed by 8 tokens using the cache. The maximum difference between the two sets of logits should be around 1e-5: if caching were wrong, generation would drift.

**Why GQA and RoPE.** RoPE gives relative position without a learned position table; GQA cuts memory for long context and fast generation.

## 10. Training Arguments

In [ ]:
from datasets import load_from_disk

# 1. Load the pre-packed datasets from disk
train_dataset = load_from_disk("packed_pretrain_data/train")
val_dataset = load_from_disk("packed_pretrain_data/val")

# 2. Set format to PyTorch tensors
train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

**Code cell 14. What this cell does.** Loads the packed data again and switches it to PyTorch tensors.

**Line by line.** `load_from_disk` reads both splits; `set_format(type="torch", columns=["input_ids", "labels"])` makes indexing return tensors instead of Python lists, which the data loader needs.

In [ ]:
from datasets import load_from_disk
import torch

DATA_DIR = "packed_pretrain_data"  # or your Drive path if this is a fresh session
train_dataset = load_from_disk(f"{DATA_DIR}/train")
val_dataset = load_from_disk(f"{DATA_DIR}/val")

train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

print(f"Train blocks: {len(train_dataset)} | Val blocks: {len(val_dataset)}")

Train blocks: 190794 | Val blocks: 10207


**Code cell 15. What this cell does.** The same loading step as the previous cell, but from a configurable folder, with a count printed.

**Line by line.** `DATA_DIR` can point to the local folder or to the Drive copy after a fresh session. It loads, sets the torch format, and prints the number of train and validation blocks.

## 11. Training

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


**Code cell 16. What this cell does.** Selects the GPU (or CPU) and prints it.

**Line by line.** `torch.device("cuda" if torch.cuda.is_available() else "cpu")` and a `print`. It is repeated here so this section can run on its own after a restart.

In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

BLOCK_LEN = len(train_dataset[0]["input_ids"])
assert BLOCK_LEN <= model.max_seq_len, f"Block length {BLOCK_LEN} > model.max_seq_len {model.max_seq_len}"

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 4
    BATCH_SIZE = 4                      # if OOM: BATCH_SIZE = 2 and GRADIENT_ACCUMULATION_STEPS = 8
    MAX_STEPS_PER_EPOCH = 20000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 4
    WARMUP_STEPS = 500                  # optimizer steps
    COOLDOWN_STEPS = 1000               # optimizer steps
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = "gqa_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/gqa_best{suffix}.pt"
LAST_PATH = f"{CKPT_DIR}/gqa_last{suffix}.pt"

# ============================================================
# 3. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=custom_collate_fn)
val_blocks = min(len(val_dataset), 400)
val_loader = DataLoader(Subset(val_dataset, range(val_blocks)), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=custom_collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
tokens_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * BLOCK_LEN

print(f"Train blocks: {len(train_dataset):,} | Val blocks used: {val_blocks}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Tokens/optimizer step: {tokens_per_step:,} | Tokens seen in total: {total_steps * tokens_per_step / 1e6:.0f}M")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history}, path)

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Train blocks: 190,794 | Val blocks used: 400
Micro-batches/epoch: 20000 | Optimizer steps/epoch: 5000 | Total: 20000
Tokens/optimizer step: 32,768 | Tokens seen in total: 655M
Using bf16: True

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...


Epoch 1/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 500: val_loss=4.3661 (ppl 78.7)

step 1000: val_loss=3.6533 (ppl 38.6)

step 1500: val_loss=3.2694 (ppl 26.3)

step 2000: val_loss=3.0202 (ppl 20.5)

step 2500: val_loss=2.8833 (ppl 17.9)

step 3000: val_loss=2.7986 (ppl 16.4)

step 3500: val_loss=2.7346 (ppl 15.4)

step 4000: val_loss=2.6811 (ppl 14.6)

step 4500: val_loss=2.6459 (ppl 14.1)

step 5000: val_loss=2.6095 (ppl 13.6)

📊 Epoch 1 -> Train Loss: 3.2116 | Val Loss: 2.6095 (ppl 13.6)



Epoch 2/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 5500: val_loss=2.5804 (ppl 13.2)

step 6000: val_loss=2.5518 (ppl 12.8)

step 6500: val_loss=2.5379 (ppl 12.7)

step 7000: val_loss=2.5225 (ppl 12.5)

step 7500: val_loss=2.4982 (ppl 12.2)

step 8000: val_loss=2.4830 (ppl 12.0)

step 8500: val_loss=2.4624 (ppl 11.7)

step 9000: val_loss=2.4536 (ppl 11.6)

step 9500: val_loss=2.4439 (ppl 11.5)

step 10000: val_loss=2.4282 (ppl 11.3)

📊 Epoch 2 -> Train Loss: 2.4509 | Val Loss: 2.4282 (ppl 11.3)



Epoch 3/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 10500: val_loss=2.4097 (ppl 11.1)

step 11000: val_loss=2.4036 (ppl 11.1)

step 11500: val_loss=2.4004 (ppl 11.0)

step 12000: val_loss=2.3927 (ppl 10.9)

step 12500: val_loss=2.3858 (ppl 10.9)

step 13000: val_loss=2.3775 (ppl 10.8)

step 13500: val_loss=2.3600 (ppl 10.6)

step 14000: val_loss=2.3578 (ppl 10.6)

step 14500: val_loss=2.3539 (ppl 10.5)

step 15000: val_loss=2.3473 (ppl 10.5)

📊 Epoch 3 -> Train Loss: 2.3194 | Val Loss: 2.3473 (ppl 10.5)



Epoch 4/4:   0%|          | 0/5000 [00:00<?, ?it/s]


step 15500: val_loss=2.3374 (ppl 10.4)

step 16000: val_loss=2.3448 (ppl 10.4)

step 16500: val_loss=2.3346 (ppl 10.3)

step 17000: val_loss=2.3303 (ppl 10.3)

step 17500: val_loss=2.3279 (ppl 10.3)

step 18000: val_loss=2.3208 (ppl 10.2)

step 18500: val_loss=2.3110 (ppl 10.1)

step 19000: val_loss=2.3105 (ppl 10.1)

step 19500: val_loss=2.2450 (ppl 9.4)

step 20000: val_loss=2.1949 (ppl 9.0)

📊 Epoch 4 -> Train Loss: 2.2324 | Val Loss: 2.1949 (ppl 9.0)

✅ Training complete!
Best val loss: 2.1949 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt


**Code cell 17. What this cell does.** The Phase-1 pretraining run: trains the GQA model on next-token prediction, evaluates it, and saves checkpoints.

**Section 0: guards.** It asserts that `tokenizer`, `model`, `train_dataset`, `val_dataset` and `device` exist, that the model's vocabulary equals the tokenizer's, and that a block (2,048) fits in `model.max_seq_len`.

**Section 1: settings.** `DEBUG_MODE` switches between a tiny smoke test (40 micro-batches, 1 epoch) and the real run: 4 epochs, `BATCH_SIZE = 4`, up to 20,000 micro-batches per epoch, gradient accumulation of 4, so one optimizer step sees 4 x 4 x 2,048 = 32,768 tokens. Warm-up is 500 optimizer steps, a final cool-down of 1,000 steps, evaluation every 500 steps. `LEARNING_RATE = 3e-4`, `WEIGHT_DECAY = 0.1`, gradient clip 1.0. `RESUME = True` continues from the last checkpoint if there is one.

**Section 2: paths.** Checkpoints go to `gqa_checkpoints/` on Drive (or locally with a warning if Drive is not mounted). `gqa_best.pt` holds the lowest validation loss, `gqa_last.pt` the latest state for resuming.

**Section 3: data loaders.** `custom_collate_fn` stacks the tensors into a batch. Training shuffles and drops the last incomplete batch; validation uses only the first 400 blocks to stay quick. It then computes micro-batches per epoch, optimizer steps per epoch, total steps and tokens per step, and prints them.

**Section 4: optimizer and schedule.** AdamW with `betas=(0.9, 0.95)`. Parameters with two or more dimensions (matrices, embeddings) get weight decay; one-dimensional ones (norm weights) do not, since decaying them only hurts. The learning-rate multiplier rises linearly during warm-up, stays at 1.0, and falls linearly to 0.1 in the cool-down. `use_bf16` turns on bfloat16 if the GPU supports it.

**Section 5: helpers.** `evaluate()` puts the model in eval mode, averages the loss over the validation batches under autocast, and restores train mode. `save_ckpt` saves model, optimizer, scheduler, step counters and loss history. If `RESUME` and `gqa_last.pt` exists, everything is restored and the starting epoch is recomputed.

**Section 6: the loop.** For each epoch, for each micro-batch: move to GPU, run the forward pass under autocast, divide the loss by the accumulation count, and `loss.backward()` adds gradients. Every 4 micro-batches: clip the gradient norm to 1.0, `optimizer.step()`, `scheduler.step()`, clear gradients, count a step. Every `LOG_EVERY` steps the progress bar shows loss, validation loss, learning rate, tokens per second, an ETA and peak GPU memory. Every `EVAL_EVERY` steps it evaluates, prints validation loss and perplexity, saves `gqa_best.pt` if it improved, and always saves `gqa_last.pt`. At the end of each epoch it evaluates again and saves.

**Why these choices.** Gradient accumulation gives a large effective batch on a small GPU; warm-up avoids early instability; cool-down settles the weights; saving "best" and "last" separately protects against both overfitting and runtime resets. Perplexity is `exp(loss)`; the untrained loss is about ln(32000) = 10.4.

In [ ]:
import os
import math
import time
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

BLOCK_LEN = len(train_dataset[0]["input_ids"])
assert BLOCK_LEN <= model.max_seq_len, f"Block length {BLOCK_LEN} > model.max_seq_len {model.max_seq_len}"

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 8
    MAX_STEPS_PER_EPOCH = 40            # micro-batches per epoch
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 3
    COOLDOWN_STEPS = 3
    EVAL_EVERY = 5
    LOG_EVERY = 1
    RESUME = False
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 4
    BATCH_SIZE = 4                      # if OOM: BATCH_SIZE = 2 and GRADIENT_ACCUMULATION_STEPS = 8
    MAX_STEPS_PER_EPOCH = 20000         # micro-batches per epoch (None = whole dataset)
    GRADIENT_ACCUMULATION_STEPS = 4
    WARMUP_STEPS = 500                  # optimizer steps
    COOLDOWN_STEPS = 1000               # optimizer steps
    EVAL_EVERY = 500
    LOG_EVERY = 25
    RESUME = True

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Checkpoint paths (checks Drive is really mounted)
# ============================================================
CKPT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive is not mounted — saving checkpoints locally (lost if the runtime disconnects).")
    CKPT_DIR = "gqa_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
suffix = "_debug" if DEBUG_MODE else ""
BEST_PATH = f"{CKPT_DIR}/gqa_best{suffix}.pt"
LAST_PATH = f"{CKPT_DIR}/gqa_last{suffix}.pt"

# ============================================================
# 3. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          collate_fn=custom_collate_fn)
val_blocks = min(len(val_dataset), 400)
val_loader = DataLoader(Subset(val_dataset, range(val_blocks)), batch_size=BATCH_SIZE,
                        shuffle=False, collate_fn=custom_collate_fn)

micro_per_epoch = min(len(train_loader), MAX_STEPS_PER_EPOCH) if MAX_STEPS_PER_EPOCH else len(train_loader)
steps_per_epoch = max(1, micro_per_epoch // GRADIENT_ACCUMULATION_STEPS)   # optimizer steps
total_steps = steps_per_epoch * EPOCHS
tokens_per_step = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS * BLOCK_LEN

print(f"Train blocks: {len(train_dataset):,} | Val blocks used: {val_blocks}")
print(f"Micro-batches/epoch: {micro_per_epoch} | Optimizer steps/epoch: {steps_per_epoch} | Total: {total_steps}")
print(f"Tokens/optimizer step: {tokens_per_step:,} | Tokens seen in total: {total_steps * tokens_per_step / 1e6:.0f}M")

# ============================================================
# 4. Optimizer & schedule (step counts are in OPTIMIZER steps)
# ============================================================
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]    # norm weights
optimizer = AdamW(
    [{"params": decay_params, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95)
)

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step + 1) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 5. Evaluation + checkpoint helpers
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

def save_ckpt(path, epoch):
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(), "global_step": global_step, "epoch": epoch,
                "best_val": best_val, "val_loss_history": val_loss_history}, path)

global_step, best_val, start_epoch, val_loss_history = 0, float("inf"), 0, []
if RESUME and os.path.exists(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    global_step, best_val, val_loss_history = ck["global_step"], ck["best_val"], ck["val_loss_history"]
    start_epoch = min(global_step // steps_per_epoch, EPOCHS)
    print(f"Resumed from optimizer step {global_step} (starting at epoch {start_epoch + 1})")

# ============================================================
# 6. Training loop
# ============================================================
model.train()
print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

t0, start_global = time.time(), global_step

for epoch in range(start_epoch, EPOCHS):
    if global_step >= total_steps:
        break
    progress_bar = tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad(set_to_none=True)

    epoch_loss_sum = torch.tensor(0.0, device=device)
    epoch_micro = 0
    win_loss_sum = torch.tensor(0.0, device=device)
    win_micro = 0

    for step, batch in enumerate(train_loader):
        if step >= micro_per_epoch or global_step >= total_steps:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        micro_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS    # stays on GPU, no sync
        epoch_loss_sum += micro_loss; epoch_micro += 1
        win_loss_sum += micro_loss;   win_micro += 1

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1
            progress_bar.update(1)

            if global_step % LOG_EVERY == 0:
                elapsed = time.time() - t0
                done = global_step - start_global
                tok_s = done * tokens_per_step / elapsed
                eta_h = (total_steps - global_step) * elapsed / max(1, done) / 3600
                mem = torch.cuda.max_memory_allocated() / 1e9 if device.type == "cuda" else 0.0
                progress_bar.set_postfix({
                    "loss": f"{(win_loss_sum / win_micro).item():.4f}",
                    "val": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                    "lr": f"{scheduler.get_last_lr()[0]:.6f}",
                    "tok/s": f"{tok_s:,.0f}",
                    "eta_h": f"{eta_h:.2f}",
                    "mem_GB": f"{mem:.1f}",
                })
                win_loss_sum = torch.tensor(0.0, device=device); win_micro = 0

            if global_step % EVAL_EVERY == 0:
                v = evaluate()
                val_loss_history.append(v)
                print(f"\nstep {global_step}: val_loss={v:.4f} (ppl {math.exp(v):.1f})")
                if v < best_val:
                    best_val = v
                    save_ckpt(BEST_PATH, epoch)
                save_ckpt(LAST_PATH, epoch)

    progress_bar.close()
    avg_train_loss = (epoch_loss_sum / max(1, epoch_micro)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    if avg_val_loss < best_val:
        best_val = avg_val_loss
        save_ckpt(BEST_PATH, epoch + 1)
    save_ckpt(LAST_PATH, epoch + 1)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
          f"(ppl {math.exp(avg_val_loss):.1f})\n")

print("✅ Training complete!")
print(f"Best val loss: {best_val:.4f} | Best checkpoint: {BEST_PATH}")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

# Reload the best weights before generating / evaluating:
# model.load_state_dict(torch.load(BEST_PATH, map_location=device)["model"])

Train blocks: 190,794 | Val blocks used: 400
Micro-batches/epoch: 20000 | Optimizer steps/epoch: 5000 | Total: 20000
Tokens/optimizer step: 32,768 | Tokens seen in total: 655M
Using bf16: True
Resumed from optimizer step 20000 (starting at epoch 5)

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...
✅ Training complete!
Best val loss: 2.1949 | Best checkpoint: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt


**Explanation (code cell 18).** Identical to code cell 17 of this notebook (same code, same behaviour); see the explanation under that cell.

In [ ]:
import torch
ck = torch.load(BEST_PATH, map_location="cpu")
out = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt"
torch.save(ck["model"], out)
print(f"Saved weights-only model to {out} (best val loss {ck['best_val']:.4f})")

# To load later, after building the model with the GQA model cell:
# model.load_state_dict(torch.load(out, map_location=device))

Saved weights-only model to /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt (best val loss 2.1949)


**Code cell 19. What this cell does.** Extracts just the model weights from the best checkpoint and saves them as a small file.

**Line by line.** `torch.load(BEST_PATH)` reads the full checkpoint (which also holds optimizer state); `torch.save(ck["model"], out)` writes only the weights to `gqa_final_weights.pt` on Drive; the `print` shows the path and the best validation loss. Later phases load this file.

**Why.** The optimizer state roughly triples the file size and is useless for inference or fine-tuning.

In [ ]:
import os, torch

WEIGHTS_PATH = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt"
assert os.path.exists(WEIGHTS_PATH), f"Weights file not found: {WEIGHTS_PATH}"
for name in ["CustomDecoderLM", "tokenizer", "device"]:
    assert name in globals(), f"'{name}' is not defined, so run the loader cell and the GQA model cell first."

# Rebuild the exact architecture the weights were trained with
model = CustomDecoderLM(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=768, n_layer=10, n_head=12, n_kv_head=4,
    max_seq_len=2048, tie_weights=True
).to(device)

state = torch.load(WEIGHTS_PATH, map_location=device)
model.load_state_dict(state, strict=True)    # strict=True fails loudly on any shape or key mismatch
model.eval()
print(f"Loaded {WEIGHTS_PATH} | {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

# Quick sanity check: a prompt should continue with plausible legal text
ids = tokenizer.encode("The Supreme Court held that").ids
x = torch.tensor([ids], device=device)
with torch.no_grad():
    for _ in range(20):
        nxt = model(x)["logits"][:, -1].argmax(-1, keepdim=True)
        x = torch.cat([x, nxt], dim=1)
print(tokenizer.decode(x[0].tolist()))

Loaded /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_final_weights.pt | 87.51M parameters
The Supreme Court held that the High Court was not right in holding that the order of the Tribunal was not a nullity. The


**Code cell 20. What this cell does.** Rebuilds the model and reloads the final weights, then generates a few tokens as a sanity check.

**Line by line**
- Asserts the weights file exists and that `CustomDecoderLM`, `tokenizer`, `device` are defined.
- Builds the exact architecture used in training; `load_state_dict(state, strict=True)` fails if any name or shape differs, which catches mismatches early.
- `model.eval()` turns off training behaviour.
- The check encodes `"The Supreme Court held that"`, then 20 times takes the most likely next token (`argmax`, i.e. greedy decoding) and appends it, and prints the decoded text. Plausible legal wording means the weights loaded correctly.

## 12. Perplexity

In [ ]:
import math

# After your training loop finishes:
final_val_loss = val_loss_history[-1] # Grabs the last epoch's validation loss
perplexity = math.exp(final_val_loss)

print(f"📊 Final Validation Loss: {final_val_loss:.6f}")
print(f"📈 Final Validation Perplexity: {perplexity:.4f}")

**Code cell 21. What this cell does.** Converts the final validation loss to perplexity.

**Line by line.** `final_val_loss` is the last entry of `val_loss_history`; `perplexity = exp(loss)` is the model's average "effective branching factor" per token (lower is better). Both are printed.

## 13. Plot Training and Validation Loss

## 16. Load Test Data

In [ ]:
from huggingface_hub import hf_hub_download

# Make sure filename points to the correct nested directory path
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file downloaded to:", test_file_path)

Test file downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


**Code cell 22. What this cell does.** Downloads the Phase-1 test file.

**Line by line.** `hf_hub_download` with `TEST_FILE` and the token returns the local path of `test.json`, which is printed. Test prompts are used only to generate text, never for training.

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


**Code cell 23. What this cell does.** Reads the test file into memory.

**Line by line.** `json.load` parses it into `test_data`; the number of test examples is printed.

## 18. Load Final Model for Generation

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. Find and Load Model Weights ---
model_files = glob.glob("**/pytorch_model.bin", recursive=True)
if not model_files:
    model_files = glob.glob("/content/**/*.bin", recursive=True)

if not model_files:
    raise FileNotFoundError("Could not find 'pytorch_model.bin' weights file in the VM.")

state_dict_path = model_files[0]
print(f"✅ Found model weights at: {state_dict_path}")

# --- 2. Instantiate with Checkpoint's Vocab Size (50257) ---
generation_model = CustomDecoderLM(
    vocab_size=50257,  # Matches the checkpoint shape from training
    d_model=768,
    n_layer=8,
    n_head=12,
    max_seq_len=1024
)

generation_model.load_state_dict(torch.load(state_dict_path, map_location="cpu"))

# --- 3. Move to GPU & Eval Mode ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print(f"🚀 Device: {device}")
print(f"📊 Model Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

✅ Found model weights at: drive/MyDrive/CS781_Capstone_Project_Phase_1/pytorch_model.bin
🚀 Device: cuda
📊 Model Parameters: 95.23M


**Code cell 24. What this cell does.** An early (template) cell that loads a model checkpoint for generation from a `pytorch_model.bin` file found on the machine.

**Line by line.** It searches for `pytorch_model.bin`, builds `CustomDecoderLM` with the vocabulary size and shape recorded in that old checkpoint (50,257 tokens, 8 layers, 1,024 context), loads the weights, moves the model to the GPU, and prints the parameter count.

**Note.** This cell belongs to an earlier template and does not match our final model (32,000 vocabulary, 10 layers, 2,048 context). The final model is loaded in the cell that reads `gqa_final_weights.pt`. It is kept so the notebook shows the history; do not run it.

In [ ]:
# Load your saved weights before running generation
checkpoint_path = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.bin"

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    print("✅ Successfully loaded pretrained model weights from Google Drive!")
else:
    print("⚠️ Warning: Checkpoint path not found. Model is currently using random initialization weights.")

✅ Successfully loaded pretrained model weights from Google Drive!


**Code cell 25. What this cell does.** Loads a previously saved model file from Drive if it exists.

**Line by line.** If the path exists, `load_state_dict` fills the model and `model.eval()` is called; otherwise a warning says the model still has random weights.

**Note.** The path points to the MLA model's weights from an earlier notebook, so it is irrelevant for the GQA run; it is another leftover of the template.

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
import os, math, torch

CKPT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints"
for f in sorted(os.listdir(CKPT_DIR)):
    print(f"{f:28s} {os.path.getsize(os.path.join(CKPT_DIR, f)) / 1e6:7.0f} MB")

ck = torch.load(f"{CKPT_DIR}/gqa_best.pt", map_location="cpu")
print(f"\nBest val loss: {ck['best_val']:.4f} (perplexity {math.exp(ck['best_val']):.1f})")
print(f"Optimizer steps completed: {ck['global_step']}")
print("Val loss history:", [round(v, 3) for v in ck["val_loss_history"]])

gqa_best.pt                     1050 MB
gqa_final_weights.pt             350 MB
gqa_last.pt                     1050 MB
gqa_last_debug.pt                  0 MB

Best val loss: 2.1949 (perplexity 9.0)
Optimizer steps completed: 20000
Val loss history: [4.366, 3.653, 3.269, 3.02, 2.883, 2.799, 2.735, 2.681, 2.646, 2.61, 2.61, 2.58, 2.552, 2.538, 2.523, 2.498, 2.483, 2.462, 2.454, 2.444, 2.428, 2.428, 2.41, 2.404, 2.4, 2.393, 2.386, 2.377, 2.36, 2.358, 2.354, 2.347, 2.347, 2.337, 2.345, 2.335, 2.33, 2.328, 2.321, 2.311, 2.31, 2.245, 2.195]


**Code cell 26. What this cell does.** Lists the checkpoint files and prints the training summary stored inside the best checkpoint.

**Line by line.** It lists every file in the checkpoint folder with its size in MB; loads `gqa_best.pt`; prints the best validation loss with its perplexity, the number of optimizer steps completed, and the validation-loss history (one number per evaluation). This is where the Phase-1 numbers for the report come from.

In [ ]:
import os, json, time
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# ============================================================
# 0. Settings
# ============================================================
LIMIT = None           # 10 = timed test on 10 prompts. Set to None for the full 1000 run.
LOAD_WEIGHTS = True
CKPT_PATH = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt"
OUT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
PRESET = "sample_top_p"   # which one to use is decided by comparing on held-out text, not guessed
SEED = 42
USE_BF16 = True
TARGET_WORDS = 128

PRESETS = {
    "greedy":       dict(do_sample=False, temperature=1.0, top_k=0,  top_p=1.0, repetition_penalty=1.0, no_repeat_ngram=4),
    "sample_top_p": dict(do_sample=True,  temperature=0.7, top_k=0,  top_p=0.9, repetition_penalty=1.1, no_repeat_ngram=0),
    "sample_top_k": dict(do_sample=True,  temperature=0.7, top_k=40, top_p=1.0, repetition_penalty=1.1, no_repeat_ngram=0),
}
cfg = PRESETS["greedy"]

# ============================================================
# 1. Guards + load trained weights
# ============================================================
for name in ["tokenizer", "model", "device"]:
    assert name in globals(), f"'{name}' is not defined, so run the setup cells first."
assert hasattr(model.layers[0].attn, "n_kv_head"), "The model in memory isn't the GQA model. Rebuild it with the GQA model cell."
assert tokenizer.decode(tokenizer.encode("the court held that").ids).strip() != "", "Tokenizer sanity check failed."

if LOAD_WEIGHTS:
    assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"
    ck = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ck["model"] if isinstance(ck, dict) and "model" in ck else ck)
    if isinstance(ck, dict) and "best_val" in ck:
        print(f"Loaded {CKPT_PATH} (best val loss {ck['best_val']:.4f})")
model.eval()

# ============================================================
# 2. Test data (reuses test_data if it's already loaded)
# ============================================================
if "test_data" not in globals():
    from huggingface_hub import hf_hub_download
    p = hf_hub_download(repo_id=HF_DATASET_REPO, filename="Phase_1_Pretraining/Test/test.json",
                        repo_type="dataset", token=HF_TOKEN)
    with open(p, "r", encoding="utf-8") as f:
        test_data = json.load(f)
print("Number of test examples:", len(test_data))

# ============================================================
# 3. Generation (KV cache, top-k / top-p, repetition control)
# ============================================================
eos_id = tokenizer.token_to_id("<|endoftext|>")
pad_id = tokenizer.token_to_id("[PAD]")
assert eos_id is not None, "<|endoftext|> missing from tokenizer"
BAN_IDS = [i for i in (eos_id, pad_id) if i is not None]

MAX_NEW_TOKENS = int(TARGET_WORDS * 2.2)
MAX_PROMPT_TOKENS = model.max_seq_len - MAX_NEW_TOKENS     # prompt + generation must fit the RoPE table

def pick_token(logits, generated, blocked, cfg):
    logits = logits.float().clone()
    logits[BAN_IDS] = float("-inf")                         # never stop early
    rp = cfg["repetition_penalty"]
    if rp != 1.0 and generated:                             # penalise only tokens generated so far
        idx = torch.tensor(sorted(set(generated)), device=logits.device)
        sel = logits[idx]
        logits[idx] = torch.where(sel > 0, sel / rp, sel * rp)
    if blocked:
        logits[list(blocked)] = float("-inf")
    if not cfg["do_sample"]:
        return logits.argmax().view(1)
    logits = logits / cfg["temperature"]
    if cfg["top_k"] > 0:
        kth = torch.topk(logits, min(cfg["top_k"], logits.numel())).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
    if cfg["top_p"] < 1.0:
        sl, si = torch.sort(logits, descending=True)
        pr = F.softmax(sl, dim=-1)
        sl = sl.masked_fill((pr.cumsum(-1) - pr) > cfg["top_p"], float("-inf"))
        logits = torch.full_like(logits, float("-inf")).scatter(0, si, sl)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate_continuation(prompt_text, cfg):
    ids = tokenizer.encode(prompt_text).ids[-MAX_PROMPT_TOKENS:]       # keep the END of long prompts
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    n = cfg["no_repeat_ngram"]
    ngram_seen, generated = {}, []
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(USE_BF16 and device.type == "cuda")):
        out = model(input_ids, past_key_values=None)                    # prime the KV cache with the prompt
        for step in range(MAX_NEW_TOKENS):
            logits = out["logits"][0, -1]
            blocked = ngram_seen.get(tuple(generated[-(n - 1):])) if (n > 1 and len(generated) >= n - 1) else None
            nxt = pick_token(logits, generated, blocked, cfg)
            tok = nxt.item()
            generated.append(tok)
            if n > 1 and len(generated) >= n:
                ngram_seen.setdefault(tuple(generated[-n:-1]), set()).add(tok)
            if len(generated) % 10 == 0 and len(tokenizer.decode(generated).split()) >= TARGET_WORDS:
                break
            if step == MAX_NEW_TOKENS - 1:
                break
            out = model(nxt.view(1, 1), past_key_values=out["past_key_values"])
    return " ".join(tokenizer.decode(generated).split()[:TARGET_WORDS])

# ============================================================
# 4. Run (with resume for the full run)
# ============================================================
torch.manual_seed(SEED)
items = list(test_data.items())
if LIMIT:
    items = items[:LIMIT]

drive_ok = os.path.isdir(OUT_DIR)
PARTIAL_PATH = os.path.join(OUT_DIR, "prediction_partial.json")
sig = {"preset": PRESET, "cfg": cfg, "ckpt_mtime": os.path.getmtime(CKPT_PATH) if os.path.exists(CKPT_PATH) else None}

predictions = {}
if LIMIT is None and drive_ok and os.path.exists(PARTIAL_PATH):
    with open(PARTIAL_PATH, "r", encoding="utf-8") as f:
        saved = json.load(f)
    if saved.get("sig") == sig:                     # only resume if preset AND checkpoint are unchanged
        predictions = saved["preds"]
        print(f"Resuming: {len(predictions)} predictions already done.")

t0, n_done_now = time.time(), 0
for example_id, record in tqdm(items, desc="Generating predictions"):
    key = str(example_id)
    if key in predictions:
        continue
    prompt_text = record["text"] if isinstance(record, dict) else record
    try:
        text = generate_continuation(prompt_text, cfg)
    except Exception as e:
        print(f"⚠️ Generation failed for ID {key}: {e}")
        text = ""
    predictions[key] = {"text": text}
    n_done_now += 1
    if LIMIT is None and drive_ok and n_done_now % 100 == 0:
        with open(PARTIAL_PATH, "w", encoding="utf-8") as f:
            json.dump({"sig": sig, "preds": predictions}, f, ensure_ascii=False)

elapsed = time.time() - t0
if n_done_now:
    print(f"\n{elapsed:.0f}s for {n_done_now} examples ({elapsed / n_done_now:.2f}s each)")
    if LIMIT:
        print(f"Estimated full run for {len(test_data)} prompts: {elapsed / n_done_now * len(test_data) / 60:.1f} minutes")

# ============================================================
# 5. Checks before saving
# ============================================================
texts = [predictions[str(k)]["text"] for k, _ in items]
wc = [len(t.split()) for t in texts]
uniq = [len(set(t.split())) / max(1, len(t.split())) for t in texts]
print(f"Empty predictions:     {sum(1 for t in texts if not t.strip())} / {len(texts)}")
print(f"Under {TARGET_WORDS} words:        {sum(1 for c in wc if c < TARGET_WORDS)} / {len(texts)}")
print(f"Avg unique-word ratio: {sum(uniq) / len(uniq):.2f} (heuristic; very low values suggest repetition loops)")
print(f"\n--- Sample prediction (ID {items[0][0]}) ---\n{texts[0][:500]}")
if LIMIT is None:
    assert set(predictions) == set(str(k) for k in test_data), "Some test IDs are missing from predictions."

# ============================================================
# 6. Save (a subset test never overwrites the real prediction.json)
# ============================================================
fname = "prediction.json" if LIMIT is None else "prediction_test_subset.json"
with open(fname, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
print(f"\nSaved {len(predictions)} predictions to '{fname}'")
if drive_ok:
    import shutil
    shutil.copy(fname, os.path.join(OUT_DIR, fname))
    print(f"Copied to Drive: {os.path.join(OUT_DIR, fname)}")
# To download to your laptop:
# from google.colab import files; files.download(fname)

Loaded /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt (best val loss 2.1949)
Number of test examples: 1000


Generating predictions:   0%|          | 0/1000 [00:00<?, ?it/s]


2255s for 1000 examples (2.25s each)
Empty predictions:     0 / 1000
Under 128 words:        6 / 1000
Avg unique-word ratio: 0.47 (heuristic; very low values suggest repetition loops)

--- Sample prediction (ID 1) ---
a settled principle of law that the writ petition is not to be dismissed on the ground of laches. The writ petition is to be dismissed as having become infructuous on the ground that the petitioner has not approached the Court within reasonable time. The delay in challenging the acquisition proceedings is fatal and writ petition is liable to be dismissed. The delay is fatal and the writ petition cannot be dismissed on that ground. The delay of one year in challenging the notification is fatal an

Saved 1000 predictions to 'prediction.json'
Copied to Drive: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/prediction.json


**Code cell 27. What this cell does.** Generates the Phase-1 test output: a 128-word continuation for each test prompt, saved as `prediction.json`.

**Section 0: settings.** `LIMIT = None` runs every prompt (set it to 10 for a timing test). `CKPT_PATH` is the best checkpoint. `PRESETS` are three decoding recipes: `greedy` (always the most likely token, with a ban on repeating any 4-gram), `sample_top_p` (temperature 0.7, nucleus sampling with p = 0.9, repetition penalty 1.1) and `sample_top_k` (top 40). `cfg` is the chosen recipe; the comment says the choice was made by comparison on held-out text. `TARGET_WORDS = 128` is the required length.

**Section 1: guards and weights.** Asserts that tokenizer, model and device exist, that the model is the GQA one (it has `n_kv_head`), and that the tokenizer round-trips. Loads the best checkpoint's weights and sets eval mode.

**Section 2: test data.** Reuses `test_data` if already loaded; otherwise downloads `test.json`.

**Section 3: generation.**
- `MAX_NEW_TOKENS = int(128 * 2.2)`: about 2.2 tokens per word is a safe upper bound. `MAX_PROMPT_TOKENS` is the context minus that, so prompt plus output fits the 2,048 RoPE table.
- `pick_token(logits, generated, blocked, cfg)` chooses the next token: it bans end-of-text and padding (so generation never stops early), applies the repetition penalty (positive logits of already-used tokens are divided, negative ones multiplied, so they get less likely), blocks tokens that would complete a repeated n-gram, and then either takes the argmax (greedy) or samples after dividing by the temperature and cutting to top-k and/or top-p.
- `generate_continuation(prompt_text, cfg)` encodes the prompt (keeping the end if too long), runs the model once to fill the KV cache, then loops: pick a token, append it, feed only that token with the cache. It stops when the decoded text reaches 128 words (checked every 10 tokens) or the token limit is hit, and returns exactly the first 128 words.

**Section 4: the run.** Seeds the generator, optionally truncates to `LIMIT`, and loads `prediction_partial.json` to resume only if the decoding preset and the checkpoint's modification time match (so stale partial results are never mixed in). Generation errors are caught and recorded as empty text rather than crashing the run. Every 100 examples the partial result is saved to Drive.

**Section 5: checks.** Counts empty predictions, predictions under 128 words, and the average unique-word ratio (a very low ratio signals a repetition loop), prints a sample, and asserts all test IDs are present.

**Section 6: save.** Writes `prediction.json` (or `prediction_test_subset.json` for a subset, so a partial test never overwrites the real file) and copies it to Drive.

## Fine Tuning

In [ ]:
import collections

def strict_scores(gold, pred, labels):
    """
    gold, pred: {doc_id: [[start, end, label, (text)], ...]}
    Exact match on (doc_id, start, end, label). Macro over `labels`.
    Returns dict with macro_p / macro_r / macro_f1, per-label dict, and micro_f1.
    """
    def to_set(d):
        out = set()
        for did, ents in d.items():
            for e in ents:
                out.add((str(did), int(e[0]), int(e[1]), e[2]))
        return out

    G, P = to_set(gold), to_set(pred)
    tp, gp, pp = collections.Counter(), collections.Counter(), collections.Counter()
    for x in G: gp[x[3]] += 1
    for x in P: pp[x[3]] += 1
    for x in G & P: tp[x[3]] += 1

    per = {}
    for lab in labels:
        p = tp[lab] / pp[lab] if pp[lab] else 0.0
        r = tp[lab] / gp[lab] if gp[lab] else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        per[lab] = {"p": p, "r": r, "f1": f, "support": gp[lab], "pred": pp[lab], "tp": tp[lab]}

    n = len(labels)
    macro_p = sum(v["p"] for v in per.values()) / n
    macro_r = sum(v["r"] for v in per.values()) / n
    macro_f1 = sum(v["f1"] for v in per.values()) / n
    TP, PP, GG = sum(tp.values()), sum(pp.values()), sum(gp.values())
    mp, mr = (TP / PP if PP else 0.0), (TP / GG if GG else 0.0)
    micro_f1 = 2 * mp * mr / (mp + mr) if (mp + mr) else 0.0
    return {"macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
            "micro_f1": micro_f1, "per_label": per}

def print_scores(res):
    print(f"MACRO  P={res['macro_p']:.4f}  R={res['macro_r']:.4f}  F1={res['macro_f1']:.4f}  | micro F1={res['micro_f1']:.4f}")
    print(f"{'label':10s} {'P':>7s} {'R':>7s} {'F1':>7s} {'gold':>6s} {'pred':>6s} {'tp':>6s}")
    for lab, v in sorted(res["per_label"].items(), key=lambda kv: -kv[1]["support"]):
        print(f"{lab:10s} {v['p']:7.3f} {v['r']:7.3f} {v['f1']:7.3f} {v['support']:6d} {v['pred']:6d} {v['tp']:6d}")

# ---------- self-test on toy data ----------
_L = ["JUDGE", "COURT", "DATE"]
_gold = {"d1": [[0, 5, "JUDGE", "Smith"], [10, 20, "COURT", "High Court"]],
         "d2": [[3, 9, "DATE", "1 May"]]}

# perfect predictions (with and without optional text field)
_r = strict_scores(_gold, {"d1": [[0, 5, "JUDGE"], [10, 20, "COURT", "x"]], "d2": [[3, 9, "DATE"]]}, _L)
assert abs(_r["macro_f1"] - 1.0) < 1e-9, _r

# boundary off by one and wrong label must both count as wrong
_r = strict_scores(_gold, {"d1": [[0, 6, "JUDGE"], [10, 20, "JUDGE"]], "d2": [[3, 9, "DATE"]]}, _L)
assert _r["per_label"]["JUDGE"]["tp"] == 0 and _r["per_label"]["COURT"]["tp"] == 0
assert _r["per_label"]["DATE"]["f1"] == 1.0
assert abs(_r["macro_f1"] - 1/3) < 1e-9, _r["macro_f1"]

# empty predictions -> zeros; duplicate predictions must not inflate counts
assert strict_scores(_gold, {}, _L)["macro_f1"] == 0.0
_r = strict_scores(_gold, {"d2": [[3, 9, "DATE"], [3, 9, "DATE"]]}, _L)
assert _r["per_label"]["DATE"]["pred"] == 1 and _r["per_label"]["DATE"]["p"] == 1.0
print("✅ strict scorer self-test passed")

✅ strict scorer self-test passed


**Code cell 28. What this cell does.** Defines the strict scorer used for the NER task in Phase 2 and tests it on toy data.

**`strict_scores(gold, pred, labels)`**
- `gold` and `pred` map a document id to a list of entities `[start, end, label, (text)]`.
- `to_set` turns each into a set of tuples `(doc_id, start, end, label)`. A prediction counts only if document, both boundaries and label all match exactly ("strict" matching). A set also removes duplicate predictions.
- `G & P` is the true positives. Counters hold, per label, the number of gold entities (`gp`), predicted entities (`pp`) and true positives (`tp`).
- For each label: precision = tp / predicted, recall = tp / gold, F1 = their harmonic mean (0 when undefined).
- Macro scores average the per-label values, so every entity type counts equally however rare it is. Micro F1 pools all counts first.

**`print_scores`** prints macro P/R/F1, micro F1 and a per-label table sorted by support.

**Self-test.** Perfect predictions give macro-F1 1.0 (and an optional text field is ignored); an off-by-one boundary or a wrong label counts as wrong (giving 1/3 here); empty predictions give 0; duplicated predictions do not raise precision above 1.

**Why.** The metric must match the task's official definition before any model is trained, otherwise dev scores mean nothing.

In [ ]:
import os, json, math, time, random, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

for name in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "strict_scores", "print_scores"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cell that defines it."

# ============================================================
# 0. Config
# ============================================================
WINDOW, STRIDE = 1024, 768            # tokens per window / step (256 overlap)
VAL_FRACTION, SEED = 0.15, 42
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0

P2_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2"
if not os.path.isdir("/content/drive/MyDrive"):
    print("⚠️ Drive not mounted — saving Phase-2 files locally (lost if the runtime disconnects).")
    P2_DIR = "CS781_Capstone_Project_Phase_2"
os.makedirs(P2_DIR, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if use_bf16 else torch.float16
PAD_ID = tokenizer.token_to_id("[PAD]")
PAD_ID = 1 if PAD_ID is None else PAD_ID
def amp(): return torch.amp.autocast(device_type="cuda", dtype=AMP_DTYPE)

# ============================================================
# 1. Load Phase-2 files
# ============================================================
BASE = "Phase_2_LNER"
def _path(fn): return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"{BASE}/{fn}", repo_type="dataset", token=HF_TOKEN)
def _json(fn):
    with open(_path(fn), "r", encoding="utf-8") as f: return json.load(f)

text1, gold1 = _json("Train/fold1-text.json"), _json("Train/fold1-gold.json")
text2, gold2 = _json("Train/fold2-text.json"), _json("Train/fold2-gold.json")
text3 = _json("Test/fold3-text.json")
with open(_path("Train/labels.txt"), "r", encoding="utf-8") as f:
    LABELS = [l.split()[0] for l in f if l.strip()]
print(f"Labels ({len(LABELS)}): {LABELS}")
assert len(LABELS) == 12, "labels.txt did not parse into 12 labels — paste me the first lines of the file."

def as_text(v): return v.get("text") if isinstance(v, dict) else v

def parse_entity(e):
    if isinstance(e, (list, tuple)):
        return int(e[0]), int(e[1]), e[2], (e[3] if len(e) > 3 else None)
    if isinstance(e, dict):
        g = lambda *ks: next((e[k] for k in ks if k in e), None)
        return (int(g("start", "start_char", "begin")), int(g("end", "end_char")),
                g("label", "type", "entity"), g("text", "entity_text", "span"))
    raise ValueError(f"Unknown annotation format: {e!r}")

def doc_entities(gold, doc_id):
    v = gold.get(doc_id, [])
    if isinstance(v, dict):
        v = next((x for x in v.values() if isinstance(x, list)), [])
    return [parse_entity(e) for e in v]

# ---- offset convention: is the gold `end` exclusive (python slice) or inclusive? ----
excl = incl = 0
for tx, gd in [(text1, gold1), (text2, gold2)]:
    for did, v in tx.items():
        t = as_text(v)
        for s, e, lab, et in doc_entities(gd, did):
            if et is None: continue
            if t[s:e] == et: excl += 1
            elif t[s:e + 1] == et: incl += 1
print(f"Offset test -> text[s:e]==entity: {excl} | text[s:e+1]==entity: {incl}")
if excl == 0 and incl == 0:
    print("⚠️ Could not verify the convention (no entity-text field matched). Assuming END-EXCLUSIVE.")
END_EXCL = excl >= incl
print("Gold end offset is", "EXCLUSIVE" if END_EXCL else "INCLUSIVE")

# ============================================================
# 2. BIO label space + token labelling / span decoding
# ============================================================
L2I = {"O": 0}
for lab in LABELS:
    L2I["B-" + lab] = len(L2I); L2I["I-" + lab] = len(L2I)
I2L = {v: k for k, v in L2I.items()}
NL = len(L2I)
print(f"Classes: {NL} (O + B/I for each of {len(LABELS)} labels)")

def label_tokens(offsets, ents):
    """ents: [(start, end_exclusive, label)] -> BIO ids per token. Overlapping entities: first (longest) wins."""
    n = len(offsets)
    starts = np.array([o[0] for o in offsets]); ends = np.array([o[1] for o in offsets])
    y = [0] * n; conflicts = 0
    for s, e, lab in sorted(ents, key=lambda x: (x[0], -(x[1] - x[0]))):
        i = int(np.searchsorted(ends, s, side="right"))   # first token ending after s
        first = True
        while i < n and starts[i] < e:
            if ends[i] > starts[i]:
                if y[i] != 0: conflicts += 1
                else:
                    y[i] = L2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return y, conflicts

def decode_spans(text, offsets, tags):
    """Token tag ids -> [(start, end_exclusive, label)], whitespace trimmed."""
    spans, cur = [], None
    for i, t in enumerate(tags):
        a, b = offsets[i]
        if b <= a: continue
        name = I2L[int(t)]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        pre, lab = name.split("-", 1)
        if pre == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [a, b, lab]
        else:
            cur[1] = b
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:
        while s < e and text[s].isspace(): s += 1
        while e > s and text[e - 1].isspace(): e -= 1
        if e > s: out.append((s, e, lab))
    return out

def spans_out(text, offsets, tags):
    return [[int(s), int(e if END_EXCL else e - 1), lab, text[s:e]] for s, e, lab in decode_spans(text, offsets, tags)]

# ============================================================
# 3. Build docs, split by DOCUMENT, windows
# ============================================================
def build_docs(text_d, gold_d, prefix=""):
    docs = []
    for did, v in text_d.items():
        t = as_text(v); enc = tokenizer.encode(t)
        d = {"id": prefix + str(did), "text": t, "ids": enc.ids, "offsets": enc.offsets}
        if gold_d is not None:
            raw = doc_entities(gold_d, did)
            bad = {lab for _, _, lab, _ in raw if lab not in LABELS}
            assert not bad, f"Gold labels not in labels.txt: {bad}"
            d["gold_raw"] = [[s, e, lab] for s, e, lab, _ in raw]
            d["labels"], d["conflicts"] = label_tokens(
                d["offsets"], [(s, e if END_EXCL else e + 1, lab) for s, e, lab, _ in raw])
        docs.append(d)
    return docs

t0 = time.time()
all_docs = build_docs(text1, gold1, "f1:") + build_docs(text2, gold2, "f2:")
test_docs = build_docs(text3, None)
print(f"Tokenized {len(all_docs)} train docs + {len(test_docs)} test docs in {time.time()-t0:.0f}s")

order = list(range(len(all_docs))); random.Random(SEED).shuffle(order)
n_val = max(1, int(VAL_FRACTION * len(all_docs)))
val_docs = [all_docs[i] for i in order[:n_val]]
train_docs = [all_docs[i] for i in order[n_val:]]

def make_windows(n, W=WINDOW, S=STRIDE):
    if n <= W: return [0]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n: starts.append(n - W)
    return starts

train_windows = [(d["ids"][s:s + WINDOW], d["labels"][s:s + WINDOW])
                 for d in train_docs for s in make_windows(len(d["ids"]))]

tok_tr = sum(len(d["ids"]) for d in train_docs)
print(f"Train docs: {len(train_docs)} ({tok_tr:,} tokens) -> {len(train_windows)} windows of ≤{WINDOW} "
      f"(~{sum(len(x) for x,_ in train_windows)/1e6:.2f}M tokens/epoch)")
print(f"Val docs: {len(val_docs)} | Test docs: {len(test_docs)} | "
      f"docs longer than WINDOW: {sum(len(d['ids'])>WINDOW for d in all_docs)} | max tokens: {max(len(d['ids']) for d in all_docs)}")
print(f"Entity-token conflicts (overlapping gold): {sum(d['conflicts'] for d in all_docs)}")
val_cnt = collections.Counter(l for d in val_docs for _, _, l in d["gold_raw"])
print("Val entity counts:", dict(val_cnt.most_common()))

# class weights (sqrt inverse frequency, clipped)
cnt = np.bincount(np.concatenate([np.array(d["labels"]) for d in train_docs]), minlength=NL).astype(np.float64)
w = np.ones(NL)
if USE_CLASS_WEIGHTS:
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, CLASS_W_CLIP))
CLASS_W = torch.tensor(w, dtype=torch.float32, device=device) if USE_CLASS_WEIGHTS else None
print(f"O tokens: {100*cnt[0]/cnt.sum():.1f}% | class weights (min/max entity): {w[1:].min():.1f}/{w[1:].max():.1f}")

def collate_windows(items):
    L = max(len(x) for x, _ in items)
    X = torch.full((len(items), L), PAD_ID, dtype=torch.long)
    Y = torch.full((len(items), L), -100, dtype=torch.long)
    for i, (x, y) in enumerate(items):
        X[i, :len(x)] = torch.tensor(x); Y[i, :len(y)] = torch.tensor(y)
    return X, Y

# ============================================================
# 4. ORACLE round-trip: gold -> token labels -> spans -> strict score  (the ceiling)
# ============================================================
o_docs = train_docs + val_docs
o_gold = {d["id"]: d["gold_raw"] for d in o_docs}
o_pred = {d["id"]: spans_out(d["text"], d["offsets"], d["labels"]) for d in o_docs}
o_res = strict_scores(o_gold, o_pred, LABELS)
print("\n=== ORACLE (ceiling for any model with this tokenizer + labelling) ===")
print_scores(o_res)
_lost = []
for d in o_docs:
    got = {(s, e, l) for s, e, l, _ in o_pred[d["id"]]}
    for s, e, l in d["gold_raw"]:
        if (s, e, l) not in got and len(_lost) < 5: _lost.append((d["id"], s, e, l, d["text"][s:e if END_EXCL else e + 1]))
print("Examples of gold spans the round-trip cannot reproduce:", _lost)
assert o_res["macro_f1"] > 0.8, "Oracle ceiling too low — offsets/tokenizer alignment is off. Paste this output to me."

# ============================================================
# 5. NER wrapper around the GQA decoder (no change to the LM class)
# ============================================================
class NERModel(nn.Module):
    """Token classifier: GQA decoder backbone (causal) + linear head.
    shift=k: token i is classified from the hidden state at position i+k (k-token lookahead)."""
    def __init__(self, lm, d_model, n_labels, dropout=0.1, shift=0):
        super().__init__()
        self.lm = lm
        self.shift = shift
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)

    def forward(self, input_ids):
        lm = self.lm
        x = lm.token_embedding(input_ids)
        for layer in lm.layers:
            x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        out = self.head(self.drop(lm.norm(x)))
        k = self.shift
        if k > 0:   # position i now holds the prediction for token i (made from position i+k)
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

# ============================================================
# 6. Prediction (windows averaged) + test-file writer
# ============================================================
@torch.no_grad()
def predict_docs(ner, docs, batch=8):
    ner.eval(); out = {}
    for d in docs:
        n = len(d["ids"])
        probs = torch.zeros(n, NL, device=device)
        starts = make_windows(n)
        for i in range(0, len(starts), batch):
            chunk = starts[i:i + batch]
            seqs = [d["ids"][s:s + WINDOW] for s in chunk]
            L = max(len(q) for q in seqs)
            x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
            for j, q in enumerate(seqs): x[j, :len(q)] = torch.tensor(q)
            with amp(): lg = ner(x.to(device))
            p = F.softmax(lg.float(), dim=-1)
            k = ner.shift
            for j, (s, q) in enumerate(zip(chunk, seqs)):
                valid = len(q) if (s + len(q) >= n) else len(q) - k   # last k tokens of a window have no lookahead
                probs[s:s + valid] += p[j, :valid]
        out[d["id"]] = spans_out(d["text"], d["offsets"], probs.argmax(-1).tolist())
    return out

def write_test_predictions(ner, name):
    preds = predict_docs(ner, test_docs)
    path = f"{P2_DIR}/prediction_{name}.json"
    with open(path, "w", encoding="utf-8") as f: json.dump(preds, f, ensure_ascii=False)
    print(f"Saved {len(preds)} test docs / {sum(len(v) for v in preds.values())} entities -> {path}")
    return path

# ============================================================
# 7. Trainer (optionally with distillation from a teacher NER model)
# ============================================================
def train_ner(ner, name, teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
              backbone_lr=1e-4, head_lr=1e-3, batch_size=8, weight_decay=0.01,
              warmup_frac=0.06, debug=False):
    tw = train_windows[:batch_size * 10] if debug else train_windows
    vdocs = val_docs[:10] if debug else val_docs
    if debug: epochs = 1; name = name + "_debug"
    vgold = {d["id"]: d["gold_raw"] for d in vdocs}
    ner.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)

    groups = {}
    for n_, p in ner.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
         for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / batch_size); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(optimizer, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))

    n_train = sum(p.numel() for p in ner.parameters() if p.requires_grad)
    print(f"[{name}] trainable params: {n_train/1e6:.2f}M | windows: {len(tw)} | steps/epoch: {spe} | "
          f"KD: {'alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None and alpha > 0 else 'off'}")
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()

    best_f1, best_res, bad, history = -1.0, None, 0, []
    best_path = f"{P2_DIR}/{name}_best.pt"; t0 = time.time()

    if teacher is not None: assert teacher.shift == ner.shift, "teacher and student must use the same shift"
    for ep in range(epochs):
        ner.train(); idx = list(range(len(tw))); random.shuffle(idx)
        loss_sum = torch.tensor(0.0, device=device); nb = 0
        bar = tqdm(range(0, len(idx), batch_size), desc=f"{name} ep {ep+1}/{epochs}")
        for i in bar:
            X, Y = collate_windows([tw[j] for j in idx[i:i + batch_size]])
            X, Y = X.to(device), Y.to(device)
            if ner.shift > 0: Y[:, -ner.shift:] = -100        # the last k columns have no lookahead
            with amp(): lg = ner(X)
            lg = lg.float()
            loss = F.cross_entropy(lg.reshape(-1, NL), Y.reshape(-1), weight=CLASS_W, ignore_index=-100)
            if teacher is not None and alpha > 0:
                with torch.no_grad(), amp(): tl = teacher(X)
                m = Y != -100
                kd = F.kl_div(F.log_softmax(lg[m] / T, -1), F.softmax(tl.float()[m] / T, -1),
                              reduction="batchmean") * (T * T)
                loss = (1 - alpha) * loss + alpha * kd
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(ner.parameters(), 1.0)
            optimizer.step(); sched.step()
            loss_sum += loss.detach(); nb += 1
            if nb % 20 == 0: bar.set_postfix(loss=f"{(loss_sum / nb).item():.4f}")

        res = strict_scores(vgold, predict_docs(ner, vdocs), LABELS)
        history.append({"epoch": ep + 1, "train_loss": (loss_sum / max(1, nb)).item(), "macro_f1": res["macro_f1"]})
        print(f"epoch {ep+1}: train loss {history[-1]['train_loss']:.4f} | val macro P/R/F1 = "
              f"{res['macro_p']:.4f}/{res['macro_r']:.4f}/{res['macro_f1']:.4f} | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best_f1:
            best_f1, best_res, bad = res["macro_f1"], res, 0
            torch.save({"model": ner.state_dict(), "epoch": ep + 1, "val_macro_f1": best_f1}, best_path)
        else:
            bad += 1
            if bad >= patience: print(f"Early stop (no improvement for {patience} epochs)."); break

    ner.load_state_dict(torch.load(best_path, map_location=device, weights_only=False)["model"])
    mem = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print(f"✅ [{name}] best val macro-F1 {best_f1:.4f} | {(time.time()-t0)/60:.1f} min | peak mem {mem:.1f} GB | saved {best_path}")
    return {"res": best_res, "history": history, "best_path": best_path, "minutes": (time.time() - t0) / 60,
            "peak_gb": mem, "trainable_params": n_train}

print("\n✅ Phase-2 toolkit ready.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Labels (12): ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
Offset test -> text[s:e]==entity: 8393 | text[s:e+1]==entity: 0
Gold end offset is EXCLUSIVE
Classes: 25 (O + B/I for each of 12 labels)
Tokenized 70 train docs + 35 test docs in 2s
Train docs: 60 (399,913 tokens) -> 530 windows of ≤1024 (~0.54M tokens/epoch)
Val docs: 10 | Test docs: 35 | docs longer than WINDOW: 70 | max tokens: 23710
Entity-token conflicts (overlapping gold): 0
Val entity counts: {'PREC': 212, 'STAT': 171, 'DATE': 157, 'AUTH': 150, 'COURT': 97, 'RESP': 89, 'CASENO': 75, 'APP': 66, 'JUDGE': 32, 'WIT': 23, 'R.COUNSEL': 22, 'A.COUNSEL': 20}
O tokens: 90.8% | class weights (min/max entity): 6.1/10.0

=== ORACLE (ceiling for any model with this tokenizer + labelling) ===
MACRO  P=0.9866  R=0.9866  F1=0.9866  | micro F1=0.9819
label            P       R      F1   gold   pred     tp
DATE         1.000   1.000   1.000   1575   1575   1575
STAT         0.

**Code cell 29. What this cell does.** The Phase-2 toolkit for legal NER (named-entity recognition with 12 entity types). It loads the data, converts character spans to token labels and back, defines the model wrapper, the prediction code and the trainer. Everything NER-related in later cells calls functions from this cell.

**Setup (sections 0-1)**
- Asserts that the tokenizer, device, dataset id, token and the scorer exist.
- `WINDOW, STRIDE = 1024, 768`: documents are cut into windows of 1,024 tokens that move forward 768 tokens (256 overlap). `VAL_FRACTION = 0.15` of the documents become a validation set. `USE_CLASS_WEIGHTS` with a clip of 10 weights rare entity classes more.
- `P2_DIR`: the Phase-2 folder on Drive. Seeds for `random`, `numpy` and `torch` make runs repeatable. `AMP_DTYPE` is bfloat16 when supported, else float16, and `amp()` returns the autocast context.
- `PAD_ID` is the padding token id.
- `_path`/`_json` download and parse a file from the `Phase_2_LNER` folder. Folds 1 and 2 (text + gold annotations) are the training data, fold 3 is the unlabeled test text, and `labels.txt` gives the 12 label names (asserted to be 12).
- `parse_entity` / `doc_entities` accept several annotation formats (lists or dictionaries with different key names) and return `(start, end, label, text)`.
- **Offset convention check.** For every gold entity with a text field it tests whether `text[start:end]` equals the entity (end exclusive) or `text[start:end+1]` does (end inclusive), counts both, and sets `END_EXCL` accordingly. This avoids an off-by-one error that would silently ruin every score.

**BIO labels (section 2)**
- Each entity label becomes two classes, `B-label` (first token of an entity) and `I-label` (continuation), plus `O` (outside): 12 x 2 + 1 = 25 classes.
- `label_tokens(offsets, ents)`: for each gold entity (sorted by start, longest first) it finds the first token that ends after the entity starts (`searchsorted`), then labels all tokens that begin before the entity ends: `B-` for the first, `I-` for the rest. Overlapping entities are counted as conflicts and the earlier one wins.
- `decode_spans` does the reverse: it merges consecutive tokens into spans (a `B-` or a label change starts a new span), then trims whitespace from both ends of each span. `spans_out` converts the spans to the output format, respecting the end convention.

**Data (section 3)**
- `build_docs` tokenizes each text with offsets, stores the gold entities and token labels. The split into train and validation is by document (a shuffled order with a fixed seed), so no document appears in both.
- `make_windows(n)` gives the start index of each window: every `STRIDE` tokens, plus a final window aligned to the end so the last tokens are never missed.
- It prints the sizes, how many documents exceed one window, the number of overlapping-entity conflicts, and the validation entity counts.
- **Class weights**: for each entity class, `sqrt(count of O tokens / count of this class)` clipped to [1, 10], so a class 100 times rarer than O gets weight 10. This counters the fact that most tokens are `O`.
- `collate_windows` pads a batch to its longest window and puts -100 (ignored by the loss) in padded label positions.

**Oracle check (section 4).** It turns the gold annotation into token labels and back into spans and scores that against the gold. This is the ceiling for any model using this tokenizer and labelling scheme (no model can exceed it, because a span boundary inside a token cannot be reproduced). It asserts the ceiling is above 0.8; a low value would mean an alignment bug.

**Model wrapper (section 5): `NERModel`.** It reuses the pretrained decoder without changing the language-model class: embeds the tokens, runs the layers (with `past_kv=None`), applies the final norm, dropout, and a linear head to 25 classes. `shift=k` is an optional lookahead: because a causal decoder cannot see the right context, the label of token i is read from the hidden state k positions later, and the last k positions are padded with zeros. With `shift=0` the model is purely causal.

**Prediction (section 6).** `predict_docs` runs each document in windows, sums the softmax probabilities from overlapping windows into one array per token, takes the argmax, and decodes spans. For lookahead models the last k tokens of a window (which have no lookahead) are skipped unless the window is the document's last. `write_test_predictions` runs this on the test documents and saves `prediction_<name>.json`.

**Trainer (section 7): `train_ner`.**
- Optional knowledge distillation: if a `teacher` is passed with `alpha > 0`, its logits are used as soft targets.
- Two learning rates: `backbone_lr = 1e-4` for the pretrained layers, `head_lr = 1e-3` for the new head (a fresh head needs bigger steps). Parameters are grouped so only matrices get weight decay. AdamW with betas (0.9, 0.98), a 6% warm-up then linear decay, gradient clipping at 1.0. `debug=True` runs a tiny 1-epoch test.
- Each batch: forward under autocast, class-weighted cross-entropy (ignoring -100). With a teacher, the loss becomes `(1 - alpha) * CE + alpha * KD`, where KD is the KL divergence between the student's and the teacher's softened distributions (temperature T, multiplied by T squared to keep gradient scale).
- After each epoch it computes the strict macro-F1 on the validation documents, keeps the best checkpoint, and stops early after `patience` epochs without improvement.
- At the end it reloads the best checkpoint and returns the result dictionary: scores, history, path, minutes, peak GPU memory and the number of trainable parameters. Those numbers feed the report tables.

In [ ]:
for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "train_windows", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

DEBUG_MODE = False      # first run: tiny smoke test. Flip to False for the real run.

TEACHER_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt"
assert os.path.exists(TEACHER_CKPT), f"Checkpoint not found: {TEACHER_CKPT}"

# ---- load the pretrained Phase-1 model ----
lm = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                     n_kv_head=4, max_seq_len=2048, tie_weights=True)
ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
lm.load_state_dict(ck["model"] if "model" in ck else ck)
lm.to(device)

# sanity check: the loaded weights should give val loss ≈ 2.2 on Phase-1 val blocks
if "val_dataset" in globals():
    lm.eval(); ls = []
    with torch.no_grad():
        for i in range(0, 32, 4):
            ids = torch.stack([val_dataset[j]["input_ids"] for j in range(i, i + 4)]).to(device).long()
            with amp(): ls.append(lm(ids, labels=ids)["loss"].float().item())
    print(f"Loaded teacher: val loss ≈ {np.mean(ls):.4f} (expect ≈ 2.2)")

# ---- full fine-tuning ----
ner_teacher = NERModel(lm, d_model=768, n_labels=NL, dropout=0.1)
out_t = train_ner(ner_teacher, "ner_teacher_gqa", epochs=10, patience=3,
                  backbone_lr=1e-4, head_lr=1e-3, batch_size=8, debug=DEBUG_MODE)

print("\n=== Validation (strict) — best epoch ===")
print_scores(out_t["res"])
if not DEBUG_MODE:
    write_test_predictions(ner_teacher, "sub1_gqa_full_ft")
else:
    print("\n*** DEBUG_MODE smoke test finished. If it ran cleanly, set DEBUG_MODE=False and re-run. ***")

Loaded teacher: val loss ≈ 2.1824 (expect ≈ 2.2)
[ner_teacher_gqa] trainable params: 87.53M | windows: 530 | steps/epoch: 67 | KD: off


ner_teacher_gqa ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.1636 | val macro P/R/F1 = 0.1507/0.3003/0.1982 | 0.1 min


ner_teacher_gqa ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.1451 | val macro P/R/F1 = 0.1643/0.3477/0.2204 | 0.3 min


ner_teacher_gqa ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (7 outputs left out)

ner_teacher_gqa ep 7/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 7: train loss 0.0048 | val macro P/R/F1 = 0.2639/0.3307/0.2868 | 1.1 min


ner_teacher_gqa ep 8/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 8: train loss 0.0030 | val macro P/R/F1 = 0.2699/0.3267/0.2878 | 1.3 min
Early stop (no improvement for 3 epochs).
✅ [ner_teacher_gqa] best val macro-F1 0.3063 | 1.3 min | peak mem 17.8 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_teacher_gqa_best.pt

=== Validation (strict) — best epoch ===
MACRO  P=0.2725  R=0.3661  F1=0.3063  | micro F1=0.3704
label            P       R      F1   gold   pred     tp
PREC         0.176   0.269   0.213    212    324     57
STAT         0.610   0.696   0.650    171    195    119
DATE         0.867   0.955   0.909    157    173    150
AUTH         0.113   0.107   0.110    150    141     16
COURT        0.483   0.753   0.589     97    151     73
RESP         0.140   0.090   0.110     89     57      8
CASENO       0.143   0.400   0.211     75    210     30
APP          0.079   0.136   0.100     66    114      9
JUDGE        0.457   0.656   0.538     32     46     21
WIT          0.000   0.000   0.000     23     47      0
R.COU

**Code cell 30. What this cell does.** Fine-tunes the whole GQA model for NER (full fine-tuning) and writes the first test submission. This is the NER "teacher" that the distilled student later learns from.

**Line by line**
- The assert loop checks that the toolkit cell has been run (`train_ner`, `NERModel`, `write_test_predictions`, the data and constants).
- `DEBUG_MODE`: `True` runs a tiny smoke test first; `False` is the real run (it is `False` here).
- `TEACHER_CKPT`: the best Phase-1 language-model checkpoint.
- `lm = CustomDecoderLM(...)` builds the same architecture as in Phase 1 and `load_state_dict` fills it with the pretrained weights. This is the rule that every later model starts from our own Phase-1 weights.
- The sanity check evaluates 8 batches of Phase-1 validation blocks and prints the average loss; it should be about 2.2, proving the weights loaded correctly.
- `NERModel(lm, d_model=768, n_labels=NL, dropout=0.1)` adds the 25-class head on top.
- `train_ner(..., epochs=10, patience=3, backbone_lr=1e-4, head_lr=1e-3, batch_size=8)` trains all 87.5M parameters; training stops early after 3 epochs without validation improvement.
- Then it prints the per-label validation scores and, if not in debug mode, writes `prediction_sub1_gqa_full_ft.json`, the file for submission 1.

In [ ]:
def make_cw(clip):
    w = np.ones(NL)
    for c in range(1, NL): w[c] = float(np.clip(math.sqrt(cnt[0] / max(cnt[c], 1.0)), 1.0, clip))
    return torch.tensor(w, dtype=torch.float32, device=device)

def fresh_teacher_ner(dropout, shift):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck_ = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck_["model"] if "model" in ck_ else ck_)
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout, shift=shift).to(device)

VARIANTS = [   # (tag, class-weight clip, backbone_lr, dropout, epochs, seed, shift)
    ("s0_cw10_seed2",      10.0, 1e-4, 0.1, 10, 2, 0),   # old setting, new seed -> noise estimate
    ("s1_cw10",            10.0, 1e-4, 0.1, 10, 1, 1),
    ("s1_noW",             None, 1e-4, 0.1, 10, 1, 1),
    ("s2_noW",             None, 1e-4, 0.1, 10, 1, 2),
    ("s1_noW_lr5e-5_do.3", None, 5e-5, 0.3,  8, 1, 1),
]
orig_cw, rows = CLASS_W, []
for tag, clip, lr, do, ep, seed, sh in VARIANTS:
    random.seed(seed); torch.manual_seed(seed)
    CLASS_W = None if clip is None else make_cw(clip)
    m = fresh_teacher_ner(do, sh)
    o = train_ner(m, f"abl_{tag}", epochs=ep, patience=ep, backbone_lr=lr, head_lr=1e-3, batch_size=8)
    h = [x["macro_f1"] for x in o["history"]]
    lf, _ = lenient_scores({d["id"]: d["gold_raw"] for d in val_docs}, predict_docs(m, val_docs))
    rows.append((tag, max(h), h[-1], o["res"]["macro_p"], o["res"]["macro_r"], lf))
    os.remove(o["best_path"]); del m; torch.cuda.empty_cache()
CLASS_W = orig_cw

print(f"\n{'variant':20s} {'best F1':>8s} {'last F1':>8s} {'P':>7s} {'R':>7s} {'lenient':>8s}")
for r in rows: print(f"{r[0]:20s} {r[1]:8.4f} {r[2]:8.4f} {r[3]:7.4f} {r[4]:7.4f} {r[5]:8.4f}")

[abl_s0_cw10_seed2] trainable params: 87.53M | windows: 530 | steps/epoch: 67 | KD: off


abl_s0_cw10_seed2 ep 1/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 1.1735 | val macro P/R/F1 = 0.1630/0.3138/0.2051 | 0.1 min


abl_s0_cw10_seed2 ep 2/10:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.1544 | val macro P/R/F1 = 0.1913/0.3159/0.2317 | 0.3 min


abl_s0_cw10_seed2 ep 3/10:   0%|          | 0/67 [00:00<?, ?it/s]

... (87 outputs left out)

abl_s1_noW_lr5e-5_do.3 ep 7/8:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 7: train loss 0.0031 | val macro P/R/F1 = 0.2749/0.3177/0.2893 | 1.0 min


abl_s1_noW_lr5e-5_do.3 ep 8/8:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 8: train loss 0.0017 | val macro P/R/F1 = 0.2752/0.3141/0.2875 | 1.2 min
✅ [abl_s1_noW_lr5e-5_do.3] best val macro-F1 0.2893 | 1.2 min | peak mem 18.5 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/abl_s1_noW_lr5e-5_do.3_best.pt

variant               best F1  last F1       P       R  lenient
s0_cw10_seed2          0.3065   0.2922  0.2745  0.3600   0.5460
s1_cw10                0.2946   0.2946  0.2686  0.3435   0.5352
s1_noW                 0.2829   0.2785  0.2618  0.3234   0.5569
s2_noW                 0.2818   0.2788  0.2691  0.3116   0.5290
s1_noW_lr5e-5_do.3     0.2893   0.2875  0.2749  0.3177   0.5163


**Code cell 31. What this cell does.** An ablation study on the NER model: five variants that change one thing at a time, to see which choice matters.

**Helpers**
- `make_cw(clip)` recomputes the class-weight vector with a different clip value.
- `fresh_teacher_ner(dropout, shift)` builds a new NER model from the pretrained Phase-1 weights each time, so runs do not contaminate each other.

**`VARIANTS`** (each row: tag, class-weight clip, backbone learning rate, dropout, epochs, seed, lookahead shift):
1. `s0_cw10_seed2`: the original setting with a different random seed. This measures run-to-run noise: any difference smaller than this is not meaningful.
2. `s1_cw10`: lookahead of 1 token with the same weights.
3. `s1_noW`: lookahead 1 and no class weights.
4. `s2_noW`: lookahead 2, no class weights.
5. `s1_noW_lr5e-5_do.3`: lookahead 1, no weights, a lower learning rate and more dropout.

**Loop.** For each variant it sets the seeds, swaps the global `CLASS_W`, trains with `patience = epochs` (so no early stopping, to get the full curve), records the best and last validation F1, precision, recall, and a lenient score from `lenient_scores`. It deletes the temporary checkpoint and frees GPU memory, then restores the original class weights.

**Note.** `lenient_scores` (a looser scorer that gives credit for overlapping spans) is not defined in any cell of the saved notebook; it was defined in a cell that is no longer present, so this cell cannot be re-run as is.

**Output.** A table of the five variants with best F1, last F1, P, R and lenient F1 for the report's ablation.

In [ ]:
from torch.utils.data import DataLoader, Subset

for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "train_dataset", "val_dataset", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

# ============================================================
# 0. Settings
# ============================================================
DEBUG_MODE = True      # tiny smoke test first; flip to False for the real run
RUN_STAGE1, RUN_STAGE2 = True, True
USE_KD = True              # False -> same-size student trained WITHOUT the teacher (baseline for the report)
S1_ALPHA, S1_T = 0.5, 2.0  # stage 1 loss = (1-a)*CE + a*T^2*KL
S2_ALPHA, S2_T = 0.5, 2.0  # stage 2 (NER) the same idea

STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)     # ~43M, about half the teacher
TEACHER_CFG = dict(d_model=768, n_layer=10, n_head=12, n_kv_head=4)
TEACHER_LM_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt"
TEACHER_NER_CKPT = f"{P2_DIR}/ner_teacher_gqa{'_debug' if DEBUG_MODE else ''}_best.pt"

if DEBUG_MODE:
    S1_STEPS, S1_EVAL_EVERY, S1_WARM, S1_COOL = 20, 10, 5, 5
else:
    S1_STEPS, S1_EVAL_EVERY, S1_WARM, S1_COOL = 4000, 500, 200, 600   # 4000 steps ≈ 131M tokens
S1_BATCH, S1_ACCUM, S1_LR, S1_WD = 4, 4, 5e-4, 0.1

EXP = ("student_kd" if USE_KD else "student_nokd") + ("_debug" if DEBUG_MODE else "")
S1_DIR = f"{P2_DIR}/{EXP}_stage1"; os.makedirs(S1_DIR, exist_ok=True)
S1_BEST, S1_LAST = f"{S1_DIR}/best.pt", f"{S1_DIR}/last.pt"

V = tokenizer.get_vocab_size()
def build_lm(cfg): return CustomDecoderLM(vocab_size=V, max_seq_len=2048, tie_weights=True, **cfg)
def load_sd(path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    return ck["model"] if "model" in ck else ck

student = build_lm(STUDENT_CFG).to(device)
n_stu = sum(p.numel() for p in student.parameters())
n_tea = sum(p.numel() for p in build_lm(TEACHER_CFG).parameters())
print(f"Student: {n_stu/1e6:.2f}M params | Teacher: {n_tea/1e6:.2f}M | ratio {100*n_stu/n_tea:.0f}%")
assert 0.40 < n_stu / n_tea < 0.60, "Student is not about half the teacher — adjust STUDENT_CFG."

# ============================================================
# 1. STAGE 1 — LM training on Phase-1 blocks (with or without the teacher)
# ============================================================
def blocks_collate(batch): return torch.stack([x["input_ids"] for x in batch])

if RUN_STAGE1:
    teacher_lm = None
    if USE_KD:
        teacher_lm = build_lm(TEACHER_CFG); teacher_lm.load_state_dict(load_sd(TEACHER_LM_CKPT))
        teacher_lm.to(device).eval()
        for p in teacher_lm.parameters(): p.requires_grad_(False)

    train_loader = DataLoader(train_dataset, batch_size=S1_BATCH, shuffle=True, drop_last=True, collate_fn=blocks_collate)
    val_loader = DataLoader(Subset(val_dataset, range(min(len(val_dataset), 200))), batch_size=S1_BATCH,
                            shuffle=False, collate_fn=blocks_collate)

    @torch.no_grad()
    def s1_eval(m):
        m.eval(); tot = torch.tensor(0.0, device=device); n = 0
        for ids in val_loader:
            ids = ids.to(device).long()
            with amp(): tot += m(ids, labels=ids)["loss"].float()
            n += 1
        m.train() if m is student else None
        return (tot / max(1, n)).item()

    if USE_KD:
        tv = s1_eval(teacher_lm); print(f"Teacher val loss on the same blocks: {tv:.4f} (ppl {math.exp(tv):.1f})")

    dec = [p for p in student.parameters() if p.dim() >= 2]; nodec = [p for p in student.parameters() if p.dim() < 2]
    opt = torch.optim.AdamW([{"params": dec, "weight_decay": S1_WD}, {"params": nodec, "weight_decay": 0.0}],
                            lr=S1_LR, betas=(0.9, 0.95))
    def lr_lambda(s):
        if s < S1_WARM: return (s + 1) / S1_WARM
        if s > S1_STEPS - S1_COOL: return max(0.1, (S1_STEPS - s) / max(1, S1_COOL))
        return 1.0
    sch = LambdaLR(opt, lr_lambda)

    step, best_val = 0, float("inf")
    if os.path.exists(S1_LAST) and not DEBUG_MODE:
        ck = torch.load(S1_LAST, map_location=device, weights_only=False)
        student.load_state_dict(ck["model"]); opt.load_state_dict(ck["optimizer"]); sch.load_state_dict(ck["scheduler"])
        step, best_val = ck["step"], ck["best_val"]; print(f"Resumed stage 1 from step {step}")

    def save_s1(path):
        torch.save({"model": student.state_dict(), "optimizer": opt.state_dict(), "scheduler": sch.state_dict(),
                    "step": step, "best_val": best_val}, path)

    def batches_forever():
        while True:
            for b in train_loader: yield b
    gen = batches_forever()

    student.train(); t0, step0 = time.time(), step
    ce_acc = torch.tensor(0.0, device=device); kd_acc = torch.tensor(0.0, device=device); nlog = 0
    tokens_per_step = S1_BATCH * S1_ACCUM * len(train_dataset[0]["input_ids"])
    print(f"\n🚀 Stage 1 ({'KD' if USE_KD else 'no KD'}): {S1_STEPS} steps × {tokens_per_step:,} tokens")
    bar = tqdm(total=S1_STEPS, initial=step)
    while step < S1_STEPS:
        for _ in range(S1_ACCUM):
            ids = next(gen).to(device).long()
            with amp():
                s_logits = student(ids)["logits"][:, :-1]
                if USE_KD:
                    with torch.no_grad(): t_logits = teacher_lm(ids)["logits"][:, :-1]
            s_logits = s_logits.float()
            ce = F.cross_entropy(s_logits.reshape(-1, V), ids[:, 1:].reshape(-1))
            if USE_KD:
                kd = F.kl_div(F.log_softmax(s_logits / S1_T, -1).reshape(-1, V),
                              F.softmax(t_logits.float() / S1_T, -1).reshape(-1, V),
                              reduction="batchmean") * (S1_T ** 2)
                loss = (1 - S1_ALPHA) * ce + S1_ALPHA * kd
                kd_acc += kd.detach() / S1_ACCUM
            else:
                loss = ce
            (loss / S1_ACCUM).backward()
            ce_acc += ce.detach() / S1_ACCUM
        torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
        opt.step(); sch.step(); opt.zero_grad(set_to_none=True)
        step += 1; nlog += 1; bar.update(1)

        if step % 25 == 0 or step == S1_STEPS:
            el = time.time() - t0; tok_s = (step - step0) * tokens_per_step / el
            eta = (S1_STEPS - step) * el / max(1, step - step0) / 60
            bar.set_postfix(ce=f"{(ce_acc/nlog).item():.3f}", kd=f"{(kd_acc/nlog).item():.3f}",
                            tok_s=f"{tok_s:,.0f}", eta_min=f"{eta:.0f}")
            ce_acc.zero_(); kd_acc.zero_(); nlog = 0
        if step % S1_EVAL_EVERY == 0 or step == S1_STEPS:
            v = s1_eval(student); student.train()
            print(f"\nstep {step}: student val loss {v:.4f} (ppl {math.exp(v):.1f})")
            if v < best_val: best_val = v; save_s1(S1_BEST)
            save_s1(S1_LAST)
    bar.close()
    student.load_state_dict(torch.load(S1_BEST, map_location=device, weights_only=False)["model"])
    print(f"✅ Stage 1 done. Best student val loss {best_val:.4f} (ppl {math.exp(best_val):.1f})")
    del teacher_lm
    torch.cuda.empty_cache()

# ============================================================
# 2. STAGE 2 — NER fine-tuning of the student (distilled from the fine-tuned teacher)
# ============================================================
if RUN_STAGE2:
    if not RUN_STAGE1:
        student.load_state_dict(load_sd(S1_BEST)); student.to(device)
    ner_student = NERModel(student, d_model=STUDENT_CFG["d_model"], n_labels=NL, dropout=0.1)

    t_ner = None
    if USE_KD:
        assert os.path.exists(TEACHER_NER_CKPT), f"Fine-tune the teacher first (cell 2). Missing: {TEACHER_NER_CKPT}"
        t_lm = build_lm(TEACHER_CFG)
        t_ner = NERModel(t_lm, d_model=TEACHER_CFG["d_model"], n_labels=NL)
        t_ner.load_state_dict(load_sd(TEACHER_NER_CKPT)); t_ner.to(device).eval()

    out_s = train_ner(ner_student, f"ner_{EXP}", teacher=t_ner, alpha=S2_ALPHA if USE_KD else 0.0, T=S2_T,
                      epochs=20, patience=5, backbone_lr=3e-4, head_lr=1e-3, batch_size=8, debug=DEBUG_MODE)
    print("\n=== Student validation (strict) — best epoch ===")
    print_scores(out_s["res"])
    if not DEBUG_MODE:
        write_test_predictions(ner_student, f"sub2_{EXP}")
    else:
        print("\n*** DEBUG_MODE smoke test finished. If it ran cleanly, set DEBUG_MODE=False and re-run. ***")

Student: 43.21M params | Teacher: 87.51M | ratio 49%
Teacher val loss on the same blocks: 2.2067 (ppl 9.1)

🚀 Stage 1 (KD): 4000 steps × 32,768 tokens


  0%|          | 0/4000 [00:00<?, ?it/s]


step 500: student val loss 3.9869 (ppl 53.9)

step 1000: student val loss 3.3787 (ppl 29.3)

step 1500: student val loss 3.0747 (ppl 21.6)

step 2000: student val loss 2.8721 (ppl 17.7)

step 2500: student val loss 2.7889 (ppl 16.3)

step 3000: student val loss 2.7295 (ppl 15.3)

step 3500: student val loss 2.6490 (ppl 14.1)

step 4000: student val loss 2.5347 (ppl 12.6)
✅ Stage 1 done. Best student val loss 2.5347 (ppl 12.6)
[ner_student_kd] trainable params: 43.23M | windows: 530 | steps/epoch: 67 | KD: alpha=0.50 T=2.0


ner_student_kd ep 1/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.0419 | val macro P/R/F1 = 0.1819/0.2719/0.2101 | 0.1 min


ner_student_kd ep 2/20:   0%|          | 0/67 [00:00<?, ?it/s]

... (29 outputs left out)

ner_student_kd ep 17/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 17: train loss 0.0135 | val macro P/R/F1 = 0.2974/0.3617/0.3141 | 2.6 min


ner_student_kd ep 18/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 18: train loss 0.0125 | val macro P/R/F1 = 0.2949/0.3576/0.3126 | 2.7 min
Early stop (no improvement for 5 epochs).
✅ [ner_student_kd] best val macro-F1 0.3219 | 2.7 min | peak mem 13.9 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_student_kd_best.pt

=== Student validation (strict) — best epoch ===
MACRO  P=0.3035  R=0.3734  F1=0.3219  | micro F1=0.4052
label            P       R      F1   gold   pred     tp
PREC         0.182   0.250   0.211    212    291     53
STAT         0.568   0.608   0.588    171    183    104
DATE         0.834   0.962   0.893    157    181    151
AUTH         0.185   0.113   0.140    150     92     17
COURT        0.503   0.814   0.622     97    157     79
RESP         0.286   0.090   0.137     89     28      8
CASENO       0.262   0.440   0.328     75    126     33
APP          0.135   0.106   0.119     66     52      7
JUDGE        0.361   0.406   0.382     32     36     13
WIT          0.000   0.000   0.000     23     25      

**Code cell 32. What this cell does.** Knowledge distillation (the required KD experiment of Phase 2): trains a smaller student (about 43M parameters, half the teacher) in two stages, with the fine-tuned GQA model as teacher.

**Section 0: settings.** `DEBUG_MODE`, `RUN_STAGE1` / `RUN_STAGE2` switches, and `USE_KD` (`False` trains the same student without a teacher, to measure the benefit of distillation). Loss weights: `alpha = 0.5`, temperature `T = 2.0` for both stages. `STUDENT_CFG` is `d_model=576, n_layer=7` (the teacher is 768 wide, 10 layers). `S1_STEPS = 4000` optimizer steps (about 131M tokens) for the real run, learning rate 5e-4. The checkpoint folder is named after the experiment (`student_kd` or `student_nokd`). `build_lm` builds a model of a given configuration; `load_sd` loads weights from a checkpoint file whether it is wrapped or not.

The cell prints the student and teacher parameter counts and asserts the student is 40-60% of the teacher.

**Stage 1: language-model distillation on the Phase-1 text blocks.**
- If `USE_KD`, the teacher language model is loaded, frozen (`requires_grad_(False)`) and used only to produce soft targets.
- Data loaders give batches of packed blocks. `s1_eval` computes validation loss.
- The optimizer is AdamW (weight decay only on matrices) with the same warm-up and cool-down shape as Phase 1; if `last.pt` exists the run resumes.
- `batches_forever` cycles through the data indefinitely.
- Each optimizer step accumulates 4 micro-batches. For each: the student predicts next-token logits (`[:, :-1]` drops the last position, which has no target). **Cross-entropy** `ce` is against the real next token. With KD, **`kd`** is the KL divergence between the softened student distribution (`log_softmax(s / T)`) and the softened teacher distribution (`softmax(t / T)`), multiplied by `T^2` so its gradient has the right size. The loss is `(1 - alpha) * ce + alpha * kd`. Without KD it is just `ce`.
- Softening with a temperature above 1 makes the teacher's distribution flatter, so the student also learns which wrong words are "nearly right" (the teacher's dark knowledge).
- Logging shows CE, KD, tokens per second and an ETA; every `S1_EVAL_EVERY` steps it evaluates and saves `best.pt` and `last.pt`. At the end the best weights are reloaded and the teacher is deleted to free memory.

**Stage 2: NER fine-tuning of the student.** The student is wrapped in `NERModel`. With KD, the fine-tuned NER teacher (`ner_teacher_gqa_best.pt`) is loaded as `t_ner`, and `train_ner` is called with `teacher=t_ner, alpha=0.5, T=2.0` (20 epochs, patience 5, backbone learning rate 3e-4). The result is printed per label, and the test predictions are saved as `prediction_sub2_<experiment>.json` unless in debug mode.

**Why two stages.** Stage 1 transfers general language ability to the small model; stage 2 transfers the task skill.

 LoRA & LoRMA


In [ ]:
for name in ["train_ner", "NERModel", "write_test_predictions", "CustomDecoderLM", "TEACHER_CKPT", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first."

DEBUG_MODE = False    # smoke test first (1 epoch, tiny); set False for the real runs

# ============================================================
# 1. Adapter layers (written from scratch)
# ============================================================
class LoRALinear(nn.Module):
    """y = W0 x + (alpha/r) * B (A x)       A: r x in, B: out x r (B=0 at init)"""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """LoRMA+: W = ((alpha/r) B A + I) W0 ; B,A act on the OUTPUT side.  y = W0 x ;  out = y + (alpha/r) B (A y)
       A: r x out, B: out x r (B=0 at init -> exact identity)."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x)
        return y + self.scale * F.linear(F.linear(y, self.A), self.B)

def inject_adapters(ner, kind, r, alpha, targets=("q_proj", "k_proj", "v_proj", "out_proj")):
    for p in ner.parameters(): p.requires_grad_(False)           # freeze everything first
    cls = LoRALinear if kind == "lora" else LoRMAPlusLinear
    for layer in ner.lm.layers:
        for nm in targets:
            base = getattr(layer.attn, nm)
            setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))   # new A,B are trainable
    for p in ner.head.parameters(): p.requires_grad_(True)       # the NER head is always trained

@torch.no_grad()
def check_merge(mod):
    """Max |adapter output - merged-weight output| on random input. Should be ~1e-5 or smaller."""
    W0 = mod.base.weight
    x = torch.randn(4, W0.shape[1], device=W0.device)
    ref = mod(x)
    Wm = W0 + mod.scale * (mod.B @ mod.A) if isinstance(mod, LoRALinear) else W0 + (mod.scale * (mod.B @ mod.A)) @ W0
    return (ref - F.linear(x, Wm)).abs().max().item()

def build_best_ner(dropout=0.1):
    """Fresh model from the PRETRAINED Phase-1 checkpoint (never the fine-tuned objects in memory)."""
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck_ = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck_["model"] if "model" in ck_ else ck_)
    return NERModel(lm_, d_model=768, n_labels=NL, dropout=dropout).to(device)

# ============================================================
# 2. Runs: LoRA (r=8) and LoRMA+ (r=10) on the best GQA model, two learning rates each
# ============================================================
RESULTS = []
for kind, r in [("lora", 8), ("lorma", 10)]:
    for lr in [3e-4, 1e-3]:
        random.seed(1); torch.manual_seed(1)
        ner = build_best_ner()

        # identity check: with B=0 the adapted model must equal the original
        ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
        ner.eval()
        with torch.no_grad(): before = ner(ids)
        inject_adapters(ner, kind, r, alpha=2 * r)
        with torch.no_grad(): after = ner(ids)
        d = (before - after).abs().max().item()
        assert d < 1e-4, f"{kind}: not identity at init (max diff {d})"
        print(f"[{kind}] identity-at-init OK (max diff {d:.1e})")

        name = f"ner_best_{kind}_lr{lr:g}"
        out = train_ner(ner, name, epochs=20, patience=5, backbone_lr=lr, head_lr=1e-3,
                        batch_size=8, debug=DEBUG_MODE)
        mod0 = ner.lm.layers[0].attn.q_proj
        print(f"[{kind}] merge check on layer-0 q_proj: max diff {check_merge(mod0):.1e}")
        if not DEBUG_MODE: write_test_predictions(ner, f"abl_{name}")
        h = [x["macro_f1"] for x in out["history"]]
        RESULTS.append((kind, lr, out["trainable_params"], max(h), out["res"]["macro_p"], out["res"]["macro_r"],
                        out["minutes"], out["peak_gb"]))
        del ner; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for k, lr, tp, f1, p, r_, mi, gb in RESULTS:
    print(f"{k:7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")
print(f"\nFull fine-tuning reference: 87.53M trainable, val macro-F1 0.3063")

[lora] identity-at-init OK (max diff 0.0e+00)
[ner_best_lora_lr0.0003] trainable params: 0.43M | windows: 530 | steps/epoch: 67 | KD: off


ner_best_lora_lr0.0003 ep 1/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 2.4151 | val macro P/R/F1 = 0.0659/0.1962/0.0953 | 0.1 min


ner_best_lora_lr0.0003 ep 2/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.3817 | val macro P/R/F1 = 0.1453/0.3081/0.1939 | 0.3 min


ner_best_lora_lr0.0003 ep 3/20:   0%|          | 0/67 [00:00<?, ?it/s]

... (97 outputs left out)

ner_best_lorma_lr0.001 ep 16/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 16: train loss 0.0056 | val macro P/R/F1 = 0.2463/0.3390/0.2815 | 2.2 min


ner_best_lorma_lr0.001 ep 17/20:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 17: train loss 0.0041 | val macro P/R/F1 = 0.2421/0.3284/0.2742 | 2.4 min
Early stop (no improvement for 5 epochs).
✅ [ner_best_lorma_lr0.001] best val macro-F1 0.2923 | 2.4 min | peak mem 14.8 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_best_lorma_lr0.001_best.pt
[lorma] merge check on layer-0 q_proj: max diff 7.2e-07
Saved 35 test docs / 5867 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_abl_ner_best_lorma_lr0.001.json

method       lr  trainable  best F1       P       R   min    GB
lora      3e-04    428,825   0.2349  0.1885  0.3309   1.7  16.4
lora      1e-03    428,825   0.2629  0.2179  0.3405   1.7  15.0
lorma     3e-04    428,825   0.2358  0.1877  0.3303   1.6  14.8
lorma     1e-03    428,825   0.2923  0.2457  0.3713   2.4  14.8

Full fine-tuning reference: 87.53M trainable, val macro-F1 0.3063


**Code cell 33. What this cell does.** Implements LoRA and LoRMA+ from scratch and trains the NER model with each (the required LoRMA technique, plus LoRA as a comparison).

**`LoRALinear(base, r, alpha)`**: wraps a frozen linear layer. Output = `W0 x + (alpha / r) * B (A x)`. `A` is `r x in` (random start with Kaiming uniform), `B` is `out x r` (zeros), so at the start the correction is exactly 0. Only `A` and `B` train: `r * (in + out)` parameters instead of `in * out`.

**`LoRMAPlusLinear(base, r, alpha)`**: the multiplicative variant. It first computes `y = W0 x`, then returns `y + (alpha / r) * B (A y)`, which equals multiplying the weight by `(alpha/r) B A + I` on the output side (`A` is `r x out`, `B` is `out x r`, `B = 0` at start so it is exactly the identity).

**`inject_adapters(ner, kind, r, alpha, targets)`**: freezes every parameter, replaces the four attention projections (`q_proj`, `k_proj`, `v_proj`, `out_proj`) of every layer by adapter-wrapped copies, and re-enables training of the NER head.

**`check_merge(mod)`**: verifies that an adapter can be merged into one weight matrix (`W0 + scale * B A` for LoRA; `W0 + (scale * B A) W0` for LoRMA+) and gives the same output on random input. This is what makes adapters free at inference.

**`build_best_ner()`**: always rebuilds the model from the pretrained Phase-1 checkpoint, never from a model already fine-tuned in memory.

**Runs.** For LoRA (rank 8) and LoRMA+ (rank 10), at learning rates 3e-4 and 1e-3: set the seed, build the model, record the output on a random input before and after injecting adapters and assert they differ by less than 1e-4 (identity at initialisation), train with `train_ner` (adapter learning rate passed as `backbone_lr`), run the merge check, write test predictions, and store trainable parameters, best F1, precision, recall, time and memory. `alpha = 2 * r` is used. A final table lists all four runs, with the full fine-tuning reference printed for comparison.

**Why ranks 8 and 10.** The two methods use slightly different shapes of correction; ranks were chosen so that their trainable parameter counts are of similar size for a fair comparison.

 Prefix-Tuning

In [ ]:
for name in ["train_ner", "NERModel", "write_test_predictions", "build_best_ner", "train_docs", "NL", "P2_DIR"]:
    assert name in globals(), f"'{name}' is not defined — run the earlier cells first (build_best_ner comes from the LoRA/LoRMA cell)."

DEBUG_MODE = False      # smoke test first; set False for the real runs
PREFIX_LEN = 32

class PrefixNERModel(NERModel):
    """Prefix tuning: frozen backbone + trainable per-layer key/value prefixes (+ the NER head).
    Prefixes are fed to the existing attention through its past_kv argument."""
    def __init__(self, lm, d_model, n_labels, init_ids, dropout=0.1, shift=0):
        super().__init__(lm, d_model, n_labels, dropout, shift)
        for p in self.parameters(): p.requires_grad_(False)          # freeze backbone first
        lm.eval()
        with torch.no_grad():
            dev = next(lm.parameters()).device
            pkv = lm(torch.tensor([init_ids], device=dev))["past_key_values"]   # real K/V of the first P tokens
        self.prefix_k = nn.ParameterList([nn.Parameter(k.detach().clone()) for k, _ in pkv])
        self.prefix_v = nn.ParameterList([nn.Parameter(v.detach().clone()) for _, v in pkv])
        for p in self.head.parameters(): p.requires_grad_(True)        # head is always trained

    def hidden(self, input_ids):
        lm = self.lm; B = input_ids.shape[0]
        x = lm.token_embedding(input_ids)
        for i, layer in enumerate(lm.layers):
            pkv = (self.prefix_k[i].expand(B, -1, -1, -1), self.prefix_v[i].expand(B, -1, -1, -1))
            x, _ = layer(x, lm.cos, lm.sin, past_kv=pkv)
        return lm.norm(x)

    def forward(self, input_ids):
        out = self.head(self.drop(self.hidden(input_ids)))
        k = self.shift
        if k > 0:
            out = torch.cat([out[:, k:], out.new_zeros(out.shape[0], k, out.shape[2])], dim=1)
        return out

@torch.no_grad()
def plain_hidden(lm, ids):
    x = lm.token_embedding(ids)
    for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
    return lm.norm(x)

PREFIX_RESULTS = []
for lr in [1e-3, 5e-3]:
    random.seed(1); torch.manual_seed(1)
    base = build_best_ner()
    init_ids = train_docs[0]["ids"][:PREFIX_LEN]
    ner = PrefixNERModel(base.lm, d_model=768, n_labels=NL, init_ids=init_ids).to(device)

    # correctness check: prefix path == running the model on [prefix tokens + input], at initialisation
    ner.eval()
    ids = torch.randint(0, tokenizer.get_vocab_size(), (1, 64), device=device)
    with torch.no_grad():
        h_prefix = ner.hidden(ids)
        full = torch.cat([torch.tensor([init_ids], device=device), ids], dim=1)
        h_full = plain_hidden(ner.lm, full)[:, PREFIX_LEN:]
    d = (h_prefix - h_full).abs().max().item()
    assert d < 1e-3, f"prefix path does not match the reference run (max diff {d})"
    print(f"[prefix] init check OK (max diff {d:.1e})")

    name = f"ner_best_prefix_lr{lr:g}"
    out = train_ner(ner, name, epochs=30, patience=6, backbone_lr=lr, head_lr=1e-3,
                    batch_size=8, weight_decay=0.0, debug=DEBUG_MODE)
    if not DEBUG_MODE: write_test_predictions(ner, f"abl_{name}")
    h = [x["macro_f1"] for x in out["history"]]
    PREFIX_RESULTS.append((lr, out["trainable_params"], max(h), out["res"]["macro_p"], out["res"]["macro_r"],
                           out["minutes"], out["peak_gb"]))
    del ner, base; torch.cuda.empty_cache()

print(f"\n{'method':7s} {'lr':>7s} {'trainable':>10s} {'best F1':>8s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for lr, tp, f1, p, r_, mi, gb in PREFIX_RESULTS:
    print(f"{'prefix':7s} {lr:7.0e} {tp:10,d} {f1:8.4f} {p:7.4f} {r_:7.4f} {mi:5.1f} {gb:5.1f}")
print("\nReference (val macro-F1): full fine-tuning 0.3063 | LoRMA+ 0.2923 | LoRA 0.2629")

[prefix] init check OK (max diff 4.8e-05)
[ner_best_prefix_lr0.001] trainable params: 0.18M | windows: 530 | steps/epoch: 67 | KD: off


ner_best_prefix_lr0.001 ep 1/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 1: train loss 3.2331 | val macro P/R/F1 = 0.0086/0.0966/0.0157 | 0.1 min


ner_best_prefix_lr0.001 ep 2/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 2: train loss 0.7988 | val macro P/R/F1 = 0.0480/0.2167/0.0770 | 0.3 min


ner_best_prefix_lr0.001 ep 3/30:   0%|          | 0/67 [00:00<?, ?it/s]

... (95 outputs left out)

ner_best_prefix_lr0.005 ep 21/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 21: train loss 0.1345 | val macro P/R/F1 = 0.1509/0.2932/0.1948 | 2.8 min


ner_best_prefix_lr0.005 ep 22/30:   0%|          | 0/67 [00:00<?, ?it/s]

epoch 22: train loss 0.1322 | val macro P/R/F1 = 0.1572/0.3000/0.2031 | 3.0 min
Early stop (no improvement for 6 epochs).
✅ [ner_best_prefix_lr0.005] best val macro-F1 0.2118 | 3.0 min | peak mem 14.7 GB | saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/ner_best_prefix_lr0.005_best.pt
Saved 35 test docs / 8128 entities -> /content/drive/MyDrive/CS781_Capstone_Project_Phase_2/prediction_abl_ner_best_prefix_lr0.005.json

method       lr  trainable  best F1       P       R   min    GB
prefix    1e-03    183,065   0.1704  0.1233  0.2928   4.1  14.7
prefix    5e-03    183,065   0.2118  0.1668  0.3064   3.0  14.7

Reference (val macro-F1): full fine-tuning 0.3063 | LoRMA+ 0.2923 | LoRA 0.2629


**Code cell 34. What this cell does.** Prefix tuning: a third parameter-efficient method for NER. The backbone stays frozen and the model learns a short sequence of extra key and value vectors in every layer.

**`PrefixNERModel`** (extends `NERModel`)
- Freezes all parameters. It runs the real first 32 tokens (`init_ids`) of a training document through the model once and copies the resulting keys and values of every layer into trainable parameters `prefix_k` and `prefix_v`. Starting from real activations is a better initialisation than random numbers.
- `hidden(input_ids)`: for each layer it passes the layer's prefix as `past_kv`, so every real token attends to the 32 prefix positions as if they were earlier tokens. The existing attention code needed no change.
- `forward` adds dropout and the head, with the same optional lookahead as `NERModel`.

**`plain_hidden`** runs the model normally and is used for the correctness check.

**Runs.** Two learning rates (1e-3 and 5e-3). For each: build the pretrained model, create the prefix model, and check that the prefix path at initialisation gives the same hidden states as running the model on `[prefix tokens + input]` and discarding the prefix positions (max difference below 1e-3). Then `train_ner` (30 epochs, patience 6, no weight decay), save predictions, and collect trainable parameters, F1, precision, recall, time and memory. The final table prints the results next to the earlier reference values.

**Why this is a distinct technique.** LoRA/LoRMA+ change the weight matrices; prefix tuning changes none of them and only adds context the attention can read.

## Rhetorical Role Prediction

In [ ]:
import os, json, glob, shutil, torch
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shutil.copy(f"{DRIVE_PROJECT_DIR}/legal_bpe_32k_FINAL.json", "legal_bpe_32k_FINAL.json")
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
assert tokenizer.get_vocab_size() == 32000
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# HF_TOKEN = "..."   # set it as before (the Phase-2 setup cell used it)
print("device:", device, "| vocab:", tokenizer.get_vocab_size())
for p in sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True)): print(" ", p)

device: cuda | vocab: 32000
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt
  /content/drive/MyDrive/CS781_Capstone_Proj

**Explanation (code cell 35).** Identical to code cell 5 of this notebook (same code, same behaviour); see the explanation under that cell.

In [ ]:
import os, json, math, time, random, pickle, collections
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
from huggingface_hub import hf_hub_download
for n in ["tokenizer", "device", "HF_TOKEN", "HF_DATASET_REPO"]: assert n in globals(), f"'{n}' missing, run the setup cell"

P3_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_3"; os.makedirs(P3_DIR + "/submission", exist_ok=True)
WINDOW, MAX_SENT_TOK = 1024, 128
USE_CLASS_WEIGHTS, CLASS_W_CLIP = True, 10.0
PAD_ID = tokenizer.token_to_id("[PAD]"); NL_IDS = tokenizer.encode("\n").ids
def amp(): return torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# ---------------- data ----------------
def _load(f): return json.load(open(hf_hub_download(HF_DATASET_REPO, f"Phase_3_RR/{f}", repo_type="dataset", token=HF_TOKEN), encoding="utf-8"))
L2I = _load("Train/label_vocab.json"); LABELS = sorted(L2I, key=L2I.get); NC = len(LABELS)

def norm_docs(raw, domain):
    items = list(raw.items()) if isinstance(raw, dict) else raw
    docs = []
    for it in items:
        did, v = (it if isinstance(it, tuple) else (it["id"], it))
        text = v if isinstance(v, list) else v["text"]
        labels = v.get("labels") if isinstance(v, dict) else None
        d = dict(id=did, domain=domain, text=text, labels=labels)
        d["sent_ids"] = [tokenizer.encode(s.strip()).ids[:MAX_SENT_TOK] + NL_IDS for s in text]
        d["y"] = [L2I[l] for l in labels] if labels is not None else None
        if labels is not None: assert len(labels) == len(text)
        docs.append(d)
    return docs

train_docs = norm_docs(_load("Train/CL_train.json"), "CL") + norm_docs(_load("Train/IT_train.json"), "IT")
val_docs   = norm_docs(_load("Train/CL_dev.json"), "CL")   + norm_docs(_load("Train/IT_dev.json"), "IT")
test_docs  = norm_docs(_load("Test/CL_test.json"), "CL")   + norm_docs(_load("Test/IT_test.json"), "IT")
assert len({d["id"] for d in test_docs}) == len(test_docs), "test doc ids are not unique across domains"
for nm, ds in [("train", train_docs), ("val", val_docs), ("test", test_docs)]:
    print(f"{nm:5s} docs={len(ds)} sents={sum(len(d['text']) for d in ds)} "
          f"CL={sum(d['domain']=='CL' for d in ds)} IT={sum(d['domain']=='IT' for d in ds)}")

cnt = np.bincount([y for d in train_docs for y in d["y"]], minlength=NC).astype(float)
CLASS_W = torch.tensor(np.clip(np.sqrt(cnt.sum() / (NC * np.maximum(cnt, 1))), 0, CLASS_W_CLIP), dtype=torch.float32, device=device)
print("class weights:", {LABELS[i]: round(float(CLASS_W[i]), 2) for i in range(NC)})

# ---------------- context modes / windows ----------------
CTX = {"sentence": dict(max_sents=1, stride_frac=0.5),        # each sentence alone
       "local8":   dict(max_sents=8, stride_frac=0.5),        # up to 8 consecutive sentences
       "document": dict(max_sents=10**9, stride_frac=0.75)}   # as many sentences as fit in 1024 tokens
BW = {"sentence": 64, "local8": 32, "document": 8}            # windows per batch

def make_windows(sent_ids, max_sents, stride_frac, max_tokens=WINDOW):
    n, lens, wins, a = len(sent_ids), [len(x) for x in sent_ids], [], 0
    while True:
        b, tot = a, 0
        while b < n and tot + lens[b] <= max_tokens and (b - a) < max_sents: tot += lens[b]; b += 1
        wins.append((a, b))
        if b >= n: break
        c, acc, target = a, 0, max(1, int(stride_frac * tot))
        while c < b and acc < target: acc += lens[c]; c += 1
        a = max(c, a + 1)
    return wins

def all_windows(docs, mode): return [(di, a, b) for di, d in enumerate(docs) for a, b in make_windows(d["sent_ids"], **CTX[mode])]

def collate(wins, docs):
    seqs, spans, ys = [], [], []
    for di, a, b in wins:
        d = docs[di]; ids, sp = [], []
        for k in range(a, b): s = len(ids); ids += d["sent_ids"][k]; sp.append((s, len(ids)))
        seqs.append(ids); spans.append(sp)
        if d["y"] is not None: ys += d["y"][a:b]
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    for j, s in enumerate(seqs): x[j, :len(s)] = torch.tensor(s)
    return x, spans, (torch.tensor(ys) if ys else None)

# ---------------- model: backbone + sentence pooling + head ----------------
def pool(H, spans):
    jw, st, en = [], [], []
    for j, sp in enumerate(spans):
        for s, e in sp: jw.append(j); st.append(s); en.append(e)
    jw, st, en = [torch.tensor(t, device=H.device) for t in (jw, st, en)]
    last = H[jw, en - 1]                                                   # delimiter token: has seen the whole sentence
    cs = torch.cat([torch.zeros_like(H[:, :1]).float(), H.float().cumsum(1)], dim=1)
    mean = ((cs[jw, en] - cs[jw, st]) / (en - st).unsqueeze(1)).to(H.dtype)  # mean over the sentence's tokens
    return torch.cat([last, mean], -1)

class RRModel(nn.Module):
    """Decoder backbone (causal) -> per-sentence features [last token ; mean] -> linear head."""
    def __init__(self, lm, d_model, n_labels=NC, dropout=0.1):
        super().__init__()
        self.lm, self.drop, self.head = lm, nn.Dropout(dropout), nn.Linear(2 * d_model, n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def hidden(self, ids):
        lm = self.lm; x = lm.token_embedding(ids)
        for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        return lm.norm(x)
    def forward(self, ids, spans): return self.head(self.drop(pool(self.hidden(ids), spans)))

# ---------------- metrics (own implementation) ----------------
def rr_scores(gold, pred):
    g, p = np.array(gold), np.array(pred); labs = sorted(set(g.tolist()) | set(p.tolist())); per = {}
    for l in labs:
        tp = int(((g == l) & (p == l)).sum()); fp = int(((g != l) & (p == l)).sum()); fn = int(((g == l) & (p != l)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[LABELS[l]] = dict(p=pr, r=rc, f1=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int((g == l).sum()))
    m = lambda k: float(np.mean([v[k] for v in per.values()]))
    return dict(macro_f1=m("f1"), macro_p=m("p"), macro_r=m("r"), acc=float((g == p).mean()), per=per)

# ---------------- prediction ----------------
@torch.no_grad()
def predict_probs(model, docs, mode):
    model.eval(); bw = BW[mode]; out = {}
    for d in docs:
        n = len(d["sent_ids"]); P = torch.zeros(n, NC, device=device); C = torch.zeros(n, device=device)
        wins = [(0, a, b) for a, b in make_windows(d["sent_ids"], **CTX[mode])]
        for i in range(0, len(wins), bw):
            chunk = wins[i:i + bw]; x, spans, _ = collate(chunk, [d])
            with amp(): lg = model(x.to(device), spans)
            p = F.softmax(lg.float(), -1); k = 0
            for _, a, b in chunk: P[a:b] += p[k:k + b - a]; C[a:b] += 1; k += b - a
        out[d["id"]] = (P / C.unsqueeze(1)).cpu().numpy()
    return out

def evaluate(model, docs, mode):
    P = predict_probs(model, docs, mode)
    G = {dm: ([], []) for dm in ("CL", "IT", "ALL")}
    for d in docs:
        pr = P[d["id"]].argmax(1).tolist()
        for key in (d["domain"], "ALL"): G[key][0].extend(d["y"]); G[key][1].extend(pr)
    res = rr_scores(*G["ALL"]); res["by_domain"] = {dm: rr_scores(*G[dm]) for dm in ("CL", "IT") if G[dm][0]}
    return res, P

def print_rr(res):
    print(f"ALL  macro-F1 {res['macro_f1']:.4f} | P {res['macro_p']:.4f} | R {res['macro_r']:.4f} | acc {res['acc']:.4f}")
    for dm, r in res["by_domain"].items(): print(f"  {dm}  macro-F1 {r['macro_f1']:.4f} | P {r['macro_p']:.4f} | R {r['macro_r']:.4f} | acc {r['acc']:.4f}")

def write_test_predictions(model, name, mode):
    P = predict_probs(model, test_docs, mode)
    allp = {d["id"]: [LABELS[i] for i in P[d["id"]].argmax(1)] for d in test_docs}
    for d in test_docs: assert len(allp[d["id"]]) == len(d["text"])
    for suffix, sel in [("all", allp), ("CL", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                        ("IT", {d["id"]: allp[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
        json.dump(sel, open(f"{P3_DIR}/prediction_{name}_{suffix}.json", "w"), ensure_ascii=False)
    pickle.dump(P, open(f"{P3_DIR}/probs_test_{name}.pkl", "wb"))
    print(f"Saved prediction_{name}_{{all,CL,IT}}.json ({len(allp)} docs, {sum(map(len, allp.values()))} sentences)")

# ---------------- trainer (optional distillation from a teacher RRModel) ----------------
def train_rr(model, name, mode="document", teacher=None, alpha=0.0, T=2.0, epochs=10, patience=3,
             backbone_lr=1e-4, head_lr=1e-3, bw=None, weight_decay=0.01, warmup_frac=0.06, debug=False, seed=1):
    random.seed(seed); torch.manual_seed(seed); bw = bw or BW[mode]
    tw = all_windows(train_docs, mode); vdocs = val_docs
    if debug:
        epochs, name = 1, name + "_debug"; tw = tw[:bw * 4]
        vdocs = [next(d for d in val_docs if d["domain"] == dm) for dm in ("CL", "IT")]
    model.to(device)
    if teacher is not None:
        teacher.to(device).eval()
        for p in teacher.parameters(): p.requires_grad_(False)
    groups = {}
    for n_, p in model.named_parameters():
        if p.requires_grad: groups.setdefault((n_.startswith("head."), p.dim() >= 2), []).append(p)
    opt = torch.optim.AdamW([{"params": ps, "lr": head_lr if h else backbone_lr, "weight_decay": weight_decay if dec else 0.0}
                             for (h, dec), ps in groups.items()], betas=(0.9, 0.98))
    spe = math.ceil(len(tw) / bw); total = spe * epochs; warm = max(1, int(warmup_frac * total))
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] mode={mode} windows={len(tw)} trainable={n_train:,} {'KD alpha=%.2f T=%.1f' % (alpha, T) if teacher is not None else ''}")
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    best, best_state, bad, hist = -1.0, None, 0, []
    cw = CLASS_W if USE_CLASS_WEIGHTS else None
    for ep in range(epochs):
        model.train(); random.shuffle(tw); tl, nb = 0.0, 0
        for i in range(0, len(tw), bw):
            x, spans, y = collate(tw[i:i + bw], train_docs); x, y = x.to(device), y.to(device)
            with amp():
                lg = model(x, spans).float()
                loss = F.cross_entropy(lg, y, weight=cw)
                if teacher is not None:
                    with torch.no_grad(): tlg = teacher(x, spans).float()
                    kd = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(tlg / T, -1), reduction="batchmean") * T * T
                    loss = (1 - alpha) * loss + alpha * kd
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
            opt.step(); sched.step(); tl += loss.item(); nb += 1
        res, _ = evaluate(model, vdocs, mode)
        hist.append(dict(epoch=ep + 1, loss=tl / nb, macro_f1=res["macro_f1"], **{f"f1_{k}": v["macro_f1"] for k, v in res["by_domain"].items()}))
        print(f"  ep{ep+1:2d} loss {tl/nb:.4f} | val macro-F1 {res['macro_f1']:.4f} "
              + " ".join(f"{k} {v['macro_f1']:.3f}" for k, v in res["by_domain"].items()) + f" | {(time.time()-t0)/60:.1f} min")
        if res["macro_f1"] > best:
            best, bad = res["macro_f1"], 0
            names = {n_ for n_, p in model.named_parameters() if p.requires_grad}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if k in names}
        else:
            bad += 1
            if bad >= patience: print("  early stop"); break
    model.load_state_dict(best_state, strict=False)
    path = f"{P3_DIR}/rr_{name}_best.pt"; torch.save(best_state, path)
    res, P = evaluate(model, vdocs, mode)
    print(f"[{name}] BEST on dev:"); print_rr(res)
    return dict(res=res, history=hist, best_path=path, minutes=(time.time() - t0) / 60,
                peak_gb=torch.cuda.max_memory_allocated() / 1e9, trainable_params=n_train, val_probs=P)

# ---------------- sanity checks (no model needed) ----------------
for mode in CTX:
    W = all_windows(train_docs, mode)
    cover = collections.defaultdict(set)
    for di, a, b in W: cover[di].update(range(a, b))
    assert all(len(cover[di]) == len(d["sent_ids"]) for di, d in enumerate(train_docs)), f"{mode}: some sentences not covered"
    mx = max(sum(len(train_docs[di]["sent_ids"][k]) for k in range(a, b)) for di, a, b in W)
    print(f"{mode:9s} windows={len(W):5d} | every sentence covered | longest window {mx} tokens (<= {WINDOW})")
    assert mx <= WINDOW
tt = sorted(sum(map(len, d["sent_ids"])) for d in test_docs)
print("test tokens/doc median", tt[len(tt) // 2], "max", tt[-1], "| val labels:", collections.Counter(LABELS[y] for d in val_docs for y in d["y"]).most_common(4), "...")
print("✅ Phase-3 toolkit ready")

label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

train docs=80 sents=16845 CL=40 IT=40
val   docs=10 sents=2142 CL=5 IT=5
test  docs=10 sents=2197 CL=5 IT=5
class weights: {'Fact': 0.56, 'Issue': 3.48, 'ArgumentPetitioner': 0.9, 'ArgumentRespondent': 0.98, 'Statute': 1.09, 'Dissent': 2.14, 'PrecedentReliedUpon': 0.82, 'PrecedentNotReliedUpon': 1.51, 'PrecedentOverruled': 10.0, 'RulingByLowerCourt': 1.22, 'RatioOfTheDecision': 0.55, 'RulingByPresentCourt': 2.05, 'None': 2.09}
sentence  windows=16845 | every sentence covered | longest window 129 tokens (<= 1024)
local8    windows= 3694 | every sentence covered | longest window 825 tokens (<= 1024)
document  windows=  892 | every sentence covered | longest window 1024 tokens (<= 1024)
test tokens/doc median 7457 max 35993 | val labels: [('Fact', 444), ('RatioOfTheDecision', 403), ('ArgumentPetitioner', 400), ('ArgumentRespondent', 272)] ...
✅ Phase-3 toolkit ready


**Code cell 36. What this cell does.** The Phase-3 toolkit for rhetorical-role (RR) labelling: it loads the data, builds sentence windows, defines the model, scorer, predictor and trainer. Later RR cells call these functions.

**Task.** Each document is a list of sentences, and every sentence gets one of 13 rhetorical-role labels (for example facts, arguments, ruling). Two domains are mixed: CL (Competition Law) and IT (Income Tax). The metric is macro-F1.

**Setup and data**
- `WINDOW = 1024` tokens per window, `MAX_SENT_TOK = 128` tokens kept per sentence. `NL_IDS` is the token id of a newline, appended after each sentence as a delimiter.
- `_load(f)` downloads a file from `Phase_3_RR/` and parses it. `L2I` is the label vocabulary (label to id); `LABELS` the names in id order; `NC` their count.
- `norm_docs(raw, domain)` turns each document into a dictionary: `sent_ids` (each sentence tokenized, cut to 128 tokens, with a newline id added), `y` (label ids) and the domain. It asserts that sentences and labels have equal length. Train, validation and test sets are built for CL and IT and concatenated; test document ids must be unique across domains. The sizes are printed.
- Class weights: `sqrt(total / (classes * count))`, clipped at 10, so rare roles weigh more.

**Context modes.** `CTX` defines three ways to show the model context: `sentence` (each sentence alone), `local8` (up to 8 consecutive sentences) and `document` (as many consecutive sentences as fit in 1,024 tokens). `BW` is how many windows go in one batch.
- `make_windows(sent_ids, max_sents, stride_frac)`: starts at sentence a, adds sentences until the token or sentence limit is reached, records `(a, b)`, then moves the start forward by `stride_frac` of the window's tokens, so consecutive windows overlap. `all_windows` applies it to every document.
- `collate` concatenates the sentences of each window into one token sequence, records each sentence's `(start, end)` span, pads the batch with the pad id, and returns the labels.

**Model.**
- `pool(H, spans)` makes one vector per sentence from the hidden states. It takes (1) the state at the sentence's last token (the newline), which in a causal model has seen the whole sentence, and (2) the mean over the sentence's tokens, computed efficiently with a cumulative sum. The two are concatenated (size 2d).
- `RRModel`: the pretrained decoder (`hidden` runs embeddings, layers and final norm), dropout, and a linear head from 2d to 13 labels. This `RRModel` and its pooling idea are reused in Phases 4, 5, 6 and 7.

**Metrics.** `rr_scores` computes per-label precision, recall and F1, macro averages and accuracy. `evaluate` predicts, takes the argmax per sentence, and scores overall and per domain. `print_rr` prints a summary.

**Prediction.** `predict_probs` slides the windows over each document and averages the class probabilities of sentences seen in more than one window. `write_test_predictions` saves predictions for all test documents, and separately for CL and IT, plus the probabilities.

**Trainer `train_rr`.** Seeds, builds windows, optional teacher (frozen), two learning rates (backbone 1e-4, head 1e-3), AdamW with a warm-up and linear decay. For each batch it computes class-weighted cross-entropy, and with a teacher adds a distillation term `T^2 * KL(softmax(student/T) || softmax(teacher/T))` mixed by `alpha`. After each epoch it evaluates on the validation set; the best trainable parameters are kept (only the parameters that were trainable), early stopping uses `patience`. At the end it reloads the best parameters, saves them, and returns scores, history, time, peak memory, trainable parameters and validation probabilities.

**Sanity checks.** For each mode it asserts that every sentence is covered by some window and that no window exceeds 1,024 tokens, then prints test document length statistics and the most common validation labels.

In [ ]:
DEBUG_MODE = False
assert "CustomDecoderLM" in globals()
TEACHER_CKPT = f"{DRIVE_PROJECT_DIR}/gqa_checkpoints/gqa_best.pt"
assert os.path.exists(TEACHER_CKPT), TEACHER_CKPT

def build_gqa_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)      # strict: pretrained Phase-1 weights
    return RRModel(lm_, 768, NC, dropout).to(device)

RR_RESULTS = {}
MODES = ["document"] if DEBUG_MODE else ["document", "local8", "sentence"]
for mode in MODES:
    model = build_gqa_rr()
    out = train_rr(model, f"gqa_full_{mode}", mode=mode, epochs=10, patience=3,
                   backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
    RR_RESULTS[mode] = out
    if not DEBUG_MODE:
        write_test_predictions(model, f"gqa_full_{mode}", mode)
    del model; torch.cuda.empty_cache()

print(f"\n{'mode':10s} {'F1(all)':>8s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
for m, o in RR_RESULTS.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{m:10s} {r['macro_f1']:8.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} "
          f"{r['macro_p']:7.4f} {r['macro_r']:7.4f} {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

**Code cell 37. What this cell does.** Full fine-tuning of the GQA model for rhetorical roles, in three context modes.

**Line by line**
- `TEACHER_CKPT` is the Phase-1 best checkpoint; `build_gqa_rr()` constructs the GQA language model, loads those pretrained weights (strictly), and wraps it in `RRModel` with a 13-way head.
- `MODES`: `document`, `local8`, `sentence` for the real run (only `document` in debug).
- For each mode: build a fresh model, train with `train_rr` (10 epochs, patience 3, backbone learning rate 1e-4, head 1e-3), store the result, write the test predictions, delete the model and clear GPU memory.
- The final table lists, per mode, overall macro-F1, the CL and IT scores, precision, recall, minutes and peak memory.

**Why three modes.** It answers the analysis question "does context matter?" by changing only how much surrounding text the model sees.

In [ ]:
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"
TEACHER_RR_CKPT = f"{P3_DIR}/rr_gqa_full_document_best.pt"
for p in (STAGE1_CKPT, TEACHER_RR_CKPT): assert os.path.exists(p), f"missing: {p}"
DEBUG_MODE = False   # True first, then False
STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)

def build_lm(cfg):
    return CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), max_seq_len=2048, tie_weights=True, **cfg)

# ---- teacher: the fine-tuned RR model ----
teacher = RRModel(build_lm(dict(d_model=768, n_layer=10, n_head=12, n_kv_head=4)), 768, NC).to(device)
teacher.load_state_dict(torch.load(TEACHER_RR_CKPT, map_location="cpu"), strict=False)
teacher.eval()
tres, _ = evaluate(teacher, val_docs, "document")
print("Teacher (loaded) dev:"); print_rr(tres)       # must match the earlier 0.5373, otherwise loading went wrong
assert abs(tres["macro_f1"] - 0.5373) < 0.01, "teacher did not load correctly"

def build_student():
    lm_ = build_lm(STUDENT_CFG)
    ck = torch.load(STAGE1_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)    # stage-1 student: random init, LM-distilled from the teacher
    return RRModel(lm_, STUDENT_CFG["d_model"], NC).to(device)

n_t = sum(p.numel() for p in teacher.lm.parameters()); n_s = sum(p.numel() for p in build_student().lm.parameters())
print(f"Backbone params: teacher {n_t/1e6:.1f}M | student {n_s/1e6:.1f}M ({100*n_s/n_t:.0f}%)")

STU = {}
for tag, alpha in [("kd", 0.5), ("nokd", 0.0)]:
    stu = build_student()
    out = train_rr(stu, f"student_{tag}", mode="document", teacher=teacher if alpha > 0 else None,
                   alpha=alpha, T=2.0, epochs=20, patience=5, backbone_lr=3e-4, head_lr=1e-3, debug=DEBUG_MODE)
    STU[tag] = out
    if not DEBUG_MODE: write_test_predictions(stu, f"student_{tag}_document", "document")
    del stu; torch.cuda.empty_cache()

print(f"\n{'model':16s} {'F1':>7s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'params':>9s} {'min':>5s} {'GB':>5s}")
r = tres; bd = r["by_domain"]
print(f"{'teacher (full)':16s} {r['macro_f1']:7.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} {r['macro_p']:7.4f} {r['macro_r']:7.4f} {n_t/1e6:8.1f}M {'-':>5s} {'-':>5s}")
for tag, o in STU.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{'student '+tag:16s} {r['macro_f1']:7.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} {r['macro_p']:7.4f} {r['macro_r']:7.4f} {n_s/1e6:8.1f}M {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

Teacher (loaded) dev:
ALL  macro-F1 0.5373 | P 0.5606 | R 0.5277 | acc 0.5630
  CL  macro-F1 0.4650 | P 0.4574 | R 0.5117 | acc 0.5408
  IT  macro-F1 0.5700 | P 0.6307 | R 0.5450 | acc 0.6046
Backbone params: teacher 87.5M | student 43.2M (49%)
[student_kd] mode=document windows=892 trainable=43,228,237 KD alpha=0.50 T=2.0
  ep 1 loss 4.2595 | val macro-F1 0.3708 CL 0.323 IT 0.407 | 0.3 min
  ep 2 loss 2.7617 | val macro-F1 0.3684 CL 0.310 IT 0.419 | 0.5 min
  ep 3 loss 1.7398 | val macro-F1 0.4007 CL 0.304 IT 0.437 | 0.8 min
  ep 4 loss 0.9599 | val macro-F1 0.4810 CL 0.386 IT 0.492 | 1.0 min
  ep 5 loss 0.6236 | val macro-F1 0.4440 CL 0.345 IT 0.512 | 1.3 min
  ep 6 loss 0.4504 | val macro-F1 0.4530 CL 0.380 IT 0.480 | 1.5 min
  ep 7 loss 0.2920 | val macro-F1 0.4037 CL 0.316 IT 0.481 | 1.8 min
  ep 8 loss 0.2146 | val macro-F1 0.4128 CL 0.396 IT 0.438 | 2.0 min
  ep 9 loss 0.1665 | val macro-F1 0.4256 CL 0.354 IT 0.470 | 2.3 min
  early stop
[student_kd] BEST on dev:
ALL  macro-F1 0

**Code cell 38. What this cell does.** Knowledge distillation for rhetorical roles: a smaller student (about 43M parameters) is trained with the fine-tuned GQA RR model as teacher, and again without a teacher for comparison.

**Line by line**
- `STAGE1_CKPT`: the student language model from Phase 2 (already distilled from the teacher on Phase-1 text). `TEACHER_RR_CKPT`: the fine-tuned teacher from the full fine-tuning run in `document` mode. Both files must exist.
- `build_lm(cfg)` builds a decoder of the given size.
- **Teacher**: an `RRModel` on the large decoder; `load_state_dict(..., strict=False)` loads the saved trainable parameters. It is evaluated on the dev set, and the cell asserts the macro-F1 is within 0.01 of the earlier value 0.5373. If not, loading failed, and every distillation result would be meaningless.
- `build_student()`: the small decoder with its stage-1 weights, wrapped in `RRModel` with a head of width 2 x 576.
- It prints the backbone parameter counts and their ratio.
- **Two runs**: `kd` (`alpha = 0.5`, temperature 2.0, teacher active) and `nokd` (`alpha = 0`, no teacher). Both train in `document` mode for up to 20 epochs with patience 5, backbone learning rate 3e-4. The test predictions are written for each.
- The final table puts the teacher and both students side by side (overall F1, CL, IT, precision, recall, parameters, time, memory).

**Why the `nokd` run.** The difference between the two students, with identical architecture and starting weights, isolates what the teacher's soft targets add.

In [ ]:
# --- longer-window context modes (GQA supports up to 2048 positions) ---
CTX["doc1536"] = dict(max_sents=10**9, stride_frac=0.75, max_tokens=1536); BW["doc1536"] = 5
CTX["doc2048"] = dict(max_sents=10**9, stride_frac=0.75, max_tokens=2048); BW["doc2048"] = 4

DEBUG_MODE = False    # True first, then False
assert "build_gqa_rr" in globals(), "run the teacher cell first (it defines build_gqa_rr)"
for mode in ["doc1536", "doc2048"]:
    W = all_windows(train_docs, mode); assert max(b - a for _, a, b in W) > 0
    print(mode, "windows:", len(W))

LONG = {}
for mode in (["doc2048"] if DEBUG_MODE else ["doc1536", "doc2048"]):
    model = build_gqa_rr()
    out = train_rr(model, f"gqa_full_{mode}", mode=mode, epochs=10, patience=3,
                   backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
    LONG[mode] = out
    if not DEBUG_MODE: write_test_predictions(model, f"gqa_full_{mode}", mode)
    del model; torch.cuda.empty_cache()

print(f"\n{'mode':10s} {'F1(all)':>8s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s} {'min':>5s} {'GB':>5s}")
print(f"{'document':10s} {0.5373:8.4f} {0.4650:7.4f} {0.5700:7.4f} {0.5606:7.4f} {0.5277:7.4f}   (1024, earlier run)")
for m, o in LONG.items():
    r = o["res"]; bd = r["by_domain"]
    print(f"{m:10s} {r['macro_f1']:8.4f} {bd['CL']['macro_f1']:7.4f} {bd['IT']['macro_f1']:7.4f} "
          f"{r['macro_p']:7.4f} {r['macro_r']:7.4f} {o['minutes']:5.1f} {o['peak_gb']:5.1f}")

doc1536 windows: 598
doc2048 windows: 455
[gqa_full_doc1536] mode=doc1536 windows=598 trainable=87,526,669 
  ep 1 loss 2.2291 | val macro-F1 0.4089 CL 0.334 IT 0.413 | 0.3 min
  ep 2 loss 0.9495 | val macro-F1 0.4577 CL 0.451 IT 0.473 | 0.7 min
  ep 3 loss 0.4158 | val macro-F1 0.5023 CL 0.412 IT 0.512 | 1.0 min
  ep 4 loss 0.1855 | val macro-F1 0.4911 CL 0.410 IT 0.549 | 1.3 min
  ep 5 loss 0.1156 | val macro-F1 0.5376 CL 0.503 IT 0.533 | 1.7 min
  ep 6 loss 0.0561 | val macro-F1 0.5025 CL 0.466 IT 0.484 | 2.0 min
  ep 7 loss 0.0199 | val macro-F1 0.5022 CL 0.470 IT 0.506 | 2.4 min
  ep 8 loss 0.0121 | val macro-F1 0.5277 CL 0.456 IT 0.547 | 2.7 min
  early stop
[gqa_full_doc1536] BEST on dev:
ALL  macro-F1 0.5376 | P 0.5832 | R 0.5411 | acc 0.5728
  CL  macro-F1 0.5027 | P 0.5034 | R 0.5416 | acc 0.5630
  IT  macro-F1 0.5329 | P 0.6298 | R 0.5048 | acc 0.5912
Saved prediction_gqa_full_doc1536_{all,CL,IT}.json (10 docs, 2197 sentences)
[gqa_full_doc2048] mode=doc2048 windows=455 trai

**Code cell 39. What this cell does.** Tests whether longer context windows help: the document mode with 1,536 and 2,048 tokens instead of 1,024.

**Line by line**
- Adds two entries to `CTX` (the window definitions) with `max_tokens` of 1,536 and 2,048, and to `BW` batch sizes of 5 and 4 windows (longer windows need smaller batches to fit in memory). The GQA model was built for 2,048 positions, so 2,048 is its limit.
- Prints the number of windows for each mode.
- For each mode: build a fresh pretrained model, train with `train_rr` (10 epochs, patience 3), write test predictions.
- The table prints the earlier 1,024-token result (hard-coded from the previous run: 0.5373 overall, CL 0.4650, IT 0.5700, precision 0.5606, recall 0.5277) next to the two new rows, so the comparison is on one screen.

In [ ]:
import math
USE_POS, POS_SCALE, POS_DIM = True, 3.0, 10

def collate(wins, docs):
    seqs, spans, ys = [], [], []
    for di, a, b in wins:
        d = docs[di]; n = len(d["sent_ids"]); ids, sp = [], []
        for k in range(a, b):
            s = len(ids); ids += d["sent_ids"][k]
            sp.append((s, len(ids), k / max(n - 1, 1), math.log(n) / 8))     # + relative position, doc length
        seqs.append(ids); spans.append(sp)
        if d["y"] is not None: ys += d["y"][a:b]
    L = max(len(s) for s in seqs)
    x = torch.full((len(seqs), L), PAD_ID, dtype=torch.long)
    for j, s in enumerate(seqs): x[j, :len(s)] = torch.tensor(s)
    return x, spans, (torch.tensor(ys) if ys else None)

def pool(H, spans):
    jw, st, en, rr, ll = [], [], [], [], []
    for j, sp in enumerate(spans):
        for s, e, r, l in sp: jw.append(j); st.append(s); en.append(e); rr.append(r); ll.append(l)
    jw, st, en = [torch.tensor(t, device=H.device) for t in (jw, st, en)]
    last = H[jw, en - 1]
    cs = torch.cat([torch.zeros_like(H[:, :1]).float(), H.float().cumsum(1)], dim=1)
    mean = ((cs[jw, en] - cs[jw, st]) / (en - st).unsqueeze(1)).to(H.dtype)
    feats = [last, mean]
    if USE_POS:
        r = torch.tensor(rr, device=H.device, dtype=torch.float32)
        f = [r.unsqueeze(1)] + [fn(2 * math.pi * k * r).unsqueeze(1) for k in (1, 2, 3, 4) for fn in (torch.sin, torch.cos)] \
            + [torch.tensor(ll, device=H.device, dtype=torch.float32).unsqueeze(1)]
        feats.append((torch.cat(f, 1) * POS_SCALE).to(H.dtype))
    return torch.cat(feats, -1)

class RRModel(nn.Module):
    def __init__(self, lm, d_model, n_labels=NC, dropout=0.1):
        super().__init__()
        self.lm, self.drop = lm, nn.Dropout(dropout)
        self.head = nn.Linear(2 * d_model + (POS_DIM if USE_POS else 0), n_labels)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
    def hidden(self, ids):
        lm = self.lm; x = lm.token_embedding(ids)
        for layer in lm.layers: x, _ = layer(x, lm.cos, lm.sin, past_kv=None)
        return lm.norm(x)
    def forward(self, ids, spans): return self.head(self.drop(pool(self.hidden(ids), spans)))

DEBUG_MODE = False     # True first, then False
model = build_gqa_rr()
out_pos = train_rr(model, "gqa_full_document_pos", mode="document", epochs=10, patience=3,
                   backbone_lr=1e-4, head_lr=1e-3, debug=DEBUG_MODE)
if not DEBUG_MODE: write_test_predictions(model, "gqa_full_document_pos", "document")

**Code cell 40. What this cell does.** Adds position information to the sentence features and trains again.

**Why.** In a rhetorical-role task the role of a sentence depends strongly on where it sits in the judgment (facts come early, the ruling at the end). A causal decoder knows only its position within the window, not within the document.

**Line by line**
- `USE_POS, POS_SCALE, POS_DIM = True, 3.0, 10`: the switch, a scale for the extra features, and their number.
- The new `collate` records, for each sentence, its relative position `k / (n - 1)` in the document and `log(n) / 8` (a rough measure of document length) next to its token span.
- The new `pool` builds the usual `[last token ; mean]` vector and appends 10 extra features: the relative position `r`, `sin(2 pi k r)` and `cos(2 pi k r)` for k = 1..4 (a Fourier encoding that lets a linear layer represent smooth and periodic position effects), and the length feature, all multiplied by `POS_SCALE` so they are not drowned out by the large hidden vectors.
- `RRModel` is redefined with a head of width `2 * d_model + 10`.
- A fresh pretrained model is trained in `document` mode, and the test predictions are written. The result is stored as `out_pos`.

In [ ]:
assert "RR_RESULTS" in globals() and "document" in RR_RESULTS, "re-run the teacher cell for mode 'document' first"
devP = RR_RESULTS["document"]["val_probs"]
testP = pickle.load(open(f"{P3_DIR}/probs_test_gqa_full_document.pkl", "rb"))

def trans_matrix(docs, alpha=1.0):
    C = np.full((NC, NC), alpha)
    for d in docs:
        for a, b in zip(d["y"][:-1], d["y"][1:]): C[a, b] += 1
    return np.log(C / C.sum(1, keepdims=True))

def viterbi(logp, logA, lam):
    n = logp.shape[0]; dp = np.zeros((n, NC)); bp = np.zeros((n, NC), int); dp[0] = logp[0]
    for t in range(1, n):
        s = dp[t - 1][:, None] + lam * logA          # (prev, cur)
        bp[t] = s.argmax(0); dp[t] = s.max(0) + logp[t]
    path = [int(dp[-1].argmax())]
    for t in range(n - 1, 0, -1): path.append(int(bp[t][path[-1]]))
    return path[::-1]

logA = trans_matrix(train_docs)
def decode(P, docs, lam):
    return {d["id"]: (viterbi(np.log(P[d["id"]] + 1e-9), logA, lam) if lam > 0 else P[d["id"]].argmax(1).tolist()) for d in docs}

def score_dev(preds):
    G = {k: ([], []) for k in ("CL", "IT", "ALL")}
    for d in val_docs:
        for k in (d["domain"], "ALL"): G[k][0].extend(d["y"]); G[k][1].extend(preds[d["id"]])
    return {k: rr_scores(*G[k]) for k in G}

print(f"{'lambda':>7s} {'F1':>7s} {'CL':>7s} {'IT':>7s} {'P':>7s} {'R':>7s}")
best_lam, best_f1 = 0, -1
devP = out_pos["val_probs"]; testP = pickle.load(open(f"{P3_DIR}/probs_test_gqa_full_document_pos.pkl", "rb"))
for lam in [0, 0.5, 1, 2]:
    r = score_dev(decode(devP, val_docs, lam))
    print(f"lam {lam}: F1 {r['ALL']['macro_f1']:.4f} CL {r['CL']['macro_f1']:.4f} IT {r['IT']['macro_f1']:.4f}")
    if r["ALL"]["macro_f1"] > best_f1: best_lam, best_f1 = lam, r["ALL"]["macro_f1"]
print("best lambda:", best_lam)

pred = {d["id"]: [LABELS[i] for i in decode(testP, [d], best_lam)[d["id"]]] for d in test_docs}
for d in test_docs: assert len(pred[d["id"]]) == len(d["text"])
for suf, sel in [("all", pred), ("CL", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                 ("IT", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
    json.dump(sel, open(f"{P3_DIR}/prediction_gqa_full_document_viterbi_{suf}.json", "w"), ensure_ascii=False)
print("saved prediction_gqa_full_document_viterbi_{all,CL,IT}.json")

 lambda      F1      CL      IT       P       R
lam 0: F1 0.5290 CL 0.4269 IT 0.5452
lam 0.5: F1 0.5310 CL 0.4262 IT 0.5467
lam 1: F1 0.5431 CL 0.4493 IT 0.5402
lam 2: F1 0.5423 CL 0.5182 IT 0.5367
best lambda: 1
saved prediction_gqa_full_document_viterbi_{all,CL,IT}.json


**Code cell 41. What this cell does.** Post-processing with Viterbi decoding: instead of choosing each sentence's label independently, find the label sequence that is best overall, using how often one label follows another in the training data.

**Line by line**
- It loads the validation probabilities of the position model and the saved test probabilities.
- `trans_matrix(docs, alpha=1.0)`: counts how often label a is followed by label b in the training documents, adds 1 to every count (smoothing so unseen transitions are not impossible), normalises each row and takes logs. This is `logA`.
- `viterbi(logp, logA, lam)`: dynamic programming over sentences. `dp[t][c]` is the best score of any label path ending in label c at sentence t; the score of a path is the sum of log-probabilities of its labels plus `lam` times the log transition probabilities. `bp` stores the best previous label, and the best path is read back from the end. `lam` controls how much the transitions matter; 0 means no transitions (plain argmax).
- `decode` applies it per document; `score_dev` computes macro-F1 overall and per domain.
- It tries `lam` in 0, 0.5, 1, 2, prints the dev scores, and keeps the best.
- It then decodes the test documents with the best `lam`, checks the lengths, and saves `prediction_gqa_full_document_viterbi_{all,CL,IT}.json`.

**Why.** Rhetorical roles come in runs and typical orders; the transition table lets the model fix isolated odd labels.

In [ ]:
import itertools
assert "decode" in globals() and "score_dev" in globals() and "logA" in globals(), "run the earlier Viterbi cell first (it defines these)"

cands = {}
def add(tag, getter, pkl):
    try:
        dv = getter(); tp = pickle.load(open(f"{P3_DIR}/probs_test_{pkl}.pkl", "rb"))
        assert set(dv) == {d["id"] for d in val_docs}, "dev probs are from a debug run"
        cands[tag] = (dv, tp)
    except Exception as e: print(f"skip {tag}: {type(e).__name__}")
add("doc1024", lambda: RR_RESULTS["document"]["val_probs"], "gqa_full_document")
add("local8",  lambda: RR_RESULTS["local8"]["val_probs"],   "gqa_full_local8")
add("sentence",lambda: RR_RESULTS["sentence"]["val_probs"], "gqa_full_sentence")
add("pos",     lambda: out_pos["val_probs"],                "gqa_full_document_pos")
add("doc1536", lambda: LONG["doc1536"]["val_probs"],        "gqa_full_doc1536")
add("doc2048", lambda: LONG["doc2048"]["val_probs"],        "gqa_full_doc2048")
add("stu_kd",  lambda: STU["kd"]["val_probs"],              "student_kd_document")
add("stu_nokd",lambda: STU["nokd"]["val_probs"],            "student_nokd_document")
names = list(cands); print("available:", names)

def avg(ps): return {k: np.mean([p[k] for p in ps], 0) for k in ps[0]}
def dev_f1(sub, lam):
    return score_dev(decode(avg([cands[n][0] for n in sub]), val_docs, lam))["ALL"]["macro_f1"]

for n in names: print(f"  single {n:9s} dev F1 {dev_f1([n], 0):.4f}")
rows = sorted(((dev_f1(s, 0), s) for k in range(2, len(names) + 1) for s in itertools.combinations(names, k)), reverse=True)
print(f"\nall {len(names)} models: lam0 {dev_f1(names, 0):.4f} | lam1 {dev_f1(names, 1):.4f}")
print("top subsets (dev, selected on 10 docs so slightly optimistic):")
for f, s in rows[:6]: print(f"  {f:.4f} (lam1 {dev_f1(s, 1):.4f})  {s}")

def write_ens(sub, lam, tag):
    P = avg([cands[n][1] for n in sub]); pr = decode(P, test_docs, lam)
    pred = {d["id"]: [LABELS[i] for i in pr[d["id"]]] for d in test_docs}
    for d in test_docs: assert len(pred[d["id"]]) == len(d["text"])
    for suf, sel in [("all", pred), ("CL", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "CL"}),
                     ("IT", {d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == "IT"})]:
        json.dump(sel, open(f"{P3_DIR}/prediction_{tag}_{suf}.json", "w"), ensure_ascii=False)
    print("saved", tag, "<-", sub, f"lam={lam}")

best = rows[0][1]
for lam in (0, 1):
    write_ens(names, lam, f"ens_all_l{lam}")
    write_ens(best, lam, f"ens_best_l{lam}")

available: ['doc1024', 'local8', 'sentence', 'pos', 'doc1536', 'doc2048', 'stu_kd', 'stu_nokd']
  single doc1024   dev F1 0.5373
  single local8    dev F1 0.4402
  single sentence  dev F1 0.3497
  single pos       dev F1 0.5290
  single doc1536   dev F1 0.5376
  single doc2048   dev F1 0.5614
  single stu_kd    dev F1 0.4810
  single stu_nokd  dev F1 0.4773

all 8 models: lam0 0.5564 | lam1 0.5283
top subsets (dev, selected on 10 docs so slightly optimistic):
  0.5718 (lam1 0.5413)  ('doc1024', 'doc2048', 'stu_nokd')
  0.5701 (lam1 0.5594)  ('doc1536', 'doc2048', 'stu_kd', 'stu_nokd')
  0.5688 (lam1 0.5631)  ('doc1024', 'doc2048', 'stu_kd')
  0.5684 (lam1 0.5380)  ('doc1024', 'local8', 'doc1536', 'doc2048', 'stu_kd')
  0.5678 (lam1 0.5551)  ('doc1024', 'doc1536', 'doc2048', 'stu_kd')
  0.5670 (lam1 0.5302)  ('doc2048', 'stu_nokd')
saved ens_all_l0 <- ['doc1024', 'local8', 'sentence', 'pos', 'doc1536', 'doc2048', 'stu_kd', 'stu_nokd'] lam=0
saved ens_best_l0 <- ('doc1024', 'doc2048', 's

**Code cell 42. What this cell does.** Ensembling: averages the predicted probabilities of several earlier runs and checks which subset is best on the dev set.

**Line by line**
- `cands` collects, for each earlier run, its dev probabilities and saved test probabilities (document 1024, local8, sentence, position model, 1536, 2048, and the two students). `add` skips a run if it is missing or if its dev probabilities come from a debug run.
- `avg(ps)` averages probability arrays document by document. `dev_f1(sub, lam)` is the dev macro-F1 of an averaged subset with Viterbi weight `lam`.
- It prints the single-model scores, then evaluates every subset of two or more models (`itertools.combinations`) and sorts them by dev F1. The comment notes that picking the best subset on 10 dev documents is slightly optimistic.
- `write_ens(sub, lam, tag)` averages the saved test probabilities, decodes (with or without Viterbi) and writes the three prediction files.
- Finally it writes four ensembles: all models and the best subset, each with `lam = 0` and `lam = 1`.

**Why.** Different context windows make different mistakes; averaging reduces variance. This is a legitimate use of our own models only, with no outside model involved.

## Court Judgement Prediction

In [ ]:
import json, os, numpy as np
from huggingface_hub import snapshot_download

P4_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_4"
os.makedirs(P4_DIR, exist_ok=True)

# Downloads only the Phase-4 folder from the course dataset repo (same repo as Phase 3).
# If the folder name differs, change allow_patterns.
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_4_CJPE/*", "*/Phase_4_CJPE/*"],
                          token=HF_TOKEN)
print("downloaded to:", local)
for r, _, fs in os.walk(local):
    for f in fs:
        if f.endswith(".json"): print(os.path.join(r, f))

def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_4" in r: return os.path.join(r, name)

def load(name):
    d = json.load(open(find(name)))
    print(name, type(d).__name__, len(d))
    return d

tr, dv, te = load("train.json"), load("dev.json"), load("test.json")

def peek(d):
    x = d[0] if isinstance(d, list) else {k: (v[0] if isinstance(v, list) else v) for k, v in d.items()}
    print({k: (str(v)[:80] + "…" if len(str(v)) > 80 else v) for k, v in x.items()})
for n, d in [("train", tr), ("dev", dv), ("test", te)]:
    print(n); peek(d)

# label balance + token lengths (uses your Phase-1 tokenizer from the setup cell)
def rows(d):
    return d if isinstance(d, list) else [dict(zip(d.keys(), v)) for v in zip(*d.values())]
for n, d in [("train", tr), ("dev", dv)]:
    R = rows(d)
    labs = np.array([r["label"] for r in R])
    lens = np.array([len(tokenizer.encode(r["text"]).ids) for r in R[:300]])
    print(n, "label counts:", np.bincount(labs), "| tokens (first 300 docs) median/p90/max:",
          int(np.median(lens)), int(np.percentile(lens, 90)), int(lens.max()))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Test/test.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/train.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_4_CJPE/Train/dev.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_2_LNER/Test/fold3-text.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_2_LNER/Train/fold1-text.json
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be7

**Code cell 43. What this cell does.** Downloads the Phase-4 data (court judgment prediction, CJPE) and takes a first look at it.

**Task.** Each document is a court judgment, and the label says whether the appeal was accepted or rejected (binary). The metric is macro-F1.

**Line by line**
- `P4_DIR`: the Phase-4 folder on Drive.
- `snapshot_download(..., allow_patterns=["Phase_4_CJPE/*", "*/Phase_4_CJPE/*"])` downloads only the Phase-4 files from the dataset repository and returns the local folder; the loop prints every JSON file found.
- `find(name)` locates a file by name inside the `Phase_4` folder; `load(name)` reads it and prints its type and size. `tr, dv, te` are the train, dev and test sets.
- `peek(d)` prints the first record with long values shortened, to show the fields.
- `rows(d)` converts a dictionary-of-lists layout into a list of records. For train and dev it prints the label counts (class balance) and the token length statistics (median, 90th percentile, maximum) of the first 300 documents, tokenized with our own tokenizer. These statistics decided the 4,096-token head and tail views used later.

In [ ]:
import pickle, time, numpy as np, os

CACHE = f"{P4_DIR}/cjpe_tok_cache.pkl"
HEAD_N = TAIL_N = 4096

def rows_of(d): return d
def tok_split(texts):
    out = []
    for i in range(0, len(texts), 500):
        batch = texts[i:i+500]
        # only the first/last 40k chars can reach the first/last 4096 tokens
        h = [t[:40000] for t in batch]
        tl = [t[-40000:] if len(t) > 80000 else None for t in batch]
        eh = tokenizer.encode_batch(h)
        et = tokenizer.encode_batch([x for x in tl if x is not None]) if any(x is not None for x in tl) else []
        k = 0
        for j, t in enumerate(batch):
            hi = eh[j].ids
            if tl[j] is None:                      # short enough: whole doc encoded
                ti, total = hi, len(hi)
            else:
                ti, total = et[k].ids, -1; k += 1  # total unknown (very long)
            out.append((np.array(hi[:HEAD_N], dtype=np.uint16),
                        np.array(ti[-TAIL_N:], dtype=np.uint16),
                        total if total >= 0 else 10**6))
    return out

if os.path.exists(CACHE):
    C = pickle.load(open(CACHE, "rb")); print("loaded cache")
else:
    t0 = time.time(); C = {}
    for name, d in [("train", tr), ("dev", dv), ("test", te)]:
        texts = [r["text"] for r in d]
        toks = tok_split(texts)
        C[name] = dict(ids=[r["id"] for r in d],
                       y=(np.array([r["label"] for r in d]) if name != "test" else None),
                       head=[a for a, _, _ in toks], tail=[b for _, b, _ in toks],
                       n=np.array([n for _, _, n in toks]))
        print(name, len(d), f"{(time.time()-t0)/60:.1f} min")
    pickle.dump(C, open(CACHE, "wb"))

# sanity: test 'label' field must be ignored; just check if it is a constant placeholder
print("test label field values:", np.unique([r.get("label") for r in te], return_counts=True))
print("train docs ≤4096 tok:", (C["train"]["n"] <= 4096).mean().round(3),
      "| ≤8192:", (C["train"]["n"] <= 8192).mean().round(3))
print("✅ cache ready:", {k: len(v["ids"]) for k, v in C.items()})

train 37387 1.5 min
dev 3505 1.7 min
test 1517 1.7 min
test label field values: (array([0, 1]), array([755, 762]))
train docs ≤4096 tok: 0.694 | ≤8192: 0.948
✅ cache ready: {'train': 37387, 'dev': 3505, 'test': 1517}


**Code cell 44. What this cell does.** Tokenizes all Phase-4 documents once, keeps the first and last 4,096 tokens of each, and caches the result on Drive so no later run has to tokenize again.

**Line by line**
- `HEAD_N = TAIL_N = 4096`: how many tokens to keep from the start and from the end of each document.
- `tok_split(texts)` works in batches of 500 documents. Only the first 40,000 characters can contain the first 4,096 tokens, so only those are encoded for the head (`h`). For documents longer than 80,000 characters the last 40,000 characters are also encoded (`tl`) for the tail; shorter documents are encoded whole and the tail is cut from the same ids. `encode_batch` tokenizes a list of strings. For each document it stores `head` (first 4,096 ids), `tail` (last 4,096 ids) and the total token count `n` (set to 10^6 as a marker when the document is too long to know exactly). Ids are stored as `uint16` (they are below 65,536), halving the memory.
- If the cache file exists it is loaded; otherwise each split (train, dev, test) is tokenized and saved with the document ids, labels (none for test), heads, tails and lengths. The time per split is printed.
- The last lines print a check on the test file's label field (it should be a constant placeholder; the field is never used for anything) and the share of training documents that fit in 4,096 and 8,192 tokens.

**Why head and tail.** Long judgments state the facts at the start and the decision at the end; keeping both ends is a cheap way to cover a document much longer than the model's 2,048-token window.

In [ ]:
def build_gqa_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)      # strict: pretrained Phase-1 weights
    return RRModel(lm_, 768, NC, dropout).to(device)


def cj_from(rr):
    """Phase-3 RRModel -> CJPE model: keep the Phase-1 LM, discard the old head, fresh 2-class head."""
    d = rr.lm.token_embedding.weight.shape[1]          # model width, independent of the old head
    m = CJModel(rr.lm, d, 2, 0.1).to(device)
    m.head = nn.Linear(2 * d, 2).to(device)
    nn.init.normal_(m.head.weight, std=0.02); nn.init.zeros_(m.head.bias)
    return m

**Code cell 45. What this cell does.** Two small helpers for building Phase-4 models.

**`build_gqa_rr`**: the same function as in Phase 3: it builds the GQA decoder, loads the pretrained Phase-1 weights strictly, and wraps it in `RRModel`.

**`cj_from(rr)`**: converts that model into a court-judgment classifier. It takes the language model inside (`rr.lm`), reads its width from the embedding matrix, makes a `CJModel` (the document classifier from the Phase-4 toolkit), and replaces the head with a fresh `Linear(2d, 2)` initialised with a small normal distribution and zero bias. The old 13-class head is discarded, so no rhetorical-role knowledge leaks in; only the Phase-1 language model is reused.

In [ ]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

CH = 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}   # value = max chunks per doc

# ---------- document -> chunks (from the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):                      # align chunks to the END of the judgment
        ends = list(range(L, 0, -CH))[::-1]
        return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

def collate_cj(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> mean over chunks -> 2-class head ----------
class CJModel(RRModel):
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids)                                            # (chunks, T, d)
        B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        f = torch.cat([last, mean], -1)                                 # (chunks, 2d)
        D = torch.zeros(n_docs, f.size(1), device=f.device).index_add_(0, owner, f)
        D = D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]
        return self.head(self.drop(D))

def cj_from(rr):
    """Turn a Phase-3 RRModel (13-label head) into a CJPE model: same LM, fresh 2-class head."""
    d = rr.head.in_features // 2
    return CJModel(rr.lm, d, 2, 0.1).to(device)

# ---------- inference / metrics ----------
@torch.no_grad()
def cj_logits(model, split, mode, idxs=None, chunk_batch=32):
    model.eval()
    idxs = np.arange(len(C[split]["ids"])) if idxs is None else np.asarray(idxs)
    bs = max(1, chunk_batch // MODES[mode]); out = np.zeros((len(idxs), 2), np.float32)
    for s in range(0, len(idxs), bs):
        b = idxs[s:s+bs]
        with torch.autocast("cuda", dtype=torch.bfloat16):
            lg = model(*collate_cj(split, b, mode))
        out[s:s+len(b)] = lg.float().cpu().numpy()
    return out

def cj_scores(y, lg, thr=0.5):
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pr = (p1 > thr).astype(int)
    P, R, Fm, _ = precision_recall_fscore_support(y, pr, average="macro", zero_division=0)
    return dict(f1=Fm, p=P, r=R, cm=confusion_matrix(y, pr, labels=[0, 1]))

def best_thr(y, lg):
    best = (0.5, -1.0)
    for t in np.arange(0.2, 0.81, 0.02):
        f = cj_scores(y, lg, t)["f1"]
        if f > best[1]: best = (float(t), f)
    return best

# ---------- training (CE, or KD from cached teacher logits) ----------
def train_cj(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, bs_chunks=16,
             n_train=None, kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3,
             class_weight=True, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed)
    ytr = C["train"]["y"]; ydv = C["dev"]["y"]
    dev_idx = np.arange(200) if debug else np.arange(len(ydv))
    tr_idx = rng.permutation(len(ytr))[: (128 if debug else (n_train or len(ytr)))]
    if debug: epochs, evals_per_epoch = 1, 1
    cw = None
    if class_weight:
        cnt = np.bincount(ytr, minlength=2); cw = torch.tensor(len(ytr) / (2 * cnt), dtype=torch.float, device=device)
    kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None

    head_p = [p for p in model.head.parameters()]
    hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    bs = max(1, bs_chunks // MODES[mode]); spe = math.ceil(len(tr_idx) / bs); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P4_DIR}/cj_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time()
    torch.cuda.reset_peak_memory_stats(); hist = []; done = False
    for ep in range(epochs):
        perm = rng.permutation(tr_idx); model.train()
        for s in range(0, len(perm), bs):
            b = perm[s:s+bs]
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_cj("train", b, mode)).float()
            y = torch.tensor(ytr[b], device=device)
            loss = F.cross_entropy(lg, y, weight=cw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kl = F.kl_div(F.log_softmax(lg / T, -1), F.softmax(t_lg / T, -1), reduction="batchmean") * T * T
                loss = (1 - alpha) * loss + alpha * kl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 100 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                lgd = cj_logits(model, "dev", mode, dev_idx); sc = cj_scores(ydv[dev_idx], lgd)
                hist.append((step, sc["f1"])); model.train()
                print(f"  eval@{step}: macro-F1 {sc['f1']:.4f}  P {sc['p']:.4f}  R {sc['r']:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if sc["f1"] > best[0]:
                    best = (sc["f1"], step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path):
        model.load_state_dict(torch.load(path), strict=False)
    lgd = cj_logits(model, "dev", mode, dev_idx); y = ydv[dev_idx]
    sc = cj_scores(y, lgd); thr, f_thr = best_thr(y, lgd)
    res = dict(name=name, mode=mode, f1=sc["f1"], p=sc["p"], r=sc["r"], cm=sc["cm"].tolist(), thr=thr, f1_thr=f_thr,
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9,
               trainable=n_tr, total=n_all, best_step=best[1], dev_logits=lgd)
    print(f"✅ [{name}] dev macro-F1 {sc['f1']:.4f} (P {sc['p']:.4f} R {sc['r']:.4f}) | tuned thr {thr:.2f} → {f_thr:.4f} "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB\n   confusion [[TN,FP],[FN,TP]] = {res['cm']}")
    return res

# ---------- outputs ----------
def write_pred(model, name, mode, thr=0.5):
    lg = cj_logits(model, "test", mode)
    p1 = torch.softmax(torch.tensor(lg), -1)[:, 1].numpy(); pred = (p1 > thr).astype(int)
    out = [{"id": i, "pred": int(p)} for i, p in zip(C["test"]["ids"], pred)]
    assert len(out) == len(C["test"]["ids"]) == 1517
    f = f"{P4_DIR}/prediction_{name}.json"; json.dump(out, open(f, "w"))
    np.save(f"{P4_DIR}/test_logits_{name}.npy", lg)
    print(f"wrote {f} | preds 0/1 = {np.bincount(pred, minlength=2).tolist()}  (rename to prediction.json when uploading)")

def save_kd_logits(model, name, mode):
    lg = cj_logits(model, "train", mode); np.save(f"{P4_DIR}/kd_logits_{name}.npy", lg); return lg

print("✅ Phase-4 toolkit ready")

✅ Phase-4 toolkit ready


**Code cell 46. What this cell does.** The Phase-4 toolkit: splits each judgment into 1,024-token chunks, defines the classifier, and provides evaluation, training and output-writing functions.

**Chunking**
- `CH = 1024`. `MODES` maps each document view to the maximum number of chunks: `first1` (first chunk), `last1` (last chunk), `first4` and `last4` (first or last 4,096 tokens as four chunks) and `ht4` (head + tail: the first 2,048 and last 2,048 tokens, or the whole document if it is at most 4,096 tokens).
- `get_chunks(split, i, mode)` takes the cached head or tail tokens of document i and cuts the chosen view into chunks. For `last` views the chunks are aligned to the end of the judgment, so the final chunk is always complete.
- `collate_cj` builds a batch: all chunks of all documents in the batch are stacked into one padded matrix, with their lengths and an `owner` index saying which document each chunk belongs to (an empty document gets a single dummy token).

**Model `CJModel`** (inherits from `RRModel` so it reuses `hidden`, dropout and head)
- Runs every chunk through the decoder and builds a feature per chunk: the state at the last real token and the mean over real tokens (a mask excludes padding), concatenated (2d).
- Averages the chunk features per document with `index_add_` and a count (`bincount`), giving one vector per document, and applies dropout and the head. So a long document is represented by the mean of its chunk representations.
- `cj_from(rr)` makes a `CJModel` from a model's language model with a fresh head (the final definition is in the previous cell).

**Evaluation**
- `cj_logits` runs a split in batches (chunk count limited by `chunk_batch`) under bfloat16 and returns logits.
- `cj_scores` turns the class-1 probability into a prediction at a threshold and returns macro-F1, precision, recall and the confusion matrix. `best_thr` scans thresholds from 0.2 to 0.8 and returns the best one on the dev set.

**Training `train_cj`**
- Class weights `n / (2 * count)` handle label imbalance. If distillation logits (`kd`) are given they are teacher logits for the training documents.
- Two learning rates: `lr` for the backbone (or adapters) and `head_lr` for the head. AdamW, 6% warm-up, linear decay. `evals_per_epoch` evaluations per epoch; the best parameters (only the trainable ones) are saved; early stopping uses `patience`.
- Each step: forward under bfloat16, class-weighted cross-entropy; with a teacher, `(1 - alpha) * CE + alpha * T^2 * KL(student/T, teacher/T)`; gradient clipping at 1.0.
- After step 20 it prints an estimate of the total training time. At the end it reloads the best parameters, scores the dev set, finds the best threshold, and returns a result dictionary (scores, confusion matrix, threshold, minutes, peak memory, trainable and total parameters, dev logits).

**Outputs.** `write_pred` writes the test predictions as a JSON list of `{id, pred}` (asserting 1,517 entries) and saves the logits; `save_kd_logits` stores a model's logits on the training documents so a student can later learn from them without re-running the teacher.

In [ ]:
TEACHER_CKPT = f"{DRIVE_PROJECT_DIR}/gqa_checkpoints/gqa_best.pt"
NC = 13                       # only used to build the old Phase-3 head, which cj_from() then replaces

DEBUG   = False              # True = 128 train docs / 200 dev docs, ~1-2 min. Set False for the real run.
MODE    = "last4"             # last1 | first1 | first4 | last4 | ht4
N_TRAIN = None                # None = all 37k docs; set e.g. 15000 if the time estimate is too long
EPOCHS  = 2

rr = build_gqa_rr()
model = cj_from(rr); del rr
NAME = f"gqa_full_{MODE}"
CJ = globals().get("CJ", {})
CJ[NAME] = train_cj(model, NAME, MODE, epochs=EPOCHS, lr=5e-5, head_lr=1e-3,
                    n_train=N_TRAIN, debug=DEBUG)
if not DEBUG:
    write_pred(model, NAME, MODE)

[gqa_full_last4] mode=last4 train_docs=37387 steps=18694 trainable=87.51M / 87.5M
  ~50 min training total (excl. dev evals)
  step 100/18694 loss 0.9508
  step 200/18694 loss 0.4985
  step 300/18694 loss 0.6891
  step 400/18694 loss 0.7122
  step 500/18694 loss 1.6616
  step 600/18694 loss 1.7081
  step 700/18694 loss 1.6836
  step 800/18694 loss 0.9579
  step 900/18694 loss 0.4123
  step 1000/18694 loss 1.2417
  step 1100/18694 loss 1.6231
  step 1200/18694 loss 0.0806
  step 1300/18694 loss 0.1456
  step 1400/18694 loss 1.1627
  step 1500/18694 loss 1.0129
  step 1600/18694 loss 0.4453
  step 1700/18694 loss 0.0780
  step 1800/18694 loss 0.8932
  step 1900/18694 loss 0.9170
  step 2000/18694 loss 0.6645
  step 2100/18694 loss 0.8497
  step 2200/18694 loss 0.5085
  step 2300/18694 loss 2.6926
  step 2400/18694 loss 0.7140
  step 2500/18694 loss 0.1038
  step 2600/18694 loss 2.4826
  step 2700/18694 loss 3.3922
  step 2800/18694 loss 0.4142
  step 2900/18694 loss 0.1827
  step 3000/18

**Code cell 47. What this cell does.** Fine-tunes the whole GQA model for court judgment prediction (full fine-tuning, the Phase-4 teacher).

**Line by line**
- `TEACHER_CKPT` is the Phase-1 checkpoint. `NC = 13` only exists so that `build_gqa_rr` can create a model with the old 13-class head, which is replaced straight away.
- Settings: `DEBUG` (a 1-2 minute run on 128 train and 200 dev documents), `MODE = "last4"` (the last 4,096 tokens of each judgment, where the decision usually appears), `N_TRAIN = None` (all about 37,000 training documents), `EPOCHS = 2`.
- `rr = build_gqa_rr(); model = cj_from(rr); del rr` builds the pretrained model with a fresh 2-class head and discards the temporary wrapper.
- `CJ` is a dictionary collecting the results of every Phase-4 run. `train_cj(model, NAME, MODE, epochs=2, lr=5e-5, head_lr=1e-3, ...)` trains all 87.5M parameters with a small backbone learning rate (5e-5, to avoid destroying the pretrained weights) and a larger head learning rate (1e-3).
- Unless debugging, `write_pred` writes `prediction_gqa_full_last4.json` for submission.

In [ ]:
t_lg = save_kd_logits(model, "gqa_full_last4", "last4")        # all 37k train docs, saved to Drive
print(t_lg.shape, "| teacher train accuracy:", (t_lg.argmax(1) == C["train"]["y"]).mean().round(4))

#   rr = build_gqa_rr(); model = cj_from(rr)
#   model.load_state_dict(torch.load(f"{P4_DIR}/cj_gqa_full_last4_best.pt"), strict=False)

(37387, 2) | teacher train accuracy: 0.9579


**Code cell 48. What this cell does.** Saves the teacher's predictions on the training documents so a student can be distilled from them later.

**Line by line.** `save_kd_logits(model, "gqa_full_last4", "last4")` runs the fine-tuned teacher over all training documents and stores the logits as a `.npy` file on Drive (about 37,000 x 2 numbers); the cell prints the array shape and the teacher's accuracy on the training set. The commented lines show how to rebuild and reload the teacher after a runtime restart.

**Why cache them.** Distilling from stored logits means the large teacher does not need to be loaded again during student training, which saves memory and time.

In [ ]:
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"
STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)
def build_lm(cfg):
    return CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), max_seq_len=2048, tie_weights=True, **cfg)

def build_cj_student():
    lm_ = build_lm(STUDENT_CFG)
    ck = torch.load(STAGE1_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)          # Phase-2 stage-1 student
    return cj_from(RRModel(lm_, STUDENT_CFG["d_model"], 13))

DEBUG   = False        # True first (~1-2 min), then False
MODE    = "last4"       # same view as the teacher
RUN_NOKD = False        # True also trains a no-teacher student baseline (costs another ~half hour)

kd_lg = np.load(f"{P4_DIR}/kd_logits_gqa_full_last4.npy")
assert kd_lg.shape == (len(C["train"]["ids"]), 2)

CJ = globals().get("CJ", {})
stu = build_cj_student()
print("student backbone params:", sum(p.numel() for p in stu.lm.parameters()) / 1e6, "M")
NAME = f"student_kd_{MODE}"
CJ[NAME] = train_cj(stu, NAME, MODE, epochs=2, lr=2e-4, head_lr=1e-3, kd=kd_lg, alpha=0.5, T=2.0, debug=DEBUG)
if not DEBUG:
    write_pred(stu, NAME, MODE)
del stu; torch.cuda.empty_cache()

if RUN_NOKD and not DEBUG:
    stu = build_cj_student(); NAME = f"student_nokd_{MODE}"
    CJ[NAME] = train_cj(stu, NAME, MODE, epochs=2, lr=2e-4, head_lr=1e-3, debug=False)
    del stu; torch.cuda.empty_cache()

student backbone params: 43.213248 M
[student_kd_last4] mode=last4 train_docs=37387 steps=18694 trainable=43.22M / 43.2M
  ~34 min training total (excl. dev evals)
  step 100/18694 loss 0.9877
  step 200/18694 loss 1.0478
  step 300/18694 loss 1.7748
  step 400/18694 loss 0.8892
  step 500/18694 loss 1.2192
  step 600/18694 loss 1.6822
  step 700/18694 loss 1.8313
  step 800/18694 loss 1.4654
  step 900/18694 loss 0.9772
  step 1000/18694 loss 0.7447
  step 1100/18694 loss 1.9169
  step 1200/18694 loss 1.3897
  step 1300/18694 loss 1.6111
  step 1400/18694 loss 1.3510
  step 1500/18694 loss 3.7161
  step 1600/18694 loss 2.2114
  step 1700/18694 loss 1.7126
  step 1800/18694 loss 1.2037
  step 1900/18694 loss 1.6970
  step 2000/18694 loss 0.5380
  step 2100/18694 loss 1.2732
  step 2200/18694 loss 1.2621
  step 2300/18694 loss 2.4057
  step 2400/18694 loss 1.2806
  step 2500/18694 loss 1.3678
  step 2600/18694 loss 1.6593
  step 2700/18694 loss 2.1572
  step 2800/18694 loss 1.9237
  ste

**Code cell 49. What this cell does.** Knowledge distillation for Phase 4: trains the small student (about 43M parameters) to imitate the stored teacher logits.

**Line by line**
- `STAGE1_CKPT` and `STUDENT_CFG`: the student language model from Phase 2 (7 layers, width 576). `build_cj_student()` loads those weights into a fresh model and wraps it as a court-judgment classifier with a fresh head.
- Settings: `MODE = "last4"` (the same view as the teacher, so both see the same input), `RUN_NOKD` to also train a baseline without a teacher.
- It loads the saved teacher logits and asserts their shape (one row of 2 logits per training document).
- `train_cj(stu, NAME, MODE, epochs=2, lr=2e-4, head_lr=1e-3, kd=kd_lg, alpha=0.5, T=2.0)`: the loss is half ordinary cross-entropy on the true labels and half `T^2 * KL` between the softened student and teacher distributions (temperature 2). The student's backbone learning rate (2e-4) is higher than the teacher's because the small model is further from convergence.
- Test predictions are written, memory is freed, and optionally a no-teacher student is trained for comparison.

In [ ]:
# If the runtime restarted, reload the teacher first:
#   rr = build_gqa_rr(); model = cj_from(rr)
#   model.load_state_dict(torch.load(f"{P4_DIR}/cj_gqa_full_last4_best.pt"), strict=False)
ydv = C["dev"]["y"]
print(f"{'view':8s} {'F1':>7s} {'P':>7s} {'R':>7s}")
for m in ["first1", "last1", "first4", "ht4", "last4"]:
    lg = cj_logits(model, "dev", m); sc = cj_scores(ydv, lg)
    print(f"{m:8s} {sc['f1']:7.4f} {sc['p']:7.4f} {sc['r']:7.4f}")

view          F1       P       R
first1    0.5560  0.5561  0.5560
last1     0.6888  0.7045  0.6932
first4    0.6713  0.6732  0.6720
ht4       0.7115  0.7165  0.7128
last4     0.7170  0.7218  0.7182


**Code cell 50. What this cell does.** Evaluates the fine-tuned teacher with five different document views on the dev set, without retraining, to see which part of the judgment carries the signal.

**Line by line.** For each view (`first1`, `last1`, `first4`, `ht4`, `last4`) it computes the dev logits with the already-trained model and prints macro-F1, precision and recall. The commented lines show how to reload the teacher after a restart.

**Why.** It answers the analysis question "which part of the document matters?" using one model, so any difference is due to the input view alone.

In [ ]:
import glob, itertools, json, os, numpy as np, torch

# 1) save dev logits of everything trained in this session (the test logits were saved by write_pred)
for k, r in globals().get("CJ", {}).items():
    if len(r["dev_logits"]) == len(C["dev"]["y"]):                  # skip DEBUG runs (200 docs)
        np.save(f"{P4_DIR}/dev_logits_{k}.npy", r["dev_logits"])

# 2) load every model that has both dev and test logits
ydv = C["dev"]["y"]; D, Tt = {}, {}
for f in glob.glob(f"{P4_DIR}/dev_logits_*.npy"):
    k = os.path.basename(f)[len("dev_logits_"):-4]; tf = f"{P4_DIR}/test_logits_{k}.npy"
    if os.path.exists(tf) and len(np.load(f)) == len(ydv):
        D[k] = torch.softmax(torch.tensor(np.load(f)), -1).numpy()
        Tt[k] = torch.softmax(torch.tensor(np.load(tf)), -1).numpy()
print("models:", list(D))

# 3) every subset of size >= 2, equal-weight probability average, scored on dev
def dev_f1(p): return cj_scores(ydv, np.log(p + 1e-9))["f1"]
rows = [((k,), dev_f1(D[k])) for k in D]
for n in range(2, len(D) + 1):
    for sub in itertools.combinations(D, n):
        rows.append((sub, dev_f1(np.mean([D[k] for k in sub], 0))))
for sub, f in sorted(rows, key=lambda x: -x[1]): print(f"{f:.4f}  {' + '.join(sub)}")

# 4) write the best ensemble (only if it has >= 2 members)
best = max([r for r in rows if len(r[0]) >= 2], key=lambda x: x[1], default=None)
if best:
    sub, f = best
    p1 = np.mean([Tt[k] for k in sub], 0)[:, 1]; pred = (p1 > 0.5).astype(int)
    out = [{"id": i, "pred": int(p)} for i, p in zip(C["test"]["ids"], pred)]
    assert len(out) == 1517
    fn = f"{P4_DIR}/prediction_ens_best.json"; json.dump(out, open(fn, "w"))
    print(f"\nwrote {fn} | members: {sub} | dev macro-F1 {f:.4f}")

models: ['gqa_full_last4', 'student_kd_last4']
0.7252  gqa_full_last4 + student_kd_last4
0.7170  gqa_full_last4
0.5495  student_kd_last4

wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_4/prediction_ens_best.json | members: ('gqa_full_last4', 'student_kd_last4') | dev macro-F1 0.7252


**Code cell 51. What this cell does.** Ensembles all Phase-4 models whose predictions were saved, by averaging probabilities, and writes the best combination.

**Line by line**
1. Saves the dev logits of every model trained in this session (skipping debug runs, which only have 200 documents). The test logits were saved earlier by `write_pred`.
2. Finds every model that has both dev and test logits, converts them to probabilities with softmax, and lists the model names.
3. Scores every single model and every subset of two or more models (equal-weight average of probabilities) by dev macro-F1 and prints them sorted from best to worst.
4. Takes the best subset with at least two members, averages its test probabilities, labels a document as class 1 if the probability exceeds 0.5, checks that there are 1,517 predictions, and writes `prediction_ens_best.json`.

**Why.** Averaging models that make different errors usually helps. The caveat is that choosing the best subset on the dev set makes its dev score slightly optimistic.

## Legal Statute Identification

In [ ]:
import json, os, pickle, time, numpy as np
from collections import Counter
from huggingface_hub import snapshot_download

P5_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_5"
os.makedirs(P5_DIR, exist_ok=True)

local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_5_LSI/*", "*/Phase_5_LSI/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_5" in r: return os.path.join(r, name)
tr, dv, te = [json.load(open(find(n))) for n in ("train.json", "dev.json", "test.json")]
print("sizes:", len(tr), len(dv), len(te))
print("train keys:", list(tr[0].keys()), "| test keys:", list(te[0].keys()))
print("example labels:", tr[0]["labels"][:10], "| text start:", tr[0]["text"][:100].replace("\n", " "))

# ---- label statistics ----
ctr, ctd = Counter(l for r in tr for l in r["labels"]), Counter(l for r in dv for l in r["labels"])
allids = sorted(set(ctr) | set(ctd))
print(f"\nstatute ids: train {len(ctr)} | dev {len(ctd)} | union {len(allids)} | min id {min(allids)} max id {max(allids)}")
print("dev ids unseen in train:", len(set(ctd) - set(ctr)), "| train ids absent in dev:", len(set(ctr) - set(ctd)))
fr = np.array(sorted(ctr.values(), reverse=True))
print("train positives per statute: max", fr[0], "median", int(np.median(fr)), "min", fr[-1])
print("statutes with <=5 train positives:", int((fr <= 5).sum()), "| <=20:", int((fr <= 20).sum()), "| >=500:", int((fr >= 500).sum()))
nl = np.array([len(r["labels"]) for r in tr])
print("labels per train doc: mean %.2f median %d max %d | docs with 1 label: %.1f%%" % (nl.mean(), np.median(nl), nl.max(), 100 * (nl == 1).mean()))
print("duplicate ids inside a doc:", sum(len(r["labels"]) != len(set(r["labels"])) for r in tr))
print("top 10 statutes (id, train count):", ctr.most_common(10))

# ---- tokenise once: keep first/last 4096 tokens (same scheme as Phase 4) ----
def tok_split(texts, N=4096):
    out = []
    for i in range(0, len(texts), 500):
        b = texts[i:i + 500]
        eh = tokenizer.encode_batch([t[:40000] for t in b])
        long_ = [j for j, t in enumerate(b) if len(t) > 80000]
        et = dict(zip(long_, tokenizer.encode_batch([b[j][-40000:] for j in long_]))) if long_ else {}
        for j in range(len(b)):
            hi = eh[j].ids; ti, total = (et[j].ids, 10**6) if j in et else (hi, len(hi))
            out.append((np.array(hi[:N], dtype=np.uint16), np.array(ti[-N:], dtype=np.uint16), total))
    return out

CACHE = f"{P5_DIR}/lsi_tok_cache.pkl"
if os.path.exists(CACHE):
    C = pickle.load(open(CACHE, "rb")); print("loaded cache")
else:
    t0 = time.time(); C = {}
    for name, d in [("train", tr), ("dev", dv), ("test", te)]:
        tk = tok_split([r["text"] for r in d])
        C[name] = dict(ids=[r["id"] for r in d], labels=[r.get("labels") for r in d],
                       head=[a for a, _, _ in tk], tail=[b for _, b, _ in tk], n=np.array([n for _, _, n in tk]))
        print(name, f"{(time.time() - t0) / 60:.1f} min")
    pickle.dump(C, open(CACHE, "wb"))
n = C["train"]["n"]
print("tokens per train doc: median", int(np.median(n)), "| p90", int(np.percentile(n, 90)),
      "| <=4096: %.1f%%" % (100 * (n <= 4096).mean()), "| <=8192: %.1f%%" % (100 * (n <= 8192).mean()))
print("✅ Phase-5 cache ready")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

sizes: 42750 10181 13019
train keys: ['id', 'text', 'labels'] | test keys: ['id', 'text']
example labels: [69, 9, 3] | text start: (a), <SECTION> of the <ACT>. The gist of theprosecution case relevant <ENTITY> the purpose of this p

statute ids: train 100 | dev 100 | union 100 | min id 0 max id 99
dev ids unseen in train: 0 | train ids absent in dev: 0
train positives per statute: max 11731 median 679 min 272
statutes with <=5 train positives: 0 | <=20: 0 | >=500: 70
labels per train doc: mean 3.71 median 3 max 31 | docs with 1 label: 21.4%
duplicate ids inside a doc: 3
top 10 statutes (id, train count): [(5, 11731), (34, 10380), (97, 7881), (43, 7728), (39, 7121), (12, 4324), (13, 4136), (10, 4134), (77, 4046), (94, 3835)]
loaded cache
tokens per train doc: median 663 | p90 4286 | <=4096: 89.2% | <=8192: 97.3%
✅ Phase-5 cache ready


**Code cell 52. What this cell does.** Downloads the Phase-5 data (legal statute identification, LSI), prints statistics about the labels, and caches tokenized head and tail views of every document.

**Task.** Each judgment cites several statutes out of 100 (labels 0 to 99); the model must predict the full set (multi-label). The metric is macro-F1 over the 100 statutes.

**Line by line**
- Downloads `Phase_5_LSI/*` with `snapshot_download`; `find` locates `train.json`, `dev.json`, `test.json`; the sizes and the field names are printed. The test file has only `id` and `text`.
- **Label statistics.** `ctr` and `ctd` count how often each statute id occurs in train and dev. The cell prints the number of distinct ids (100 in both), the lowest and highest id, whether any dev id is unseen in train (none), and the distribution of positives per statute (maximum, median, minimum), the number of statutes with at most 5 or 20 positives (none) or at least 500. It also prints labels per document (mean, median, maximum, share with a single label) and the number of documents that list the same id twice. These numbers justify the weighted loss later.
- **Tokenization.** `tok_split` does the same as in Phase 4: encodes the first 40,000 characters, and the last 40,000 characters of documents longer than 80,000 characters, and keeps the first and last 4,096 token ids as `uint16` plus a length (a marker of 10^6 for very long documents).
- The cache `lsi_tok_cache.pkl` is loaded if present, otherwise built for train, dev and test (with timings) and saved. For each split it stores ids, labels (None for test), head tokens, tail tokens and lengths.
- It finally prints the token length distribution of the training documents (median, 90th percentile, share within 4,096 and 8,192 tokens).

In [ ]:
import json, time, math, os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
assert "C" in globals() and "P5_DIR" in globals() and "RRModel" in globals(), "run the cache cell and the Phase-3 toolkit first"

N_LAB, CH = 100, 1024
MODES = {"first1": 1, "last1": 1, "first4": 4, "last4": 4, "ht4": 4}
GRID = np.arange(0.05, 0.80, 0.025)

def multi_hot(lists):
    Y = np.zeros((len(lists), N_LAB), np.float32)
    for i, l in enumerate(lists): Y[i, list(set(l))] = 1          # also removes duplicate ids
    return Y
YTR, YDV = multi_hot(C["train"]["labels"]), multi_hot(C["dev"]["labels"])

# ---------- document -> chunks (views over the cached head/tail tokens) ----------
def get_chunks(split, i, mode):
    d = C[split]; n = int(d["n"][i]); head, tail = d["head"][i], d["tail"][i]
    if   mode == "first1": x = head[:CH]
    elif mode == "last1":  x = tail[-CH:]
    elif mode == "first4": x = head[:4*CH]
    elif mode == "last4":  x = tail[-4*CH:]
    elif mode == "ht4":    x = head if n <= 4*CH else np.concatenate([head[:2*CH], tail[-2*CH:]])
    L = len(x)
    if mode.startswith("last"):
        ends = list(range(L, 0, -CH))[::-1]; return [x[max(0, e-CH):e] for e in ends]
    return [x[s:s+CH] for s in range(0, L, CH)]

_CNT = {}
def chunk_counts(split, mode):
    if (split, mode) not in _CNT:
        _CNT[(split, mode)] = np.array([max(1, len(get_chunks(split, i, mode))) for i in range(len(C[split]["ids"]))])
    return _CNT[(split, mode)]

def make_batches(split, idxs, mode, max_chunks=16, rng=None, mega=64):
    """Batches of <= max_chunks chunks; documents of similar length are grouped to avoid padding waste."""
    cnt = chunk_counts(split, mode); n = np.minimum(C[split]["n"], 4*CH); idxs = np.asarray(idxs)
    if rng is not None:
        idxs = idxs[rng.permutation(len(idxs))]
        groups = [idxs[s:s+max_chunks*mega] for s in range(0, len(idxs), max_chunks*mega)]
        groups = [g[np.argsort(n[g], kind="stable")] for g in groups]
    else:
        groups = [idxs[np.argsort(n[idxs], kind="stable")]]
    out = []
    for g in groups:
        cur, c = [], 0
        for i in g:
            if cur and c + cnt[i] > max_chunks: out.append(np.array(cur)); cur, c = [], 0
            cur.append(i); c += cnt[i]
        if cur: out.append(np.array(cur))
    if rng is not None: out = [out[j] for j in rng.permutation(len(out))]
    return out

def collate_ls(split, idxs, mode):
    seqs, owner = [], []
    for j, i in enumerate(idxs):
        cs = get_chunks(split, i, mode)
        if not cs or len(cs[0]) == 0: cs = [np.zeros(1, np.uint16)]
        for c in cs: seqs.append(c); owner.append(j)
    T = max(len(s) for s in seqs)
    ids = torch.zeros(len(seqs), T, dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.from_numpy(s.astype(np.int64))
    lens = torch.tensor([len(s) for s in seqs])
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(idxs)

# ---------- model: Phase-1 LM -> chunk features -> per-chunk statute logits -> document logits ----------
class LSIModel(RRModel):
    agg = "mean"                                   # "mean" or "max" over chunks
    def forward(self, ids, lens, owner, n_docs):
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        mean = (H.float() * mask).sum(1) / lens[:, None]
        last = H[torch.arange(B, device=H.device), lens - 1].float()
        lg = self.head(self.drop(torch.cat([last, mean], -1))).float()          # (chunks, N_LAB)
        if self.agg == "max":
            return torch.full((n_docs, lg.size(1)), -1e4, device=lg.device).scatter_reduce(
                0, owner[:, None].expand_as(lg), lg, "amax", include_self=False)
        D = torch.zeros(n_docs, lg.size(1), device=lg.device).index_add_(0, owner, lg)
        return D / torch.bincount(owner, minlength=n_docs).clamp(min=1)[:, None]

def ls_from(rr):
    """Phase-3 RRModel -> LSI model: same LM, fresh 100-way head (bias = log prior odds of each statute)."""
    d = rr.lm.token_embedding.weight.shape[1]
    m = LSIModel(rr.lm, d, N_LAB, 0.1).to(device)
    m.head = nn.Linear(2 * d, N_LAB).to(device)
    nn.init.normal_(m.head.weight, std=0.02)
    p = YTR.mean(0).clip(1e-4, 1 - 1e-4)
    with torch.no_grad(): m.head.bias.copy_(torch.tensor(np.log(p / (1 - p)), dtype=torch.float))
    return m

# ---------- inference, decoding, metrics ----------
@torch.no_grad()
def ls_logits(model, split, mode, idxs=None, max_chunks=32):
    model.eval(); N = len(C[split]["ids"]); idxs = np.arange(N) if idxs is None else np.asarray(idxs)
    out = np.zeros((N, N_LAB), np.float32)
    for b in make_batches(split, idxs, mode, max_chunks, rng=None):
        with torch.autocast("cuda", dtype=torch.bfloat16):
            out[b] = model(*collate_ls(split, b, mode)).float().cpu().numpy()
    return out[idxs]

def sigmoid(z): return 1 / (1 + np.exp(-z))

def decode(P, thr, force_one=True):
    pred = P >= thr
    if force_one:
        none = ~pred.any(1); pred[none, P[none].argmax(1)] = True      # every document has at least one statute
    return pred

def ls_scores(Y, pred):
    Yb = Y.astype(bool); tp = (pred & Yb).sum(0); fp = (pred & ~Yb).sum(0); fn = (~pred & Yb).sum(0)
    f = np.where(2*tp + fp + fn > 0, 2*tp / np.maximum(2*tp + fp + fn, 1), 0.0)
    p = np.where(tp + fp > 0, tp / np.maximum(tp + fp, 1), 0.0); r = np.where(tp + fn > 0, tp / np.maximum(tp + fn, 1), 0.0)
    return dict(macro=float(f.mean()), p=float(p.mean()), r=float(r.mean()),
                micro=float(2*tp.sum() / max(2*tp.sum() + fp.sum() + fn.sum(), 1)), subset=float((pred == Yb).all(1).mean()), per_label=f)

def best_global(Y, P):
    best = (0.5, -1.0)
    for t in GRID:
        f = ls_scores(Y, decode(P, t))["macro"]
        if f > best[1]: best = (float(t), f)
    return best

def best_per_label(Y, P):
    Yb = Y.astype(bool); thr = np.full(N_LAB, 0.5)
    for j in range(N_LAB):
        pj = P[:, j][:, None] >= GRID[None, :]; yj = Yb[:, j][:, None]
        tp = (pj & yj).sum(0); fp = (pj & ~yj).sum(0); fn = (~pj & yj).sum(0)
        thr[j] = GRID[(2*tp / np.maximum(2*tp + fp + fn, 1)).argmax()]
    return thr

def honest_perlabel(Y, P):
    """Per-label thresholds tuned on one half of dev and scored on the other half (and vice versa)."""
    idx = np.arange(len(Y)); A, B = idx[::2], idx[1::2]; out = []
    for a, b in [(A, B), (B, A)]:
        out.append(ls_scores(Y[b], decode(P[b], best_per_label(Y[a], P[a])))["macro"])
    return float(np.mean(out))

# ---------- training (BCE / weighted BCE, optional KD from cached teacher logits) ----------
def train_ls(model, name, mode, epochs=2, lr=5e-5, head_lr=1e-3, wd=0.01, max_chunks=16, n_train=None,
             loss_type="wbce", kd=None, alpha=0.5, T=2.0, evals_per_epoch=2, patience=3, debug=False, seed=0):
    torch.manual_seed(seed); rng = np.random.RandomState(seed); ntr = len(YTR)
    dev_idx = np.arange(300) if debug else np.arange(len(YDV)); Ydev = YDV[dev_idx]
    tr_idx = rng.permutation(ntr)[: (160 if debug else (n_train or ntr))]
    if debug: epochs, evals_per_epoch = 1, 1
    pw = None
    if loss_type == "wbce":
        pos = YTR.sum(0); pw = torch.tensor(np.clip(np.sqrt((ntr - pos) / pos), 1, 10), dtype=torch.float, device=device)
    ytr_t = torch.tensor(YTR); kd_t = torch.tensor(kd, dtype=torch.float) if kd is not None else None
    head_p = list(model.head.parameters()); hid = {id(p) for p in head_p}
    body_p = [p for p in model.parameters() if p.requires_grad and id(p) not in hid]
    opt = torch.optim.AdamW([{"params": body_p, "lr": lr}, {"params": head_p, "lr": head_lr}], weight_decay=wd)
    spe = len(make_batches("train", tr_idx, mode, max_chunks, np.random.RandomState(seed))); total = spe * epochs
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / (total - warm)))
    every = max(1, spe // evals_per_epoch)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad); n_all = sum(p.numel() for p in model.parameters())
    print(f"[{name}] mode={mode} loss={loss_type} train_docs={len(tr_idx)} steps={total} trainable={n_tr/1e6:.2f}M / {n_all/1e6:.1f}M")

    path = f"{P5_DIR}/ls_{name}_best.pt"; best = (-1.0, None); bad = 0; step = 0; t0 = time.time(); done = False
    torch.cuda.reset_peak_memory_stats()
    for ep in range(epochs):
        model.train()
        for b in make_batches("train", tr_idx, mode, max_chunks, rng):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                lg = model(*collate_ls("train", b, mode)).float()
            y = ytr_t[b].to(device)
            loss = F.binary_cross_entropy_with_logits(lg, y, pos_weight=pw)
            if kd_t is not None:
                t_lg = kd_t[b].to(device)
                kdl = F.binary_cross_entropy_with_logits(lg / T, torch.sigmoid(t_lg / T)) * T * T
                loss = (1 - alpha) * loss + alpha * kdl
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step(); step += 1
            if step == 20: print(f"  ~{(time.time()-t0)/20*total/60:.0f} min training total (excl. dev evals)")
            if step % 200 == 0: print(f"  step {step}/{total} loss {loss.item():.4f}")
            if step % every == 0 or step == total:
                P = sigmoid(ls_logits(model, "dev", mode, dev_idx)); thr, f = best_global(Ydev, P); model.train()
                s5 = ls_scores(Ydev, decode(P, 0.5))
                print(f"  eval@{step}: macro-F1 @0.5 {s5['macro']:.4f} | @best global thr {thr:.3f}: {f:.4f}  ({(time.time()-t0)/60:.1f} min)")
                if f > best[0]:
                    best = (f, step); bad = 0
                    torch.save({k: v.detach().cpu() for k, v in model.named_parameters() if v.requires_grad}, path)
                else:
                    bad += 1
                    if bad >= patience: done = True; break
        if done: break
    if os.path.exists(path): model.load_state_dict(torch.load(path), strict=False)
    P = sigmoid(ls_logits(model, "dev", mode, dev_idx)); thr, f = best_global(Ydev, P)
    sg = ls_scores(Ydev, decode(P, thr)); tl = best_per_label(Ydev, P); sl = ls_scores(Ydev, decode(P, tl))
    hon = honest_perlabel(Ydev, P)
    res = dict(name=name, mode=mode, thr_global=thr, thr_label=tl, f1_at_05=ls_scores(Ydev, decode(P, 0.5))["macro"],
               macro_global=sg["macro"], p=sg["p"], r=sg["r"], micro=sg["micro"], subset=sg["subset"],
               macro_perlabel_optimistic=sl["macro"], macro_perlabel_honest=hon, per_label=sg["per_label"],
               minutes=(time.time()-t0)/60, peak_gb=torch.cuda.max_memory_allocated()/1e9, trainable=n_tr, total=n_all, dev_probs=P)
    print(f"✅ [{name}] dev macro-F1: @0.5 {res['f1_at_05']:.4f} | global thr {thr:.3f} → {sg['macro']:.4f} "
          f"(P {sg['p']:.4f} R {sg['r']:.4f}, micro {sg['micro']:.4f}, subset acc {sg['subset']:.4f})\n"
          f"   per-label thresholds: {sl['macro']:.4f} on the tuning data, {hon:.4f} on held-out dev halves "
          f"| {res['minutes']:.1f} min, peak {res['peak_gb']:.1f} GB")
    return res

# ---------- outputs ----------
def write_pred_ls(model, name, mode, res):
    L = ls_logits(model, "test", mode); P = sigmoid(L)
    np.save(f"{P5_DIR}/test_probs_{name}.npy", P); np.save(f"{P5_DIR}/dev_probs_{name}.npy", res["dev_probs"])
    for tag, thr in [("global", res["thr_global"]), ("perlabel", res["thr_label"])]:
        pred = decode(P, thr); ids = C["test"]["ids"]
        out = [{"id": i, "pred": [int(k) for k in np.flatnonzero(pr)]} for i, pr in zip(ids, pred)]
        assert len(out) == len(ids) == 13019 and all(0 < len(o["pred"]) == len(set(o["pred"])) and max(o["pred"]) < N_LAB for o in out)
        f = f"{P5_DIR}/prediction_{name}_{tag}.json"; json.dump(out, open(f, "w"))
        print(f"wrote {f} | mean statutes/doc {pred.sum(1).mean():.2f}")

def save_kd_ls(model, name, mode):
    L = ls_logits(model, "train", mode); np.save(f"{P5_DIR}/kd_logits_{name}.npy", L); return L

print("✅ Phase-5 toolkit ready")

✅ Phase-5 toolkit ready


**Code cell 53. What this cell does.** The Phase-5 toolkit: a multi-label document classifier, its decoding rules, metrics, trainer and output writers.

**Setup.** `N_LAB = 100` statutes, `CH = 1024`-token chunks, the same document views as Phase 4. `GRID` is the list of thresholds tried, 0.05 to 0.775 in steps of 0.025. `multi_hot` converts label lists to 0/1 vectors of length 100 (duplicates collapse). `YTR`, `YDV` are the train and dev label matrices.

**Chunking and batching.** `get_chunks`, `collate_ls` and `chunk_counts` work as in Phase 4. `make_batches` builds batches of at most `max_chunks` chunks. For training it shuffles, then sorts small groups of documents by length so that documents in a batch have similar sizes (less padding waste), and shuffles the batches.

**Model `LSIModel`** (extends `RRModel`). Each chunk gets features `[last token ; mean]` (2d), the head turns them into 100 logits per chunk, and the chunk logits are combined into a document logit by the mean (default) or the maximum over chunks (`agg`). `ls_from(rr)` creates the model from a language model with a fresh head whose bias is initialised to the log prior odds `log(p / (1 - p))` of each statute, so at the start the model already outputs the base rates instead of 0.5 for everything.

**Inference and decoding.** `ls_logits` computes document logits for a split; `sigmoid` converts to probabilities; `decode(P, thr)` labels a statute positive when its probability reaches the threshold (a single number, or one per statute) and forces at least one label per document (the most probable one), since every document has at least one statute.

**Metrics `ls_scores`.** For each statute, true positives, false positives and false negatives give precision, recall and F1; macro averages the 100 F1 values; micro pools the counts; `subset` is the share of documents whose whole label set is exactly right.

**Thresholds.** `best_global` tries each value on the grid and keeps the best macro-F1. `best_per_label` picks the best threshold for each statute separately (vectorised over the grid). Because 100 thresholds tuned on the dev set overfit it, `honest_perlabel` tunes on half of the dev documents and scores on the other half (and the reverse) and averages: a fair estimate.

**Training `train_ls`.** Weighted binary cross-entropy: each statute's positive examples are weighted by `clip(sqrt(negatives / positives), 1, 10)`. With cached teacher logits it adds a distillation term `T^2 * BCE(student/T, sigmoid(teacher/T))`, mixed with `alpha`. Two learning rates (backbone and head), AdamW, 6% warm-up, linear decay, two evaluations per epoch, the best checkpoint by dev macro-F1 at the best global threshold, early stopping. After training it reloads the best parameters and reports macro-F1 at 0.5, at the best global threshold (with precision, recall, micro and subset accuracy), per-label thresholds (optimistic and honest), time, memory and parameter counts.

**Outputs.** `write_pred_ls` saves test probabilities and writes two prediction files, one with the global threshold and one with per-label thresholds, checking that there are 13,019 predictions, each with at least one distinct valid statute. `save_kd_ls` saves the training-set logits for distillation.

In [ ]:
TEACHER_CKPT = f"{DRIVE_PROJECT_DIR}/gqa_checkpoints/gqa_best.pt"
NC = 13
def build_gqa_rr(dropout=0.1):
    lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                          n_kv_head=4, max_seq_len=2048, tie_weights=True)
    ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)
    return RRModel(lm_, 768, NC, dropout).to(device)

DEBUG   = False         # True first (~1-2 min); then False
MODE    = "ht4"         # first 2,048 + last 2,048 tokens (= the whole document for 89% of cases)
N_TRAIN = None
LS = globals().get("LS", {})

model = ls_from(build_gqa_rr())
NAME = f"gqa_full_{MODE}"
LS[NAME] = train_ls(model, NAME, MODE, epochs=2, lr=5e-5, head_lr=1e-3, n_train=N_TRAIN, loss_type="wbce", debug=DEBUG)
if not DEBUG: write_pred_ls(model, NAME, MODE, LS[NAME])

[gqa_full_ht4] mode=ht4 loss=wbce train_docs=42750 steps=9878 trainable=87.66M / 87.7M
  ~28 min training total (excl. dev evals)
  step 200/9878 loss 0.5105
  step 400/9878 loss 0.4401
  step 600/9878 loss 0.4924
  step 800/9878 loss 0.4230
  step 1000/9878 loss 0.1202
  step 1200/9878 loss 0.4206
  step 1400/9878 loss 0.3639
  step 1600/9878 loss 0.2833
  step 1800/9878 loss 0.4137
  step 2000/9878 loss 0.2391
  step 2200/9878 loss 0.3125
  step 2400/9878 loss 0.3162
  eval@2469: macro-F1 @0.5 0.3000 | @best global thr 0.475: 0.3003  (8.6 min)
  step 2600/9878 loss 0.5561
  step 2800/9878 loss 0.2880
  step 3000/9878 loss 0.1332
  step 3200/9878 loss 0.4170
  step 3400/9878 loss 0.9210
  step 3600/9878 loss 0.3813
  step 3800/9878 loss 0.2511
  step 4000/9878 loss 0.4308
  step 4200/9878 loss 0.2812
  step 4400/9878 loss 0.2660
  step 4600/9878 loss 0.2364
  step 4800/9878 loss 0.1179
  eval@4938: macro-F1 @0.5 0.3441 | @best global thr 0.525: 0.3456  (17.2 min)
  step 5000/9878 loss

**Code cell 54. What this cell does.** Full fine-tuning of the GQA model for statute identification (the Phase-5 teacher), 2 epochs.

**Line by line.** `TEACHER_CKPT`, `NC` and `build_gqa_rr` are as in Phase 3/4 (`NC = 13` only creates the throwaway old head). `MODE = "ht4"` uses the first 2,048 and last 2,048 tokens, which is the whole document for 89% of the cases. `ls_from(build_gqa_rr())` builds the model with a 100-way head whose bias is set to the log prior odds. `train_ls(..., epochs=2, lr=5e-5, head_lr=1e-3, loss_type="wbce")` trains all 87.5M parameters with the weighted binary cross-entropy. The result is stored in the dictionary `LS`; unless debugging, `write_pred_ls` writes the two prediction files (global threshold and per-label thresholds).

In [ ]:
DEBUG   = False
MODE    = "ht4"
EPOCHS  = 5
LS = globals().get("LS", {})

model = ls_from(build_gqa_rr())
NAME = f"gqa_full{EPOCHS}_{MODE}"
LS[NAME] = train_ls(model, NAME, MODE, epochs=EPOCHS, lr=5e-5, head_lr=1e-3, patience=4,
                    loss_type="wbce", debug=DEBUG)
write_pred_ls(model, NAME, MODE, LS[NAME])

[gqa_full5_ht4] mode=ht4 loss=wbce train_docs=42750 steps=24695 trainable=87.66M / 87.7M
  ~71 min training total (excl. dev evals)
  step 200/24695 loss 0.5289
  step 400/24695 loss 0.3451
  step 600/24695 loss 0.4728
  step 800/24695 loss 0.4326
  step 1000/24695 loss 0.1522
  step 1200/24695 loss 0.4370
  step 1400/24695 loss 0.3671
  step 1600/24695 loss 0.2867
  step 1800/24695 loss 0.4569
  step 2000/24695 loss 0.2579
  step 2200/24695 loss 0.3258
  step 2400/24695 loss 0.3283
  eval@2469: macro-F1 @0.5 0.2905 | @best global thr 0.500: 0.2906  (8.6 min)
  step 2600/24695 loss 0.5457
  step 2800/24695 loss 0.3511
  step 3000/24695 loss 0.1416
  step 3200/24695 loss 0.4226
  step 3400/24695 loss 0.9500
  step 3600/24695 loss 0.4299
  step 3800/24695 loss 0.2801
  step 4000/24695 loss 0.4469
  step 4200/24695 loss 0.2864
  step 4400/24695 loss 0.2897
  step 4600/24695 loss 0.2309
  step 4800/24695 loss 0.1213
  eval@4938: macro-F1 @0.5 0.3289 | @best global thr 0.500: 0.3297  (17.3 

**Code cell 55. What this cell does.** A longer variant of the previous run: the same setup trained for 5 epochs with patience 4, to test whether more training helps.

**Line by line.** Same as the previous cell except `EPOCHS = 5`, `patience = 4` and the run name `gqa_full5_ht4`. It writes the prediction files at the end. The report notes that this longer run overfitted and scored lower than the 2-epoch run, so the 2-epoch model is the one used as teacher.

In [ ]:
t_lg = save_kd_ls(model, "gqa_full5_ht4", "ht4"); print(t_lg.shape)

(42750, 100)


**Code cell 56. What this cell does.** Saves the logits of the 5-epoch model on the training documents (shape: number of training documents x 100).

**Note.** The distilled student was finally trained from the logits of the 2-epoch model, which the later cells recreate (see the cell that reloads the teacher and saves `kd_logits_gqa_full_ht4.npy`).

In [ ]:
   import pickle
   P5_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_5"
   C = pickle.load(open(f"{P5_DIR}/lsi_tok_cache.pkl", "rb"))
   print({k: len(v["ids"]) for k, v in C.items()}, "✅ Phase-5 data ready")

{'train': 42750, 'dev': 10181, 'test': 13019} ✅ Phase-5 data ready


**Code cell 57. What this cell does.** Recovery after a runtime restart: reloads the cached Phase-5 tokens from Drive.

**Line by line.** Sets `P5_DIR`, loads `lsi_tok_cache.pkl` into `C` and prints the size of each split. (The cell is indented because it was pasted from a message; it still runs.) Run this, then the toolkit cell, before anything else in Phase 5 after a restart.

In [ ]:
   TEACHER_CKPT = f"{DRIVE_PROJECT_DIR}/gqa_checkpoints/gqa_best.pt"
   NC = 13
   def build_gqa_rr(dropout=0.1):
       lm_ = CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), d_model=768, n_layer=10, n_head=12,
                             n_kv_head=4, max_seq_len=2048, tie_weights=True)
       ck = torch.load(TEACHER_CKPT, map_location="cpu", weights_only=False)
       lm_.load_state_dict(ck["model"] if "model" in ck else ck)
       return RRModel(lm_, 768, NC, dropout).to(device)
   LS = {}

**Code cell 58. What this cell does.** Recovery after a restart, part 2: redefines the helper that builds the pretrained GQA model with a throwaway head, and an empty results dictionary `LS`. The code is identical to the definition in the full fine-tuning cell.

In [ ]:
model = ls_from(build_gqa_rr())
sd = torch.load(f"{P5_DIR}/ls_gqa_full_ht4_best.pt")
print(model.load_state_dict(sd, strict=False))
P = sigmoid(ls_logits(model, "dev", "ht4")); thr, f = best_global(YDV, P)
print(f"reloaded teacher: dev macro-F1 {f:.4f} at thr {thr:.3f}  (should be about 0.3886)")
assert abs(f - 0.3886) < 0.01, "wrong weights loaded"
t_lg = save_kd_ls(model, "gqa_full_ht4", "ht4"); print(t_lg.shape)

_IncompatibleKeys(missing_keys=['lm.lm_head.weight'], unexpected_keys=[])
reloaded teacher: dev macro-F1 0.3886 at thr 0.525  (should be about 0.3886)
(42750, 100)


**Code cell 59. What this cell does.** Reloads the trained Phase-5 teacher and saves its training-set logits for distillation.

**Line by line**
- Builds a fresh model with `ls_from(build_gqa_rr())` and loads the saved trainable parameters from `ls_gqa_full_ht4_best.pt` (`strict=False`, because the file holds only the parameters that were trained).
- Recomputes the dev probabilities and the best global threshold and prints the macro-F1. The assertion requires it to be within 0.01 of 0.3886 (the teacher's earlier score), proving the right weights were loaded.
- `save_kd_ls` runs the teacher on all training documents and stores the logits as `kd_logits_gqa_full_ht4.npy`; the printed shape should be (training documents, 100).

In [ ]:
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"
STUDENT_CFG = dict(d_model=576, n_layer=7, n_head=12, n_kv_head=4)
def build_lm(cfg):
    return CustomDecoderLM(vocab_size=tokenizer.get_vocab_size(), max_seq_len=2048, tie_weights=True, **cfg)
def build_ls_student():
    lm_ = build_lm(STUDENT_CFG)
    ck = torch.load(STAGE1_CKPT, map_location="cpu", weights_only=False)
    lm_.load_state_dict(ck["model"] if "model" in ck else ck)
    return ls_from(RRModel(lm_, STUDENT_CFG["d_model"], 13))

DEBUG = False           # True first; then False
kd_lg = np.load(f"{P5_DIR}/kd_logits_gqa_full_ht4.npy"); assert kd_lg.shape == (len(YTR), N_LAB)
stu = build_ls_student()
print("student backbone params:", sum(p.numel() for p in stu.lm.parameters()) / 1e6, "M")
NAME = "student_kd_ht4"
LS[NAME] = train_ls(stu, NAME, "ht4", epochs=2, lr=2e-4, head_lr=1e-3, kd=kd_lg, alpha=0.5, T=2.0,
                    loss_type="wbce", debug=DEBUG)
if not DEBUG: write_pred_ls(stu, NAME, "ht4", LS[NAME])

student backbone params: 43.213248 M
[student_kd_ht4] mode=ht4 loss=wbce train_docs=42750 steps=9878 trainable=43.33M / 43.3M
  ~19 min training total (excl. dev evals)
  step 200/9878 loss 0.8536
  step 400/9878 loss 0.7664
  step 600/9878 loss 0.9707
  step 800/9878 loss 1.0433
  step 1000/9878 loss 0.6607
  step 1200/9878 loss 1.1362
  step 1400/9878 loss 0.9073
  step 1600/9878 loss 0.8891
  step 1800/9878 loss 1.1499
  step 2000/9878 loss 0.7351
  step 2200/9878 loss 0.8591
  step 2400/9878 loss 1.0536
  eval@2469: macro-F1 @0.5 0.3116 | @best global thr 0.525: 0.3123  (5.8 min)
  step 2600/9878 loss 1.1958
  step 2800/9878 loss 0.8279
  step 3000/9878 loss 0.6392
  step 3200/9878 loss 1.0225
  step 3400/9878 loss 1.2714
  step 3600/9878 loss 0.8376
  step 3800/9878 loss 0.8281
  step 4000/9878 loss 0.8855
  step 4200/9878 loss 0.9976
  step 4400/9878 loss 0.8855
  step 4600/9878 loss 0.8375
  step 4800/9878 loss 0.6001
  eval@4938: macro-F1 @0.5 0.3495 | @best global thr 0.500: 0

**Code cell 60. What this cell does.** Knowledge distillation for statute identification: trains the 43M student from the teacher's stored logits.

**Line by line**
- `STAGE1_CKPT` / `STUDENT_CFG`: the Phase-2 student language model (7 layers, width 576). `build_ls_student()` loads it and adds a 100-way head with prior-odds bias.
- Loads the saved teacher logits and asserts their shape equals the training label matrix.
- `train_ls(stu, "student_kd_ht4", "ht4", epochs=2, lr=2e-4, head_lr=1e-3, kd=kd_lg, alpha=0.5, T=2.0, loss_type="wbce")`: the loss is half the weighted BCE on the true labels and half the distillation term, `T^2 * BCE(student/T, sigmoid(teacher/T))`, where the teacher's softened probabilities act as soft labels for every statute.
- Writes the prediction files unless debugging.

## Abstractive Summarization

In [ ]:
import json, os, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_6" in r: return os.path.join(r, name)
def rows(d):
    if isinstance(d, list): return d
    if all(isinstance(v, dict) for v in d.values()): return [dict(id=k, **v) for k, v in d.items()]
    return [dict(zip(d.keys(), v)) for v in zip(*d.values())]
tr6, te6 = rows(json.load(open(find("train.json")))), rows(json.load(open(find("test.json"))))
print("train:", len(tr6), "| test:", len(te6), "| train keys:", list(tr6[0].keys()), "| test keys:", list(te6[0].keys()))
dk = "document" if "document" in tr6[0] else "text"
print("\nDOC start :", tr6[0][dk][:300].replace("\n", " "))
print("SUMMARY   :", tr6[0]["summary"][:300].replace("\n", " "))
print("duplicate ids:", len(tr6) - len({r["id"] for r in tr6}), "| empty summaries:", sum(not r["summary"].strip() for r in tr6))

def toklen(texts):
    return np.array([len(e.ids) for e in tokenizer.encode_batch([t[:300000] for t in texts])])
rng = np.random.RandomState(0); samp = [tr6[i] for i in rng.permutation(len(tr6))[:500]]
dl, sl = toklen([r[dk] for r in samp]), toklen([r["summary"] for r in samp])
tl = toklen([r[dk if dk in r else "document"] for r in [te6[i] for i in rng.permutation(len(te6))[:500]]])
q = lambda a: {p: int(np.percentile(a, p)) for p in (10, 50, 90, 99)} | {"max": int(a.max())}
print("\ndocument tokens (500 train docs):", q(dl))
print("summary  tokens (500 train docs):", q(sl))
print("test document tokens (500 docs) :", q(tl))
print("compression ratio (median doc/summary): %.1fx" % np.median(dl / np.maximum(sl, 1)))
for L in (1024, 2048, 3072, 4096): print(f"docs <= {L} tokens: {100 * (dl <= L).mean():.1f}%")
for L in (128, 256, 320, 512): print(f"summaries <= {L} tokens: {100 * (sl <= L).mean():.1f}%")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

train: 7030 | test: 100 | train keys: ['id', 'document', 'summary'] | test keys: ['id', 'document']

DOC start : IVIL Appeal No.768 (NT) of 1977 etc. From the judgment Order dated 9.10.1975 of the Madhya Pradesh High Court in M.C.C. No.144 of 1966. V.A. Bobde, B.R.Aggarwala and U.A.Rana for the Appellant. S.V.Deshpande and S.K.Agnihotri for the Respondents. The Judgment of the Court was delivered by RANGANATH 
SUMMARY   : The appellant, a manufacturer of cement, entered into an agreement with the Cement Manufacturing Company of India Limited, for sale of cement. Under the agreement, the appellant was to sell its cement only through the marketing company, and certain sums would be paid for the cement supplied by the m
duplicate ids: 0 | empty summaries: 2

document tokens (500 train docs): {10: 1673, 50: 3699, 90: 9867, 99: 34721, 'max': 51763}
summary  tokens (500 train docs): {10: 352, 50: 773, 90: 1833, 99: 5569, 'max': 9803}
test document tokens (500 docs) : {10: 1766, 50: 3754, 90:

**Code cell 61. What this cell does.** Downloads the Phase-6 data (abstractive summarisation) and measures document and summary lengths.

**Task.** Given a long judgment, write a summary. Scoring uses BERTScore and ROUGE against reference summaries.

**Line by line**
- Downloads `Phase_6_Summ/*`. `find` locates a file; `rows` converts several possible JSON layouts into a list of records. `tr6` and `te6` are the train and test records; their sizes and field names are printed.
- `dk` picks the document field name (`document` or `text`). The cell prints the beginning of the first document and its summary, the number of duplicate ids, and the number of empty summaries (empty summaries are removed from training later).
- `toklen(texts)` counts tokens with our tokenizer on the first 300,000 characters.
- It draws a random sample of 500 training records (fixed seed) and 500 test documents, and prints percentiles (10, 50, 90, 99, maximum) of document tokens, summary tokens and test document tokens, the median compression ratio, and the share of documents within 1,024, 2,048, 3,072 and 4,096 tokens and of summaries within 128, 256, 320 and 512 tokens.

**Why.** These statistics decide the design: how much of the document the model should read, and how long the generated summary may be.

In [ ]:
import os, json, pickle, numpy as np
from huggingface_hub import snapshot_download
P6_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_6"; os.makedirs(P6_DIR, exist_ok=True)
CACHE6 = f"{P6_DIR}/sum_tok_cache.pkl"
EOS6 = tokenizer.token_to_id("<|endoftext|>")

def doc_view(rec, n_head, n_tail=0):
    """Token view of a document: first n_head tokens, plus last n_tail tokens if the doc is longer."""
    n, h, t = rec["n"], rec["head"], rec["tail"]
    full = h if n <= 4096 else (np.concatenate([h, t[-(n - 4096):]]) if n <= 6144 else None)
    if full is not None:
        return full if n <= n_head + n_tail else np.concatenate([full[:n_head], full[n - n_tail:]] if n_tail else [full[:n_head]])
    return np.concatenate([h[:n_head], t[-n_tail:]] if n_tail else [h[:n_head]])

if os.path.exists(CACHE6):
    D6 = pickle.load(open(CACHE6, "rb")); print("loaded cache from Drive")
else:
    local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                              allow_patterns=["Phase_6_Summ/*", "*/Phase_6_Summ/*"], token=HF_TOKEN)
    def find(name):
        for r, _, fs in os.walk(local):
            if name in fs and "Phase_6" in r: return os.path.join(r, name)
    tr = [r for r in json.load(open(find("train.json"))) if r["summary"].strip()]
    te = json.load(open(find("test.json")))
    def enc(texts, bs=128):
        out = []
        for i in range(0, len(texts), bs):
            out += [e.ids for e in tokenizer.encode_batch([t[:400000] for t in texts[i:i + bs]])]
        return out
    def pack(ids): return dict(n=len(ids), head=np.array(ids[:4096], np.uint16), tail=np.array(ids[-2048:], np.uint16))
    tr_docs, tr_sums = enc([r["document"] for r in tr]), enc([r["summary"] for r in tr])
    te_docs = enc([r["document"] for r in te])
    recs = [dict(id=r["id"], **pack(d), s_len=len(s), summ=np.array(s[:1024], np.uint16)) for r, d, s in zip(tr, tr_docs, tr_sums)]
    perm = np.random.RandomState(0).permutation(len(recs))
    D6 = dict(val=[recs[i] for i in perm[:300]], train=[recs[i] for i in perm[300:]],
              test=[dict(id=r["id"], **pack(d)) for r, d in zip(te, te_docs)])
    pickle.dump(D6, open(CACHE6, "wb")); print("saved cache to Drive")

print("train:", len(D6["train"]), "| val:", len(D6["val"]), "| test:", len(D6["test"]), "| EOS id:", EOS6)
r = D6["train"][0]
print("view check: head-only 1536 ->", len(doc_view(r, 1536)), "| head 1024 + tail 512 ->", len(doc_view(r, 1024, 512)), "| doc length", r["n"])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

saved cache to Drive
train: 6728 | val: 300 | test: 100 | EOS id: 0
view check: head-only 1536 -> 1536 | head 1024 + tail 512 -> 1536 | doc length 3103


**Code cell 62. What this cell does.** Tokenizes the whole Phase-6 dataset once, builds a compact cache of document and summary token ids, and splits off a validation set.

**Line by line**
- `CACHE6`: cache file on Drive. `EOS6` is the end-of-text id.
- `doc_view(rec, n_head, n_tail)`: returns a view of a stored document: the first `n_head` tokens plus, optionally, the last `n_tail`. The cache keeps the first 4,096 and last 2,048 tokens; for documents up to 6,144 tokens the two parts overlap or touch and are joined to rebuild the full document so views can be cut anywhere; for longer ones the head and tail are used directly.
- If the cache exists it is loaded. Otherwise: download the data; drop training records with empty summaries; `enc` tokenizes the first 400,000 characters of each text in batches of 128; `pack` stores the length, first 4,096 ids and last 2,048 ids as `uint16`. Training records also store the summary length and the first 1,024 summary tokens.
- A fixed permutation (seed 0) puts the first 300 training records into `val` and the rest into `train`; the test records are packed too. Everything is saved to `sum_tok_cache.pkl`.
- The cell prints the split sizes and a check of two views of the first training document.

**Why a cache.** Tokenizing thousands of long documents takes a long time; with the cache every later run (five different models) starts immediately.

In [ ]:
import math, os, torch, torch.nn as nn, torch.nn.functional as F
KIND = "gqa"          # <-- set per notebook: gqa | strided | mla | encdec | student
assert "RRModel" in globals() and "tokenizer" in globals() and "device" in globals(), "run setup + Phase-3 toolkit first"
P1 = DRIVE_PROJECT_DIR                                    # Phase-1 folder (from the setup cell)
STAGE1_CKPT = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_2/student_kd_stage1/best.pt"

def _load(lm, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd = ck.get("model", ck.get("model_state_dict", ck)) if isinstance(ck, dict) else ck
    lm.load_state_dict({k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in sd.items()})
    return lm

def build_lm(kind):
    V = tokenizer.get_vocab_size()
    if kind == "gqa":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048, tie_weights=True),
                     f"{P1}/gqa_checkpoints/gqa_best.pt"), 768
    if kind == "strided":
        pat = alternating_patterns(10, [(256, 32), (256, 128)])
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True, layer_patterns=pat), f"{P1}/alt_stride_32_128_checkpoints/best.pt"), 768
    if kind == "mla":
        return _load(CustomDecoderLM(vocab_size=V, d_model=768, n_layer=10, n_head=12, max_seq_len=4096, qk_rope_head_dim=64,
                                     tie_weights=True), f"{P1}/DeepSeek/deepseek_mla_85m_pretrained.pt"), 768
    if kind == "encdec":
        return _load(EncoderDecoderLM(vocab_size=V, d_model=640, n_enc_layer=6, n_dec_layer=6, n_head=10, n_kv_head=5,
                                      max_src_len=1024, max_tgt_len=320, bos_id=tokenizer.token_to_id("<|endoftext|>"),
                                      tie_weights=True), f"{P1}/encdec_6x6_checkpoints/best.pt"), 640
    if kind == "student":
        return _load(CustomDecoderLM(vocab_size=V, d_model=576, n_layer=7, n_head=12, n_kv_head=4, max_seq_len=2048,
                                     tie_weights=True), STAGE1_CKPT), 576
    raise ValueError(kind)

class Backbone(RRModel):
    """Phase-1 LM + a features() method. Task heads are attached by each phase (model.head = ...)."""
    def __init__(self, lm, d, kind):
        super().__init__(lm, d, 2, 0.1); del self.head
        self.kind, self.d = kind, d
    def features(self, ids, lens):                       # ids (chunks, T) right-padded, lens (chunks,) -> (chunks, 2d)
        if self.kind == "encdec":                        # encoder takes no mask: one chunk at a time
            out = []
            for j in range(ids.shape[0]):
                n = int(lens[j]); H = self.lm.encode(ids[j:j + 1, :n])[0].float()
                out.append(torch.cat([H[n - 1], H.mean(0)], -1))
            return torch.stack(out)
        H = self.hidden(ids); B, T, _ = H.shape
        mask = (torch.arange(T, device=H.device)[None] < lens[:, None]).unsqueeze(-1)
        return torch.cat([H[torch.arange(B, device=H.device), lens - 1].float(), (H.float() * mask).sum(1) / lens[:, None]], -1)

def make_backbone(kind=None):
    lm, d = build_lm(kind or KIND); return Backbone(lm, d, kind or KIND).to(device)

# ---------- adapters (own implementations) ----------
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x): return self.base(x) + self.scale * F.linear(F.linear(x, self.A), self.B)

class LoRMAPlusLinear(nn.Module):
    """W = ((alpha/r) B A + I) W0 on the output side. Identity at init, mergeable."""
    def __init__(self, base, r, alpha):
        super().__init__(); self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
    def forward(self, x):
        y = self.base(x); return y + self.scale * F.linear(F.linear(y, self.A), self.B)

TARGETS = {k: ("q_proj", "k_proj", "v_proj", "out_proj") for k in ["gqa", "strided", "encdec", "student"]}
TARGETS["mla"] = ("q_up_proj", "kv_down_proj", "kv_up_proj", "out_proj")
def layers_of(bb): return bb.lm.enc_layers if bb.kind == "encdec" else bb.lm.layers

def inject(bb, method, r, alpha):
    """Freeze the Phase-1 LM and add adapters. Heads attached outside bb.lm stay trainable."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    cls = {"lora": LoRALinear, "lorma": LoRMAPlusLinear}[method]
    for layer in layers_of(bb):
        for nm in TARGETS[bb.kind]:
            base = getattr(layer.attn, nm); setattr(layer.attn, nm, cls(base, r, alpha).to(base.weight.device))
    return bb

def unfreeze_top(bb, k):
    """Partial (layer-wise) fine-tuning: only the top k layers (+ final norm for decoders) train."""
    for p in bb.lm.parameters(): p.requires_grad_(False)
    for layer in list(layers_of(bb))[-k:]:
        for p in layer.parameters(): p.requires_grad_(True)
    if bb.kind != "encdec":
        for p in bb.lm.norm.parameters(): p.requires_grad_(True)
    return bb

def n_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad), sum(p.numel() for p in m.parameters())

# ---------- distillation losses ----------
def kd_multiclass(s, t, T=2.0): return F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
def kd_multilabel(s, t, T=2.0): return F.binary_cross_entropy_with_logits(s / T, torch.sigmoid(t / T)) * T * T

# ---------- self-test: shapes + adapters are the identity at init ----------
CORE_SELFTEST = True
if CORE_SELFTEST:
    ids = torch.randint(0, tokenizer.get_vocab_size(), (2, 64), device=device); lens = torch.tensor([50, 64], device=device)
    for method in ["lorma", "lora"]:
        bb = make_backbone().eval()
        with torch.no_grad(): f0 = bb.features(ids, lens)
        inject(bb, method, 8, 16)
        with torch.no_grad(): f1 = bb.features(ids, lens)
        assert f0.shape == (2, 2 * bb.d) and (f0 - f1).abs().max().item() < 1e-3, f"{method}: not identity at init"
        tr, tot = n_params(bb); print(f"[{KIND}] {method}: identity OK | trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M (adapters only, no head)")
        del bb; torch.cuda.empty_cache()
    bb = unfreeze_top(make_backbone(), 4); tr, tot = n_params(bb); print(f"[{KIND}] partial top-4: trainable {tr/1e6:.2f}M of {tot/1e6:.1f}M"); del bb
    print(f"✅ core ready for KIND={KIND}")

[gqa] lorma: identity OK | trainable 0.33M of 87.8M (adapters only, no head)
[gqa] lora: identity OK | trainable 0.41M of 87.9M (adapters only, no head)
[gqa] partial top-4: trainable 25.17M of 87.5M
✅ core ready for KIND=gqa


**Code cell 63. What this cell does.** The shared "core" of Phases 4-7: builds any of the five Phase-1 models as a feature extractor, provides the adapters and the distillation losses, and runs a self-test. In each notebook only `KIND` changes.

**Line by line**
- `KIND` selects the model: `gqa`, `strided`, `mla`, `encdec` or `student`. The assert checks the earlier cells were run. `P1` is the Phase-1 folder; `STAGE1_CKPT` is the student's checkpoint from Phase 2.
- `_load(lm, path)` loads a checkpoint whatever its format: it accepts files that wrap the weights under `model` or `model_state_dict`, and removes prefixes (`_orig_mod.`, `module.`) added by `torch.compile` or data parallelism.
- `build_lm(kind)` builds the architecture for each Phase-1 model and loads its weights: GQA (768 wide, 10 layers, 12 heads, 4 KV heads, 2,048 context); strided (the same but with `layer_patterns=alternating_patterns(10, [(256, 32), (256, 128)])`, i.e. alternating sparse attention patterns); MLA (4,096 context, rotary dimension 64, latent KV projections); encoder-decoder (640 wide, 6 + 6 layers, 1,024 source and 320 target positions); and the 43M student (576 wide, 7 layers). It returns the model and its width.
- `Backbone(RRModel)`: the Phase-1 model with the head removed and a `features(ids, lens)` method that returns, for each chunk, the state at its last real token and the mean over its real tokens (width 2d). For the encoder-decoder, only the encoder is used and it processes one chunk at a time because it takes no padding mask.
- `make_backbone(kind)` builds and moves it to the device.
- **Adapters.** `LoRALinear` (`W0 x + (alpha/r) B A x`) and `LoRMAPlusLinear` (`y + (alpha/r) B A y` with `y = W0 x`, the multiplicative form), both identity at initialisation because `B` is zero. `TARGETS` lists which projections get adapters: `q_proj, k_proj, v_proj, out_proj`, and for MLA `q_up_proj, kv_down_proj, kv_up_proj, out_proj`. `layers_of` returns the encoder layers for the encoder-decoder, otherwise the decoder layers.
- `inject(bb, method, r, alpha)` freezes the language model and replaces the target projections by adapter-wrapped ones. `unfreeze_top(bb, k)` implements partial fine-tuning: freeze everything, then unfreeze the top `k` layers and the final norm.
- `n_params` returns the trainable and total parameter counts.
- `kd_multiclass` (softmax KL times T^2) and `kd_multilabel` (BCE against softened sigmoid targets times T^2) are the distillation losses for single-label and multi-label tasks.
- **Self-test.** Runs two random sequences through the backbone, injects each adapter type, asserts the features do not change (difference below 1e-3) and prints the trainable parameter counts, then prints the count for the top-4 partial setting. A failure here means the adapters are not exactly the identity at the start.

In [ ]:
for n in ["tokenizer", "device", "HF_DATASET_REPO", "HF_TOKEN", "RRModel", "build_lm", "make_backbone", "Backbone", "inject", "KIND"]:
    print("✅" if n in globals() else "❌ MISSING", n)

✅ tokenizer
✅ device
✅ HF_DATASET_REPO
✅ HF_TOKEN
✅ RRModel
✅ build_lm
✅ make_backbone
✅ Backbone
✅ inject
✅ KIND


**Code cell 64. What this cell does.** A quick check that the core objects exist (`tokenizer`, `device`, `RRModel`, `build_lm`, `make_backbone`, `Backbone`, `inject`, `KIND`, ...). Each name prints a tick or `MISSING`; use it after a restart to see what to re-run.

In [ ]:
import inspect, torch
bb = make_backbone(); lm = bb.lm; cls = type(lm)
print("class:", cls.__name__, "| KIND:", KIND)
print("public methods:", [m for m in dir(cls) if not m.startswith("_") and callable(getattr(cls, m))])
for nm in ["forward", "generate", "encode", "decode"]:
    f = getattr(cls, nm, None)
    if f is None: continue
    print(f"\n{nm}{inspect.signature(f)}\n  names used: {f.__code__.co_names}")
ids = torch.randint(0, 100, (1, 16), device=device)
with torch.no_grad():
    try:
        o = lm(ids); parts = o if isinstance(o, tuple) else (o,)
        print("\nlm(ids) ->", type(o).__name__, [getattr(t, "shape", type(t).__name__) for t in parts])
        if isinstance(o, tuple) and len(o) > 1:
            print("2nd output:", type(o[1]).__name__, "len", len(o[1]) if hasattr(o[1], "__len__") else "-")
            o2 = lm(ids[:, -1:], past_key_values=o[1])
            print("cached step ->", [getattr(t, "shape", type(t).__name__) for t in (o2 if isinstance(o2, tuple) else (o2,))])
    except Exception as e:
        print("forward probe failed:", repr(e)[:400])

class: CustomDecoderLM | KIND: gqa
public methods: ['add_module', 'apply', 'bfloat16', 'buffers', 'children', 'compile', 'cpu', 'cuda', 'double', 'eval', 'extra_repr', 'float', 'forward', 'get_buffer', 'get_extra_state', 'get_parameter', 'get_submodule', 'half', 'ipu', 'load_state_dict', 'modules', 'mtia', 'named_buffers', 'named_children', 'named_modules', 'named_parameters', 'parameters', 'register_backward_hook', 'register_buffer', 'register_forward_hook', 'register_forward_pre_hook', 'register_full_backward_hook', 'register_full_backward_pre_hook', 'register_load_state_dict_post_hook', 'register_load_state_dict_pre_hook', 'register_module', 'register_parameter', 'register_state_dict_post_hook', 'register_state_dict_pre_hook', 'requires_grad_', 'set_extra_state', 'set_submodule', 'share_memory', 'state_dict', 'to', 'to_empty', 'train', 'type', 'xpu', 'zero_grad']

forward(self, input_ids, labels=None, past_key_values=None)
  names used: ('shape', 'token_embedding', 'enumerate', 'lay

**Code cell 65. What this cell does.** Inspects the model class (a probe used while writing the code, to see which methods and outputs it has).

**Line by line.** Builds the backbone, prints the class name and its public methods, then for `forward`, `generate`, `encode` and `decode` prints the signature and the names the function uses. It runs a random 16-token input through the model and prints the output type and shapes; if the output has a cache, it runs one more cached step with the last token to confirm that cached generation works.

**Why.** The five Phase-1 models have slightly different interfaces; this probe confirmed what each returns before the generation code relied on it.

In [ ]:
import torch, torch.nn.functional as F
bb = make_backbone().eval(); lm = bb.lm
V = tokenizer.get_vocab_size(); ids = torch.randint(0, V, (1, 40), device=device)
pick_lg = lambda o: next(v for v in o.values() if torch.is_tensor(v) and v.dim() == 3)
with torch.no_grad():
    o = lm(ids); print("keys:", {k: (tuple(v.shape) if torch.is_tensor(v) else type(v).__name__) for k, v in o.items()})
    full = pick_lg(o)
    pk_key = next((k for k, v in o.items() if "past" in k or "cache" in k or isinstance(v, (tuple, list))), None)
    print("cache key:", pk_key)
    o1 = lm(ids[:, :39]); o2 = lm(ids[:, 39:40], past_key_values=o1[pk_key])
    print("cached step vs full forward, max logit diff: %.2e" % (pick_lg(o2)[:, -1] - full[:, 39]).abs().max().item())
    lab = ids.clone(); lab[:, :20] = -100
    L = lm(ids, labels=lab); loss = next(v for k, v in L.items() if "loss" in k)
    lg = pick_lg(L)
    man = F.cross_entropy(lg[:, :-1].reshape(-1, V).float(), lab[:, 1:].reshape(-1), ignore_index=-100)
    print("model loss %.4f | manual shifted CE %.4f" % (loss.item(), man.item()))

keys: {'loss': 'NoneType', 'logits': (1, 40, 32000), 'past_key_values': 'tuple'}
cache key: past_key_values
cached step vs full forward, max logit diff: 1.72e-05
model loss 12.9054 | manual shifted CE 12.9054


**Code cell 66. What this cell does.** Verifies, before any training, that the language model's cache and loss work as the summarisation code assumes.

**Line by line**
- Builds the backbone and a random 40-token input. `pick_lg` finds the 3-dimensional logits tensor inside the model's output dictionary.
- Prints the keys and shapes of the model output and finds the key that holds the cache (`past_key_values`).
- Runs the first 39 tokens, then feeds the 40th token using the cache, and prints the largest difference between that logit and the one from the full 40-token pass. It should be tiny (the earlier output shows about 1.7e-5); if not, generation would silently drift.
- Builds labels with the first 20 positions masked as -100, computes the model's own loss, and recomputes it by hand (shift by one, cross-entropy, ignore -100). The two numbers must agree: this proves that masking the document part (loss only on the summary) works.

In [ ]:
import os, time, json, numpy as np, torch
try: from rouge_score import rouge_scorer
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rouge_score"]); from rouge_score import rouge_scorer
assert KIND != "encdec", "decoder cell: enc-dec gets its own"
MAXLEN = 4096 if KIND == "mla" else 2048
CFG = dict(doc_head=2048, doc_tail=1024, sum_max=960) if KIND == "mla" else dict(doc_head=1024, doc_tail=480, sum_max=512)
BS, ACC = (4, 2) if KIND == "mla" else (8, 1)
SEP = tokenizer.encode("\n\nSUMMARY:\n").ids
assert len(SEP) + CFG["doc_head"] + CFG["doc_tail"] + CFG["sum_max"] + 1 <= MAXLEN
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
R6 = globals().get("R6", {})

def make_prompt(r): return doc_view(r, CFG["doc_head"], CFG["doc_tail"]).astype(np.int64).tolist() + SEP
def make_ex(r):
    s = r["summ"][:CFG["sum_max"]].astype(np.int64).tolist()
    if r["s_len"] <= CFG["sum_max"]: s.append(EOS6)          # teach stopping only when the whole summary fit
    return make_prompt(r), s
def batchify(exs):
    T = max(len(p) + len(s) for p, s in exs)
    ids = torch.zeros(len(exs), T, dtype=torch.long); lab = torch.full((len(exs), T), -100, dtype=torch.long)
    for i, (p, s) in enumerate(exs):
        x = p + s; ids[i, :len(x)] = torch.tensor(x); lab[i, len(p):len(x)] = torch.tensor(s)
    return ids.to(device), lab.to(device)

@torch.no_grad()
def val_loss(bb, n=None):
    bb.eval(); vs = D6["val"][:n] if n else D6["val"]; out = []
    for i in range(0, len(vs), BS):
        ids, lab = batchify([make_ex(r) for r in vs[i:i + BS]])
        with torch.autocast("cuda", dtype=torch.bfloat16): out.append(bb.lm(ids, labels=lab)["loss"].item())
    bb.train(); return float(np.mean(out))

@torch.no_grad()
def generate(bb, prompt, max_new, nrep=4):
    lm = bb.lm.eval(); out, seen = [], {}
    with torch.autocast("cuda", dtype=torch.bfloat16):
        o = lm(torch.tensor([prompt], device=device)); pk = o["past_key_values"]; lg = o["logits"][0, -1].float()
        for _ in range(max_new):
            key = tuple(out[-(nrep - 1):]) if nrep and len(out) >= nrep - 1 else None
            if key is not None and key in seen: lg[list(seen[key])] = -float("inf")
            t = int(lg.argmax())
            if t == EOS6: break
            if key is not None: seen.setdefault(key, set()).add(t)
            out.append(t)
            o = lm(torch.tensor([[t]], device=device), past_key_values=pk); pk = o["past_key_values"]; lg = o["logits"][0, -1].float()
    return out

def rouge_eval(bb, n=40, max_new=None, show=0):
    r1, rl, wl = [], [], []
    for j, r in enumerate(D6["val"][:n]):
        hyp = tokenizer.decode(generate(bb, make_prompt(r), max_new or CFG["sum_max"]))
        ref = tokenizer.decode(r["summ"].astype(np.int64).tolist())          # reference truncated at 1024 tokens (proxy)
        sc = scorer.score(ref, hyp); r1.append(sc["rouge1"].fmeasure); rl.append(sc["rougeL"].fmeasure); wl.append(len(hyp.split()))
        if j < show: print(f"\n--- val doc {j} ---\nHYP: {hyp[:600]}\nREF: {ref[:600]}")
    return dict(rouge1=float(np.mean(r1)), rougeL=float(np.mean(rl)), words=float(np.mean(wl)))

def train_sum(bb, name, lr, epochs=3, wd=0.01, patience=3, debug=False):
    params = [p for p in bb.parameters() if p.requires_grad]; assert params, "nothing trainable"
    tr = D6["train"][:64] if debug else D6["train"]; epochs = 1 if debug else epochs
    steps = epochs * (len(tr) // (BS * ACC)); warm = max(1, int(0.06 * steps)); ev = max(1, steps // (2 * epochs))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    path = f"{P6_DIR}/sm_{name}_best.pt"; best, bad, step, run, t0, stop = 1e9, 0, 0, [], time.time(), False
    ntr, ntot = n_params(bb); print(f"[{name}] trainable {ntr/1e6:.2f}M / {ntot/1e6:.1f}M | steps {steps} | eval every {ev}")
    torch.cuda.reset_peak_memory_stats(); bb.train()
    for ep in range(epochs):
        order = np.random.RandomState(ep).permutation(len(tr)); k = 0
        while k + BS * ACC <= len(tr) and step < steps and not stop:
            for _ in range(ACC):
                ids, lab = batchify([make_ex(tr[i]) for i in order[k:k + BS]]); k += BS
                with torch.autocast("cuda", dtype=torch.bfloat16): loss = bb.lm(ids, labels=lab)["loss"] / ACC
                loss.backward(); run.append(loss.item() * ACC)
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % ev == 0 or step == steps:
                vl = val_loss(bb, 16 if debug else None)
                print(f"step {step}/{steps} | train {np.mean(run):.3f} | val {vl:.3f} | {(time.time()-t0)/60:.1f} min"); run = []
                if vl < best:
                    best, bad = vl, 0; torch.save({n: p.detach().cpu() for n, p in bb.named_parameters() if p.requires_grad}, path)
                else:
                    bad += 1; stop = bad >= patience
    bb.load_state_dict(torch.load(path), strict=False)
    m = rouge_eval(bb, n=6 if debug else 40, show=1 if debug else 0)
    R6[name] = dict(val_loss=best, **m, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best val loss {best:.3f} | ROUGE-1 {m['rouge1']:.4f} | ROUGE-L {m['rougeL']:.4f} | avg words {m['words']:.0f} | peak {R6[name]['peak_gb']:.1f} GB")
    return bb

def write_pred_sum(bb, name, max_new=None):
    preds = []; t0 = time.time()
    for i, r in enumerate(D6["test"]):
        h = tokenizer.decode(generate(bb, make_prompt(r), max_new or CFG["sum_max"])).strip()
        preds.append(dict(id=r["id"], summary=h or tokenizer.decode(doc_view(r, 150).astype(np.int64).tolist())))
        if (i + 1) % 25 == 0: print(f"{i+1}/100 | {(time.time()-t0)/60:.1f} min")
    assert len(preds) == 100 and len({p["id"] for p in preds}) == 100
    p = f"{P6_DIR}/prediction_{name}.json"; json.dump(preds, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")
print("✅ summarisation toolkit ready | KIND =", KIND, "| context", MAXLEN, "|", CFG, "| BS", BS, "x ACC", ACC)

✅ summarisation toolkit ready | KIND = gqa | context 2048 | {'doc_head': 1024, 'doc_tail': 480, 'sum_max': 512} | BS 8 x ACC 1


**Code cell 67. What this cell does.** The summarisation toolkit for the decoder models: prompt construction, batching, validation loss, greedy generation, a ROUGE check, the training loop and the test-prediction writer.

**Settings.** `rouge_score` is installed if missing. `MAXLEN` is the model's context (4,096 for MLA, otherwise 2,048). `CFG` sets how much of the document is shown (`doc_head`, `doc_tail`) and the longest summary (`sum_max`): 1,024 + 480 document tokens and 512 summary tokens for the 2,048-token models, and 2,048 + 1,024 and 960 for MLA. `BS, ACC` are the batch size and gradient accumulation (4 x 2 for MLA, 8 x 1 otherwise). `SEP` is the token sequence of `"\n\nSUMMARY:\n"`. An assert checks that separator + document view + summary + end token fit in the context. `scorer` is the ROUGE-1/ROUGE-L scorer with stemming. `R6` collects the results of all runs.

**Prompt and examples**
- `make_prompt(r)` = document view (first `doc_head` + last `doc_tail` tokens) + `SEP`. So the model reads the beginning and the end of a judgment, then the cue "SUMMARY:".
- `make_ex(r)` returns the prompt and the target: the first `sum_max` summary tokens, plus the end-of-text token only if the whole summary fit (so the model learns to stop only on complete summaries).
- `batchify` concatenates prompt + target, pads to the longest, and sets the labels to -100 on the prompt, so the loss is computed on the summary only.

**Validation and generation**
- `val_loss` averages the loss over the validation set (or the first `n` records).
- `generate(bb, prompt, max_new, nrep=4)`: greedy decoding with the KV cache. It runs the prompt once, then repeatedly picks the highest-scoring token, with a **4-gram block**: if the last three generated tokens have already been followed by token `t` earlier, `t` is forbidden, which prevents loops of repeated text. It stops at the end-of-text token.
- `rouge_eval(bb, n=40)`: generates summaries for the first 40 validation documents and scores ROUGE-1 and ROUGE-L against the reference (truncated at 1,024 tokens, so only a proxy), plus the average length in words. The official BERTScore needs a pretrained model, so it is not computed here.

**Training `train_sum`**: AdamW over the trainable parameters, 6% warm-up and linear decay, evaluation twice per epoch on the 300 validation documents, saving the best trainable parameters, early stopping after `patience` evaluations without improvement, then reloading the best, running the ROUGE check and storing everything in `R6` (validation loss, ROUGE, trainable parameters in millions, peak memory, time).

**`write_pred_sum`** generates a summary for each of the 100 test documents, falls back to the start of the document if a summary is empty, checks the ids are unique and writes `prediction_<name>.json`.

In [ ]:
bb = make_backbone(); inject(bb, "lorma", 64, 128); bb = train_sum(bb, "gqa_lorma", lr=5e-4)
write_pred_sum(bb, "gqa_lorma")

[gqa_lorma] trainable 2.62M / 90.1M | steps 2523 | eval every 420
step 420/2523 | train 1.912 | val 1.847 | 2.6 min
step 840/2523 | train 1.852 | val 1.826 | 5.3 min
step 1260/2523 | train 1.789 | val 1.813 | 7.9 min
step 1680/2523 | train 1.786 | val 1.803 | 10.6 min
step 2100/2523 | train 1.725 | val 1.802 | 13.2 min
step 2520/2523 | train 1.733 | val 1.797 | 15.9 min
step 2523/2523 | train 1.636 | val 1.797 | 16.0 min

✅ [gqa_lorma] best val loss 1.797 | ROUGE-1 0.4914 | ROUGE-L 0.2768 | avg words 383 | peak 36.0 GB
25/100 | 3.2 min
50/100 | 6.4 min
75/100 | 9.8 min
100/100 | 13.3 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_lorma.json -> rename to prediction.json, zip, upload


**Code cell 68. What this cell does.** Summarisation experiment with LoRMA+ on the GQA model: rank 64, alpha 128, learning rate 5e-4.

**Line by line.** `make_backbone()` loads the pretrained GQA model; `inject(bb, "lorma", 64, 128)` freezes it and adds the multiplicative adapters to the attention projections; `train_sum` trains only those adapters (3 epochs by default); `write_pred_sum` generates the 100 test summaries. LoRMA+ is the required technique.

In [ ]:
for L in (128, 256, 512): print(L, rouge_eval(bb, n=40, max_new=L))
del bb; torch.cuda.empty_cache()

128 {'rouge1': 0.2606377245474073, 'rougeL': 0.17509594870287432, 'words': 113.25}
256 {'rouge1': 0.3949517891030153, 'rougeL': 0.2401593941372842, 'words': 223.1}
512 {'rouge1': 0.4913611402824115, 'rougeL': 0.27683663003628356, 'words': 383.425}


**Code cell 69. What this cell does.** Tests how the maximum generated length affects ROUGE.

**Line by line.** For maximum lengths of 128, 256 and 512 tokens it runs `rouge_eval` on 40 validation documents with the trained LoRMA+ model and prints the results; then it deletes the model to free GPU memory. Longer outputs raise recall but can lower precision, so this picks the length used for the submission.

In [ ]:
bb = make_backbone(); bb = train_sum(bb, "gqa_full", lr=5e-5, epochs=2)
write_pred_sum(bb, "gqa_full"); del bb; torch.cuda.empty_cache()

[gqa_full] trainable 87.51M / 87.5M | steps 1682 | eval every 420
step 420/1682 | train 1.879 | val 1.808 | 2.7 min
step 840/1682 | train 1.803 | val 1.768 | 5.4 min
step 1260/1682 | train 1.581 | val 1.765 | 8.1 min
step 1680/1682 | train 1.577 | val 1.760 | 10.8 min
step 1682/1682 | train 1.673 | val 1.760 | 11.0 min

✅ [gqa_full] best val loss 1.760 | ROUGE-1 0.5006 | ROUGE-L 0.2784 | avg words 377 | peak 38.8 GB
25/100 | 2.6 min
50/100 | 5.3 min
75/100 | 7.9 min
100/100 | 10.6 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_full.json -> rename to prediction.json, zip, upload


**Code cell 70. What this cell does.** Full fine-tuning baseline for summarisation: all GQA parameters train (learning rate 5e-5, 2 epochs), then test predictions are written and the model is freed.

In [ ]:
bb = make_backbone(); inject(bb, "lora", 64, 128); bb = train_sum(bb, "gqa_lora", lr=5e-4, epochs=2)
write_pred_sum(bb, "gqa_lora"); del bb; torch.cuda.empty_cache()

[gqa_lora] trainable 3.28M / 90.8M | steps 1682 | eval every 420
step 420/1682 | train 1.901 | val 1.835 | 2.7 min
step 840/1682 | train 1.838 | val 1.808 | 5.4 min
step 1260/1682 | train 1.758 | val 1.798 | 8.1 min
step 1680/1682 | train 1.752 | val 1.792 | 10.8 min
step 1682/1682 | train 1.884 | val 1.792 | 10.9 min

✅ [gqa_lora] best val loss 1.792 | ROUGE-1 0.4950 | ROUGE-L 0.2674 | avg words 378 | peak 36.0 GB
25/100 | 3.6 min
50/100 | 7.3 min
75/100 | 10.9 min
100/100 | 14.6 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_lora.json -> rename to prediction.json, zip, upload


**Code cell 71. What this cell does.** LoRA on the GQA model for summarisation (rank 64, alpha 128, learning rate 5e-4, 2 epochs) and its test predictions. This is the additive adapter, compared with LoRMA+.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
class AdapterLinear(nn.Module):
    def __init__(self, base, m):
        super().__init__(); self.base = base
        self.down = nn.Linear(base.out_features, m); self.up = nn.Linear(m, base.out_features)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): y = self.base(x); return y + self.up(F.gelu(self.down(y)))
def inject_houlsby(bb, m=128):
    for p in bb.lm.parameters(): p.requires_grad_(False)
    for layer in layers_of(bb):
        for owner, nm in ((layer.attn, "out_proj"), (layer.ffn, "w2")):
            base = getattr(owner, nm); setattr(owner, nm, AdapterLinear(base, m).to(base.weight.device))
    return bb

bb = make_backbone(); inject_houlsby(bb); bb = train_sum(bb, "gqa_houlsby", lr=5e-4, epochs=2)
write_pred_sum(bb, "gqa_houlsby"); del bb; torch.cuda.empty_cache()

[gqa_houlsby] trainable 3.95M / 91.5M | steps 1682 | eval every 420
step 420/1682 | train 1.900 | val 1.837 | 2.5 min
step 840/1682 | train 1.839 | val 1.808 | 5.0 min
step 1260/1682 | train 1.767 | val 1.798 | 7.6 min
step 1680/1682 | train 1.760 | val 1.791 | 10.1 min
step 1682/1682 | train 1.892 | val 1.791 | 10.2 min

✅ [gqa_houlsby] best val loss 1.791 | ROUGE-1 0.4791 | ROUGE-L 0.2577 | avg words 395 | peak 33.1 GB
25/100 | 3.1 min
50/100 | 6.3 min
75/100 | 9.5 min
100/100 | 12.9 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_houlsby.json -> rename to prediction.json, zip, upload


**Code cell 72. What this cell does.** A third kind of adapter, the bottleneck ("Houlsby-style") adapter, for summarisation.

**`AdapterLinear(base, m)`**: wraps a linear layer; its output is `y + up(gelu(down(y)))` where `y` is the original output, `down` shrinks the width to `m = 128`, `up` expands it back, and the `up` weights and bias start at zero so the module is the identity at the start.

**`inject_houlsby(bb, m)`**: freezes the language model and wraps the attention output projection and the feed-forward output projection (`w2`) of every layer.

The rest trains it (learning rate 5e-4, 2 epochs), writes test predictions and frees memory.

In [ ]:
def load_teacher(name, method=None):
    t = make_backbone("gqa")
    if method: inject(t, method, 64, 128)
    t.load_state_dict(torch.load(f"{P6_DIR}/sm_{name}_best.pt"), strict=False)
    t.eval()
    for p in t.parameters(): p.requires_grad_(False)
    return t

def train_kd(student, teacher, name, lr=1e-4, alpha=0.5, T=2.0, epochs=2, wd=0.01, patience=3, debug=False):
    params = [p for p in student.parameters() if p.requires_grad]
    tr = D6["train"][:64] if debug else D6["train"]; epochs = 1 if debug else epochs
    steps = epochs * (len(tr) // (BS * ACC)); warm = max(1, int(0.06 * steps)); ev = max(1, steps // (2 * epochs))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    path = f"{P6_DIR}/sm_{name}_best.pt"; best, bad, step, run, t0, stop = 1e9, 0, 0, [], time.time(), False
    ntr, ntot = n_params(student); print(f"[{name}] student trainable {ntr/1e6:.1f}M | steps {steps} | eval every {ev}")
    torch.cuda.reset_peak_memory_stats(); student.train()
    for ep in range(epochs):
        order = np.random.RandomState(ep).permutation(len(tr)); k = 0
        while k + BS * ACC <= len(tr) and step < steps and not stop:
            for _ in range(ACC):
                ids, lab = batchify([make_ex(tr[i]) for i in order[k:k + BS]]); k += BS
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    with torch.no_grad(): tl = teacher.lm(ids)["logits"][:, :-1]
                    sl = student.lm(ids)["logits"][:, :-1]
                m = lab[:, 1:] != -100
                s, t = sl[m].float(), tl[m].float()
                ce = F.cross_entropy(s, lab[:, 1:][m])
                kd = F.kl_div(F.log_softmax(s / T, -1), F.softmax(t / T, -1), reduction="batchmean") * T * T
                ((1 - alpha) * ce + alpha * kd).div(ACC).backward(); run.append(ce.item())
            torch.nn.utils.clip_grad_norm_(params, 1.0); opt.step(); sched.step(); opt.zero_grad(set_to_none=True); step += 1
            if step % ev == 0 or step == steps:
                vl = val_loss(student, 16 if debug else None)
                print(f"step {step}/{steps} | train CE {np.mean(run):.3f} | val CE {vl:.3f} | {(time.time()-t0)/60:.1f} min"); run = []
                if vl < best:
                    best, bad = vl, 0; torch.save({n: p.detach().cpu() for n, p in student.named_parameters() if p.requires_grad}, path)
                else:
                    bad += 1; stop = bad >= patience
    student.load_state_dict(torch.load(path), strict=False)
    mt = rouge_eval(student, n=6 if debug else 40)
    R6[name] = dict(val_loss=best, **mt, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best val CE {best:.3f} | ROUGE-1 {mt['rouge1']:.4f} | ROUGE-L {mt['rougeL']:.4f} | avg words {mt['words']:.0f}")
    return student

**Code cell 73. What this cell does.** Knowledge distillation for summarisation.

**`load_teacher(name, method)`**: builds the GQA backbone, optionally injects the same adapters as in the saved run, loads the saved trainable parameters, switches to eval mode and freezes everything.

**`train_kd(student, teacher, name, ...)`** has the same structure as `train_sum`, with these differences. For each batch the teacher produces next-token logits (no gradient) and the student produces its own; both are restricted to the summary positions (`lab != -100`). The loss is `(1 - alpha) * CE + alpha * T^2 * KL(softmax(student/T) || softmax(teacher/T))` with `alpha = 0.5`, `T = 2` and learning rate 1e-4. Validation tracks cross-entropy, the best parameters are saved, and the ROUGE check is run at the end.

In [ ]:
teacher = load_teacher("gqa_full")                     # or: load_teacher("gqa_lorma", "lorma")
st = make_backbone("student"); train_kd(st, teacher, "dbg_kd", debug=False)     # smoke test first

[dbg_kd] student trainable 43.2M | steps 1682 | eval every 420
step 420/1682 | train CE 2.245 | val CE 2.119 | 2.8 min
step 840/1682 | train CE 2.091 | val CE 2.054 | 5.7 min
step 1260/1682 | train CE 1.882 | val CE 2.028 | 8.5 min
step 1680/1682 | train CE 1.880 | val CE 2.015 | 11.3 min
step 1682/1682 | train CE 2.004 | val CE 2.015 | 11.4 min

✅ [dbg_kd] best val CE 2.015 | ROUGE-1 0.4695 | ROUGE-L 0.2537 | avg words 386


Backbone(
  (lm): CustomDecoderLM(
    (token_embedding): Embedding(32000, 576)
    (layers): ModuleList(
      (0-6): 7 x TransformerBlock(
        (attention_norm): RMSNorm()
        (attn): GroupedQueryAttention(
          (q_proj): Linear(in_features=576, out_features=576, bias=False)
          (k_proj): Linear(in_features=576, out_features=192, bias=False)
          (v_proj): Linear(in_features=576, out_features=192, bias=False)
          (out_proj): Linear(in_features=576, out_features=576, bias=False)
        )
        (ffn_norm): RMSNorm()
        (ffn): SwiGLU(
          (w1): Linear(in_features=576, out_features=1536, bias=False)
          (w2): Linear(in_features=1536, out_features=576, bias=False)
          (w3): Linear(in_features=576, out_features=1536, bias=False)
        )
      )
    )
    (norm): RMSNorm()
    (lm_head): Linear(in_features=576, out_features=32000, bias=False)
  )
  (drop): Dropout(p=0.1, inplace=False)
)

**Code cell 74. What this cell does.** Loads the fully fine-tuned GQA model as teacher (the comment shows how to load an adapter-trained teacher instead) and runs a smoke test of the distillation loop with the student (the 43M model). The name `dbg_kd` marks it as a test.

In [ ]:
st = make_backbone("student"); st = train_kd(st, teacher, "gqa_student_kd")
write_pred_sum(st, "gqa_student_kd"); del st; torch.cuda.empty_cache()

[gqa_student_kd] student trainable 43.2M | steps 1682 | eval every 420
step 420/1682 | train CE 2.245 | val CE 2.119 | 2.8 min
step 840/1682 | train CE 2.091 | val CE 2.054 | 5.7 min
step 1260/1682 | train CE 1.882 | val CE 2.028 | 8.5 min
step 1680/1682 | train CE 1.880 | val CE 2.015 | 11.3 min
step 1682/1682 | train CE 2.004 | val CE 2.015 | 11.4 min

✅ [gqa_student_kd] best val CE 2.015 | ROUGE-1 0.4695 | ROUGE-L 0.2537 | avg words 386
25/100 | 1.8 min
50/100 | 3.6 min
75/100 | 5.3 min
100/100 | 7.2 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_6/prediction_gqa_student_kd.json -> rename to prediction.json, zip, upload


**Code cell 75. What this cell does.** The real distillation run: the 43M student learns from the fine-tuned GQA teacher; its test summaries are written, and memory is freed.

In [ ]:
st = make_backbone("student"); st = train_sum(st, "gqa_student_ce", lr=1e-4, epochs=2)
del st; torch.cuda.empty_cache()

[gqa_student_ce] trainable 43.21M / 43.2M | steps 1682 | eval every 420
step 420/1682 | train 2.275 | val 2.146 | 1.8 min
step 840/1682 | train 2.130 | val 2.072 | 3.6 min
step 1260/1682 | train 1.887 | val 2.053 | 5.5 min
step 1680/1682 | train 1.876 | val 2.037 | 7.3 min
step 1682/1682 | train 1.991 | val 2.037 | 7.4 min

✅ [gqa_student_ce] best val loss 2.037 | ROUGE-1 0.4670 | ROUGE-L 0.2557 | avg words 379 | peak 28.2 GB


**Code cell 76. What this cell does.** The no-teacher baseline: the same student trained on the summaries with ordinary cross-entropy (learning rate 1e-4, 2 epochs). Comparing it with the distilled student shows how much the teacher helps.

In [ ]:
import pandas as pd; print(pd.DataFrame(R6).T.round(4))

                val_loss  rouge1  rougeL     words  trainable_M  peak_gb  \
dbg               2.1043  0.3441  0.1821  382.6667       2.6214  34.8546   
gqa_lorma         1.7972  0.4914  0.2768  383.4250       2.6214  36.0286   
gqa_full          1.7598  0.5006  0.2784  376.9750      87.5067  38.8286   
gqa_lora          1.7922  0.4950  0.2674  378.5000       3.2768  35.9728   
gqa_houlsby       1.7909  0.4791  0.2577  395.1250       3.9501  33.1306   
dbg_kd            2.5305  0.4357  0.2238  437.5000      43.2132  29.4048   
gqa_student_kd    2.0147  0.4695  0.2537  385.5000      43.2132  29.6433   
gqa_student_ce    2.0366  0.4670  0.2557  378.5750      43.2132  28.2494   

                minutes  
dbg              1.0646  
gqa_lorma       21.6329  
gqa_full        15.0769  
gqa_lora        16.6248  
gqa_houlsby     15.5720  
dbg_kd           0.5968  
gqa_student_kd  14.4899  
gqa_student_ce  10.3655  


**Code cell 77. What this cell does.** Prints the results dictionary `R6` as a table (one row per run: validation loss, ROUGE-1, ROUGE-L, average words, trainable millions, peak memory, minutes), rounded to four decimals. These numbers go into the Phase-6 report tables.

## Prior Case Retrieval

In [ ]:
import json, os
from huggingface_hub import snapshot_download
P7_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_7"; os.makedirs(P7_DIR, exist_ok=True)
local = snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset",
                          allow_patterns=["Phase_7_PCR/*", "*/Phase_7_PCR/*"], token=HF_TOKEN)
def find(name):
    for r, _, fs in os.walk(local):
        if name in fs and "Phase_7" in r: return os.path.join(r, name)
files = {n: find(n + ".json") for n in ["train", "train_candidates", "dev", "dev_candidates", "test", "test_candidates"]}
if not any(files.values()): raise SystemExit("Phase 7 files are not in the repo yet")
def short(x, n=100):
    if isinstance(x, str): return x[:n] + (f"... [{len(x)} chars]" if len(x) > n else "")
    if isinstance(x, list): return f"list[{len(x)}] " + str([short(i, 50) for i in x[:2]])
    if isinstance(x, dict): return {k: short(v, 50) for k, v in list(x.items())[:4]}
    return x
D = {}
for k, p in files.items():
    if not p: print(f"\n== {k}: NOT FOUND"); continue
    D[k] = json.load(open(p)); d = D[k]
    print(f"\n== {k}: {os.path.getsize(p)/1e6:.1f} MB | {type(d).__name__} with {len(d)} items")
    ex = d[0] if isinstance(d, list) else next(iter(d.items()))
    print("   first item:", {kk: short(v) for kk, v in ex.items()} if isinstance(ex, dict) else short(ex))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]


== train: 33.2 MB | dict with 827 items
   first item: ('0000000599', {'text': ['PETITIONER: <NAME> Vs. RESPONDENT: ROOP SINGH RATHORE & OTHERS(and connected appeal) DATE OF JUDGMENT: 07/05/1963 BENCH: <NAME>, S.K. BENCH: <NAME>, S.K. SINHA, BHUVNESHWAR P.(CJ) DAYAL, RAGHUBAR AYYANGAR, <NAME> MUDHOLKAR, J.R. CITATION: 1964 AIR 1545 1964 SCR (3) 573 CITATOR INFO : R 1970 SC 765 (9) R 1971 SC 342 (6) RF 1974 SC1185 (16) F 1984 SC 305 (9,21) R 1991 SC1557 (20,22) ACT: Election Dispute-Joinder of -parties-Joinder of candi- date who did not contest-If invalidates eletion petition- "Copy", meaning of-Defects in verification and affidavit- Maintainability of petition- Representation of the People Act , 1951 , ss. 81 , 82 , 83 , 90 . HEADNOTE: The validity of the election of the appellant to <ORG> at the third general elections held in the month of February, 1962, was challenged by two of the electors of the constituency from which the appellant was elected, by filing election petitions for s

**Code cell 78. What this cell does.** Downloads the Phase-7 data (prior case retrieval, PCR) and prints the structure of each of the six files.

**Task.** Every query is a case judgment; for each there is a pool of candidate cases, and the model must return the prior cases the query cites. Score: Micro-F1 at 7 (top 7 returned per query).

**Line by line.** Downloads `Phase_7_PCR/*`. `find` locates each of `train`, `train_candidates`, `dev`, `dev_candidates`, `test`, `test_candidates`; if none exist, the cell stops. `short` abbreviates long strings and lists for printing. For every file it loads the JSON into `D[name]`, prints its size in MB, number of items and the first item with long values shortened, so you can see the fields (`text` as a list of sentences, `relevant_candidates` for the gold ids).

In [ ]:
import numpy as np, random
assert "D" in globals(), "re-run the Phase 7 inspection cell first"
print("split sizes:", {k: len(v) for k, v in D.items()})
for sp in ["train", "dev"]:
    q, c = D[sp], D[sp + "_candidates"]; pool = set(c)
    g = np.array([len(r["relevant_candidates"]) for r in q.values()]); allg = [x for r in q.values() for x in r["relevant_candidates"]]
    print(f"\n[{sp}] gold per query: mean {g.mean():.2f} | median {np.median(g):.0f} | min {g.min()} | p90 {np.percentile(g, 90):.0f} | max {g.max()}")
    print(f"  gold ids found in own pool: {100*np.mean([x in pool for x in allg]):.1f}% | pool docs gold for >=1 query: {100*len(set(allg) & pool)/len(pool):.1f}% | gold links per gold doc: {len(allg)/max(1, len(set(allg))):.2f}")
    print(f"  query ids that also appear in their own pool: {sum(k in pool for k in q)} of {len(q)}")
    hits = 7 * g / len(pool); P, R = hits.sum() / (7 * len(g)), hits.sum() / g.sum()
    print(f"  random-ranking Micro-F1@7 ≈ {2*P*R/(P+R):.4f} (P {P:.4f}, R {R:.4f})")
Pl = {sp: set(D[sp + "_candidates"]) for sp in ["train", "dev", "test"]}
print("\npool overlap train&dev:", len(Pl["train"] & Pl["dev"]), "| train&test:", len(Pl["train"] & Pl["test"]), "| dev&test:", len(Pl["dev"] & Pl["test"]))
print("query overlap train&dev:", len(set(D["train"]) & set(D["dev"])), "| train&test:", len(set(D["train"]) & set(D["test"])), "| dev&test:", len(set(D["dev"]) & set(D["test"])))
rng = random.Random(0); tot = 0
for name, d in [("train query", D["train"]), ("train cand", D["train_candidates"]), ("dev query", D["dev"]), ("dev cand", D["dev_candidates"]), ("test query", D["test"]), ("test cand", D["test_candidates"])]:
    recs = rng.sample(list(d.values()), min(150, len(d)))
    t = np.array([len(e.ids) for e in tokenizer.encode_batch([" ".join(r["text"])[:400000] for r in recs])]); tot += t.mean() * len(d)
    hdr = np.mean([r["text"][0].lstrip().startswith("PETITIONER") for r in d.values()])
    print(f"{name:12s} tokens p10/50/90/max: {int(np.percentile(t,10))}/{int(np.median(t))}/{int(np.percentile(t,90))}/{t.max()} | sentences (median) {int(np.median([len(r['text']) for r in recs]))} | start with 'PETITIONER': {100*hdr:.0f}%")
print(f"\nestimated total tokens across all six files: {tot/1e6:.0f} M")

split sizes: {'train': 827, 'train_candidates': 4320, 'dev': 118, 'dev_candidates': 1023, 'test': 237, 'test_candidates': 1727}

[train] gold per query: mean 6.42 | median 4 | min 0 | p90 14 | max 52
  gold ids found in own pool: 100.0% | pool docs gold for >=1 query: 95.1% | gold links per gold doc: 1.29
  query ids that also appear in their own pool: 827 of 827
  random-ranking Micro-F1@7 ≈ 0.0016 (P 0.0015, R 0.0016)

[dev] gold per query: mean 8.25 | median 5 | min 1 | p90 17 | max 92
  gold ids found in own pool: 100.0% | pool docs gold for >=1 query: 89.9% | gold links per gold doc: 1.06
  query ids that also appear in their own pool: 118 of 118
  random-ranking Micro-F1@7 ≈ 0.0074 (P 0.0081, R 0.0068)

pool overlap train&dev: 436 | train&test: 740 | dev&test: 220
query overlap train&dev: 0 | train&test: 0 | dev&test: 0
train query  tokens p10/50/90/max: 1899/5863/17328/61879 | sentences (median) 39 | start with 'PETITIONER': 34%
train cand   tokens p10/50/90/max: 1222/4100/13402

**Code cell 79. What this cell does.** Computes the statistics of the retrieval data that drove the design.

**For train and dev**
- Gold per query: mean, median, minimum, 90th percentile and maximum number of relevant candidates.
- Whether the gold ids are found in the query's own pool (they are, 100%), the share of pool documents that are gold for at least one query, and how many queries a gold document serves on average.
- How many queries also appear in their own pool (all of them), which means a query must be removed from its own ranking.
- A random-ranking baseline: returning 7 random pool documents gives an expected number of hits `7 * gold / pool size`; from it, precision, recall and Micro-F1@7 are computed (about 0.0016 on train and 0.0074 on dev). This is the floor any method must beat.

**Overlap checks.** Pool overlap between splits (some documents appear in several splits' pools) and query overlap (none), which shows there is no leakage of queries across splits.

**Length statistics.** For 150 random documents from each of the six files it prints token percentiles, the median number of sentences and the share beginning with the word PETITIONER, and it extrapolates the total number of tokens in all six files (about 64 million). This number is what makes caching essential.

In [ ]:
import os, pickle, time, numpy as np
CACHE7 = f"{P7_DIR}/pcr_tok_cache.pkl"
if os.path.exists(CACHE7):
    C7 = pickle.load(open(CACHE7, "rb")); print("loaded cache from Drive")
else:
    assert "D" in globals(), "re-run the Phase 7 inspection cell first"
    t0 = time.time(); C7 = {}
    for name, d in D.items():
        keys = list(d.keys()); out = {}
        for i in range(0, len(keys), 32):
            batch = keys[i:i + 32]
            for k, e in zip(batch, tokenizer.encode_batch([" ".join(d[k]["text"]) for k in batch])):
                out[k] = np.array(e.ids, dtype=np.uint16)
        C7[name] = out; print(f"{name:17s} {len(out):5d} docs | {sum(len(v) for v in out.values())/1e6:5.1f} M tokens | {(time.time()-t0)/60:.1f} min")
    C7["gold"] = {sp: {k: r["relevant_candidates"] for k, r in D[sp].items()} for sp in ["train", "dev"]}
    pickle.dump(C7, open(CACHE7, "wb")); print("saved", CACHE7)
for sp in ["train", "dev", "test"]:
    pool, q = C7[sp + "_candidates"], C7[sp]
    msg = f"[{sp}] queries inside their own pool: {sum(k in pool for k in q)}/{len(q)}"
    if sp != "test": msg += f" | queries listing themselves as gold: {sum(k in v for k, v in C7['gold'][sp].items())}"
    print(msg)
print("train queries with zero gold:", sum(len(v) == 0 for v in C7["gold"]["train"].values()))

train               827 docs |   7.1 M tokens | 0.1 min
train_candidates   4320 docs |  34.2 M tokens | 0.7 min
dev                 118 docs |   1.2 M tokens | 0.7 min
dev_candidates     1023 docs |  11.0 M tokens | 1.0 min
test                237 docs |   2.0 M tokens | 1.0 min
test_candidates    1727 docs |  15.1 M tokens | 1.3 min
saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/pcr_tok_cache.pkl
[train] queries inside their own pool: 827/827 | queries listing themselves as gold: 0
[dev] queries inside their own pool: 118/118 | queries listing themselves as gold: 0
[test] queries inside their own pool: 237/237
train queries with zero gold: 10


**Code cell 80. What this cell does.** Tokenizes all Phase-7 documents once and caches them on Drive.

**Line by line**
- If `pcr_tok_cache.pkl` exists it is loaded into `C7`. Otherwise each of the six files is tokenized: for each document the sentences are joined with spaces, cut to 400,000 characters, tokenized in groups of 32, and stored as a `uint16` array under its id. A progress line prints the number of documents, millions of tokens and elapsed minutes. The gold lists for train and dev are stored too under `C7["gold"]`. The result is saved.
- Sanity checks for each split: how many queries are inside their own pool, and for train and dev how many queries list themselves as a gold match (this should be none); and the number of training queries with no gold candidate (a few; they carry no training signal).

In [ ]:
import time, numpy as np, torch, torch.nn.functional as F

@torch.no_grad()
def embed_split(bb, tok, ids=None, chunk=1024, bs=32):
    """Return {doc_id: (n_chunks, 2d) array}: one [last-token ; mean-token] vector per 1024-token chunk."""
    bb.eval(); ids = list(ids or tok.keys()); pieces = []
    for di, k in enumerate(ids):
        a = tok[k]
        for s in range(0, len(a), chunk): pieces.append((di, a[s:s + chunk]))
    order = sorted(range(len(pieces)), key=lambda i: len(pieces[i][1]))        # similar lengths together = little padding
    out = [None] * len(pieces)
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]; lens = [len(pieces[i][1]) for i in idx]; T = max(lens)
        x = torch.zeros(len(idx), T, dtype=torch.long)
        for j, i in enumerate(idx): x[j, :lens[j]] = torch.from_numpy(pieces[i][1].astype(np.int64))
        with torch.autocast("cuda", dtype=torch.bfloat16):
            f = bb.features(x.to(device), torch.tensor(lens, device=device))
        f = f.float().cpu().numpy()
        for j, i in enumerate(idx): out[i] = f[j]
    per_doc = {k: [] for k in ids}
    for (di, _), v in zip(pieces, out): per_doc[ids[di]].append(v)
    return {k: np.stack(v) for k, v in per_doc.items()}

def doc_matrix(ch, ids, part, pool, d):
    sl = slice(0, d) if part == "last" else slice(d, 2 * d)
    rows = []
    for k in ids:
        m = ch[k][:, sl]; rows.append(m.mean(0) if pool == "mean" else m.max(0) if pool == "max" else m[0])
    return np.stack(rows)

def f1_at_k(Qm, Cm, qids, cids, gold, center=False, k=7):
    Q, C = torch.tensor(Qm, device=device), torch.tensor(Cm, device=device)
    if center: mu = C.mean(0, keepdim=True); Q, C = Q - mu, C - mu
    S = F.normalize(Q, dim=-1) @ F.normalize(C, dim=-1).T
    pos = {c: i for i, c in enumerate(cids)}
    for qi, q in enumerate(qids):
        if q in pos: S[qi, pos[q]] = -1e9                      # a query must never retrieve itself
    top = S.topk(k, dim=1).indices.cpu().numpy()
    tp = sum(len({cids[j] for j in top[qi]} & set(gold[q])) for qi, q in enumerate(qids))
    P, R = tp / (k * len(qids)), tp / sum(len(gold[q]) for q in qids)
    return (2 * P * R / (P + R) if tp else 0.0), P, R

# ---- zero-shot baseline on dev ----
bb = make_backbone(); d = bb.d; t0 = time.time()
chq, chc = embed_split(bb, C7["dev"]), embed_split(bb, C7["dev_candidates"])
print(f"embedded dev queries + pool in {(time.time()-t0)/60:.1f} min")
qids, cids, gold = list(C7["dev"]), list(C7["dev_candidates"]), C7["gold"]["dev"]
print("random-ranking floor on dev is about 0.0074\n")
print(f"{'feature':8s} {'chunk pool':10s} {'centred':8s}   F1@7    P@7     R@7")
for part in ("mean", "last"):
    for pool in ("mean", "max", "first"):
        for center in (False, True):
            f, p, r = f1_at_k(doc_matrix(chq, qids, part, pool, d), doc_matrix(chc, cids, part, pool, d), qids, cids, gold, center)
            print(f"{part:8s} {pool:10s} {str(center):8s}  {f:.4f}  {p:.4f}  {r:.4f}")

embedded dev queries + pool in 1.0 min
random-ranking floor on dev is about 0.0074

feature  chunk pool centred    F1@7    P@7     R@7
mean     mean       False     0.2957  0.3220  0.2734
mean     mean       True      0.3068  0.3341  0.2837
mean     max        False     0.2123  0.2312  0.1963
mean     max        True      0.2312  0.2518  0.2138
mean     first      False     0.2135  0.2324  0.1973
mean     first      True      0.2101  0.2288  0.1942
last     mean       False     0.0867  0.0944  0.0802
last     mean       True      0.0867  0.0944  0.0802
last     max        False     0.0267  0.0291  0.0247
last     max        True      0.0278  0.0303  0.0257
last     first      False     0.0233  0.0254  0.0216
last     first      True      0.0200  0.0218  0.0185


**Code cell 81. What this cell does.** Retrieval building blocks and the zero-shot baseline: how good is the frozen Phase-1 model at finding related cases, without any training?

**`embed_split(bb, tok, ids, chunk=1024, bs=32)`**: cuts each document into 1,024-token chunks, sorts all chunks by length (so batches have little padding), runs them through the frozen model in batches of 32 under bfloat16 and gets the `[last token ; mean token]` vector (2d) of each chunk with `bb.features`. It returns, for each document, an array of shape (number of chunks, 2d).

**`doc_matrix(ch, ids, part, pool, d)`**: builds one vector per document. `part` chooses the half of the feature vector (`last`: the last-token state, `mean`: the token-mean state); `pool` chooses how to combine the chunks: mean over chunks, max over chunks, or the first chunk only.

**`f1_at_k(Qm, Cm, qids, cids, gold, center, k=7)`**: the evaluation. Optionally subtracts the mean candidate vector from all vectors (`center`), computes cosine similarity between each query and every candidate (normalise both and multiply), sets a query's similarity to itself to -1e9 so it can never retrieve itself, takes the top 7 per query, counts true positives against the gold lists, and returns Micro-F1, precision and recall at 7 (pooled over all queries).

**Baseline run.** On the dev set it embeds queries and pool, then prints a table for every combination: feature (`mean` or `last`), chunk pooling (`mean`, `max`, `first`) and centring (yes or no). In the earlier output the token-mean feature with mean pooling gave about 0.30 on dev, much higher than the last-token feature, so the token mean is used for retrieval.

**Why this matters.** It sets the baseline that fine-tuning must beat, and it chooses the feature used by the trained model.

In [ ]:
import pickle, os
HN_PATH = f"{P7_DIR}/hardneg_zs.pkl"
if os.path.exists(HN_PATH): HN = pickle.load(open(HN_PATH, "rb")); print("loaded hard negatives for", len(HN), "queries")
else:
    bb = make_backbone(); d = bb.d; t0 = time.time()
    chq, chc = embed_split(bb, C7["train"]), embed_split(bb, C7["train_candidates"])
    qids, cids, gold = list(C7["train"]), list(C7["train_candidates"]), C7["gold"]["train"]
    Qm, Cm = doc_matrix(chq, qids, "mean", "mean", d), doc_matrix(chc, cids, "mean", "mean", d)
    f, p, r = f1_at_k(Qm, Cm, qids, cids, gold, center=True)
    print(f"embedded train in {(time.time()-t0)/60:.1f} min | zero-shot train F1@7 {f:.4f}")
    Cc = torch.tensor(Cm, device=device); mu = Cc.mean(0, keepdim=True)
    S = F.normalize(torch.tensor(Qm, device=device) - mu, dim=-1) @ F.normalize(Cc - mu, dim=-1).T
    pos = {c: i for i, c in enumerate(cids)}; HN = {}
    for qi, q in enumerate(qids):
        if q in pos: S[qi, pos[q]] = -1e9
        for g in gold[q]: S[qi, pos[g]] = -1e9
        HN[q] = [cids[j] for j in S[qi].topk(50).indices.tolist()]
    pickle.dump(HN, open(HN_PATH, "wb")); print("saved", HN_PATH); del bb; torch.cuda.empty_cache()

embedded train in 3.3 min | zero-shot train F1@7 0.1846
saved /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/hardneg_zs.pkl


**Code cell 82. What this cell does.** Mines hard negatives for training: for each training query, the 50 candidates that the frozen model finds most similar but that are not relevant.

**Line by line**
- If `hardneg_zs.pkl` exists it is loaded into `HN`.
- Otherwise: embeds the train queries and pool with the frozen model, builds document vectors (mean feature, mean over chunks), and prints the zero-shot training F1@7 as a sanity check.
- Subtracts the mean candidate vector, normalises, and computes all query-candidate similarities in one matrix `S`.
- For each query it sets the similarity to itself and to every gold document to -1e9, so they cannot be picked, and stores the ids of the 50 most similar remaining candidates in `HN[q]`.
- Saves the dictionary and frees the model.

**Why hard negatives.** A random candidate is easy to tell apart from a relevant case. The model learns more from candidates that look relevant to the frozen model but are not; these are exactly the ones that the top-7 ranking has to separate. Training later uses the closest 30 of these 50.

In [ ]:
import random, json, torch.nn as nn
R7 = globals().get("R7", {}); HN = pickle.load(open(f"{P7_DIR}/hardneg_zs.pkl", "rb"))
TRQ = [q for q, g in C7["gold"]["train"].items() if g]; print("training queries:", len(TRQ))

class Proj(nn.Module):                                # identity at init: starts exactly at the zero-shot embedding
    def __init__(self, d):
        super().__init__(); self.lin = nn.Linear(d, d); nn.init.eye_(self.lin.weight); nn.init.zeros_(self.lin.bias)
    def forward(self, x): return self.lin(x)

def rand_chunk(a, L, rng):
    if len(a) <= L: return a
    s = rng.randint(0, len(a) - L); return a[s:s + L]

def feats(bb, proj, docs, L, K, rng):                 # differentiable: K random L-token windows per doc, mean-pooled
    chunks = [rand_chunk(a, L, rng) for a in docs for _ in range(K)]
    lens = [len(c) for c in chunks]; x = torch.zeros(len(chunks), max(lens), dtype=torch.long)
    for j, c in enumerate(chunks): x[j, :lens[j]] = torch.from_numpy(c.astype(np.int64))
    with torch.autocast("cuda", dtype=torch.bfloat16):
        f = bb.features(x.to(device), torch.tensor(lens, device=device))[:, bb.d:]
    return F.normalize(proj(f.float().view(len(docs), K, -1).mean(1)), dim=-1)

@torch.no_grad()
def embed_final(bb, proj, tok):
    ch = embed_split(bb, tok); ids = list(tok.keys())
    return ids, proj(torch.tensor(doc_matrix(ch, ids, "mean", "mean", bb.d), device=device)).cpu().numpy()

def eval_dev(bb, proj):
    bb.eval(); qi, Qm = embed_final(bb, proj, C7["dev"]); ci, Cm = embed_final(bb, proj, C7["dev_candidates"])
    return f1_at_k(Qm, Cm, qi, ci, C7["gold"]["dev"], center=False)

def train_pcr(bb, name, lr, steps=600, B=8, L=512, K=1, tau=0.05, hn_top=30, lr_head=1e-3, eval_every=200, debug=False):
    proj = Proj(bb.d).to(device); rng = random.Random(0); params = [p for p in bb.parameters() if p.requires_grad]
    ntr = sum(p.numel() for p in params) + sum(p.numel() for p in proj.parameters())
    opt = torch.optim.AdamW([dict(params=list(proj.parameters()), lr=lr_head)] + ([dict(params=params, lr=lr)] if params else []), weight_decay=0.01)
    steps, eval_every = (4, 4) if debug else (steps, eval_every); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    gold, pool, qtok = C7["gold"]["train"], C7["train_candidates"], C7["train"]
    best, t0, run, path = -1.0, time.time(), [], f"{P7_DIR}/pcr_{name}_best.pt"; bb.train(); torch.cuda.reset_peak_memory_stats()
    print(f"[{name}] trainable {ntr/1e6:.2f}M | steps {steps} | B {B} | window {L} x {K}")
    for step in range(1, steps + 1):
        qs = rng.sample(TRQ, B); pos_ids = [rng.choice(gold[q]) for q in qs]; hn_ids = [rng.choice(HN[q][:hn_top]) for q in qs]
        z = feats(bb, proj, [qtok[q] for q in qs] + [pool[p] for p in pos_ids] + [pool[h] for h in hn_ids], L, K, rng)
        logits = z[:B] @ z[B:].T / tau; cols = pos_ids + hn_ids
        mask = torch.zeros_like(logits, dtype=torch.bool)
        for i, q in enumerate(qs):
            gs = set(gold[q])
            for j, c in enumerate(cols):
                if j != i and c in gs: mask[i, j] = True          # also gold for this query: not a negative
        loss = F.cross_entropy(logits.masked_fill(mask, -1e4), torch.arange(B, device=device))
        loss.backward(); torch.nn.utils.clip_grad_norm_(params + list(proj.parameters()), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True); run.append(loss.item())
        if step % eval_every == 0 or step == steps:
            f, p, r = eval_dev(bb, proj); bb.train()
            print(f"step {step}/{steps} | loss {np.mean(run):.3f} | dev F1@7 {f:.4f} (P {p:.4f} R {r:.4f}) | {(time.time()-t0)/60:.1f} min"); run = []
            if f > best:
                best = f; torch.save(dict(bb={n: q.detach().cpu() for n, q in bb.named_parameters() if q.requires_grad}, proj=proj.state_dict()), path)
    ck = torch.load(path); bb.load_state_dict(ck["bb"], strict=False); proj.load_state_dict(ck["proj"])
    R7[name] = dict(dev_f1=best, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60)
    print(f"\n✅ [{name}] best dev F1@7 {best:.4f} | peak {R7[name]['peak_gb']:.1f} GB | {R7[name]['minutes']:.1f} min"); return bb, proj

@torch.no_grad()
def write_pred_pcr(bb, proj, name):
    bb.eval(); qi, Qm = embed_final(bb, proj, C7["test"]); ci, Cm = embed_final(bb, proj, C7["test_candidates"])
    S = F.normalize(torch.tensor(Qm, device=device), dim=-1) @ F.normalize(torch.tensor(Cm, device=device), dim=-1).T
    pos = {c: i for i, c in enumerate(ci)}
    for i, q in enumerate(qi):
        if q in pos: S[i, pos[q]] = -1e9                                  # never retrieve the query itself
    top = S.topk(7, dim=1).indices.cpu().numpy(); pred = {q: [ci[j] for j in top[i]] for i, q in enumerate(qi)}
    assert len(pred) == len(C7["test"]) and all(len(set(v)) == 7 and set(v) <= set(ci) and q not in v for q, v in pred.items())
    p = f"{P7_DIR}/prediction_pcr_{name}.json"; json.dump(pred, open(p, "w")); print("✅ wrote", p, "-> rename to prediction.json, zip, upload")
print("✅ retrieval toolkit ready")

training queries: 817
✅ retrieval toolkit ready


**Code cell 83. What this cell does.** The Phase-7 retrieval training toolkit: a learnable projection, the contrastive training loop, the dev evaluation and the test-prediction writer.

**Setup.** `R7` collects results; `HN` loads the hard negatives from the previous step; `TRQ` is the list of training queries that have at least one gold candidate (queries with none cannot be used).

**`Proj(d)`**: a linear layer initialised to the identity matrix with zero bias, so before any training the projected embedding equals the frozen model's embedding and the starting point is exactly the zero-shot ranking.

**`rand_chunk`, `feats`.** Training uses one random 512-token window of each document per step (`K` windows if more are wanted) instead of the whole document, to keep a step affordable. `feats` runs these windows through the backbone (with gradients), keeps the token-mean half of the feature vector, averages the `K` windows of a document, applies `Proj` and normalises to unit length (so a dot product is a cosine similarity).

**`embed_final` / `eval_dev`.** For evaluation (and test) all chunks of every document are embedded (no randomness), averaged, projected, and Micro-F1@7 is computed against the gold lists, without centring.

**`train_pcr(bb, name, lr, steps, B, L, K, tau, hn_top, lr_head, eval_every)`**
- Two learning rates: `lr_head` (1e-3) for the projection and `lr` for the backbone parameters that are trainable (full fine-tuning, adapters or top layers, depending on what was frozen before the call). AdamW, 6% warm-up, linear decay.
- Each step samples `B` queries, one random gold document per query (the positive) and one random hard negative per query (from its 30 nearest non-gold candidates). The `3B` documents (queries, positives, hard negatives) are embedded in one pass.
- **InfoNCE loss.** Scores are cosine similarities divided by the temperature `tau = 0.05`; row i holds query i against all `2B` candidates (the B positives and the B hard negatives of the batch). The correct answer for query i is its own positive in column i. All other columns are negatives: other queries' positives (in-batch negatives) and all hard negatives. A column that is also gold for query i is masked out (set to -1e4), so a true match is never punished. Cross-entropy over the row pushes the positive up and the negatives down.
- Every `eval_every` steps it evaluates on the dev set, prints loss, F1, precision, recall and time, and saves the backbone's trainable parameters and the projection when dev F1 improves. At the end it reloads the best and stores the results in `R7`.

**`write_pred_pcr`**: embeds test queries and test pool, computes cosine similarities, removes the query itself from its ranking, takes the top 7 for each query, asserts there are 7 distinct valid candidates for each of the test queries, and writes the dictionary `{query id: [7 candidate ids]}` as `prediction_pcr_<name>.json`.

In [ ]:
bb = make_backbone(); bb, proj = train_pcr(bb, "gqa_full_long", lr=2e-5, steps=1500, B=16, eval_every=250)
write_pred_pcr(bb, proj, "gqa_full_long")

[gqa_full_long] trainable 88.10M | steps 1500 | B 16 | window 512 x 1
step 250/1500 | loss 2.157 | dev F1@7 0.3280 (P 0.3571 R 0.3032) | 1.9 min
step 500/1500 | loss 1.821 | dev F1@7 0.3291 (P 0.3584 R 0.3042) | 3.9 min
step 750/1500 | loss 1.612 | dev F1@7 0.3391 (P 0.3692 R 0.3135) | 5.9 min
step 1000/1500 | loss 1.472 | dev F1@7 0.3480 (P 0.3789 R 0.3217) | 7.8 min
step 1250/1500 | loss 1.451 | dev F1@7 0.3457 (P 0.3765 R 0.3196) | 9.8 min
step 1500/1500 | loss 1.384 | dev F1@7 0.3457 (P 0.3765 R 0.3196) | 11.7 min

✅ [gqa_full_long] best dev F1@7 0.3480 | peak 22.0 GB | 11.7 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/prediction_pcr_gqa_full_long.json -> rename to prediction.json, zip, upload


**Code cell 84. What this cell does.** Full fine-tuning of the GQA model for retrieval, a longer run: learning rate 2e-5, 1,500 steps, 16 queries per step, evaluation every 250 steps. `train_pcr` returns the trained model and projection, and `write_pred_pcr` writes the test file `prediction_pcr_gqa_full_long.json`.

In [ ]:
bb = make_backbone(); bb, proj = train_pcr(bb, "gqa_full", lr=2e-5)
write_pred_pcr(bb, proj, "gqa_full")

[gqa_full] trainable 88.10M | steps 600 | B 8 | window 512 x 1
step 200/600 | loss 1.872 | dev F1@7 0.3013 (P 0.3281 R 0.2785) | 1.4 min
step 400/600 | loss 1.592 | dev F1@7 0.3146 (P 0.3426 R 0.2909) | 2.8 min
step 600/600 | loss 1.445 | dev F1@7 0.3224 (P 0.3511 R 0.2980) | 4.2 min

✅ [gqa_full] best dev F1@7 0.3224 | peak 11.8 GB | 4.3 min
✅ wrote /content/drive/MyDrive/CS781_Capstone_Project_Phase_7/prediction_pcr_gqa_full.json -> rename to prediction.json, zip, upload


**Code cell 85. What this cell does.** Full fine-tuning of the GQA model for retrieval with the default settings (600 steps, 8 queries per step, learning rate 2e-5, evaluation every 200 steps), and its test file `prediction_pcr_gqa_full.json`. This is the teacher for retrieval distillation.

In [ ]:
def enc_chunks(bb, proj, chunks):                      # differentiable: list of token arrays -> (n, d) unit vectors
    lens = [len(c) for c in chunks]; x = torch.zeros(len(chunks), max(lens), dtype=torch.long)
    for j, c in enumerate(chunks): x[j, :lens[j]] = torch.from_numpy(c.astype(np.int64))
    with torch.autocast("cuda", dtype=torch.bfloat16):
        f = bb.features(x.to(device), torch.tensor(lens, device=device))[:, bb.d:]
    return F.normalize(proj(f.float()), dim=-1)

def load_pcr(name, kind="gqa"):
    m = make_backbone(kind); ck = torch.load(f"{P7_DIR}/pcr_{name}_best.pt")
    m.load_state_dict(ck["bb"], strict=False); p = Proj(m.d).to(device); p.load_state_dict(ck["proj"]); m.eval()
    for q in list(m.parameters()) + list(p.parameters()): q.requires_grad_(False)
    return m, p

def train_kd_pcr(student, teacher, tproj, name, lr=5e-5, steps=1500, B=16, L=512, tau=0.05, alpha=0.5, hn_top=30,
                 lr_head=1e-3, eval_every=250, debug=False):
    sproj = Proj(student.d).to(device); rng = random.Random(0); params = [p for p in student.parameters() if p.requires_grad]
    ntr = sum(p.numel() for p in params) + sum(p.numel() for p in sproj.parameters())
    t1 = time.time(); ft, _, _ = eval_dev(teacher, tproj); t_inf = time.time() - t1
    print(f"teacher dev F1@7 {ft:.4f} | teacher dev embedding time {t_inf:.0f}s | student trainable {ntr/1e6:.1f}M")
    opt = torch.optim.AdamW([dict(params=list(sproj.parameters()), lr=lr_head), dict(params=params, lr=lr)], weight_decay=0.01)
    steps, eval_every = (4, 4) if debug else (steps, eval_every); warm = max(1, int(0.06 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (steps - s) / max(1, steps - warm))))
    gold, pool, qtok = C7["gold"]["train"], C7["train_candidates"], C7["train"]
    best, t0, run, path = -1.0, time.time(), [], f"{P7_DIR}/pcr_{name}_best.pt"; student.train(); torch.cuda.reset_peak_memory_stats()
    for step in range(1, steps + 1):
        qs = rng.sample(TRQ, B); pos_ids = [rng.choice(gold[q]) for q in qs]; hn_ids = [rng.choice(HN[q][:hn_top]) for q in qs]
        docs = [qtok[q] for q in qs] + [pool[p] for p in pos_ids] + [pool[h] for h in hn_ids]
        chunks = [rand_chunk(a, L, rng) for a in docs]                       # the same windows go to teacher and student
        zs = enc_chunks(student, sproj, chunks)
        with torch.no_grad(): zt = enc_chunks(teacher, tproj, chunks)
        ls, lt = zs[:B] @ zs[B:].T / tau, zt[:B] @ zt[B:].T / tau; cols = pos_ids + hn_ids
        mask = torch.zeros_like(ls, dtype=torch.bool)
        for i, q in enumerate(qs):
            gs = set(gold[q])
            for j, c in enumerate(cols):
                if j != i and c in gs: mask[i, j] = True
        ls, lt = ls.masked_fill(mask, -1e4), lt.masked_fill(mask, -1e4)
        ce = F.cross_entropy(ls, torch.arange(B, device=device))
        kd = F.kl_div(F.log_softmax(ls, -1), F.softmax(lt, -1), reduction="batchmean")
        loss = (1 - alpha) * ce + alpha * kd
        loss.backward(); torch.nn.utils.clip_grad_norm_(params + list(sproj.parameters()), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True); run.append((ce.item(), kd.item()))
        if step % eval_every == 0 or step == steps:
            f, p, r = eval_dev(student, sproj); student.train(); a = np.mean(run, 0)
            print(f"step {step}/{steps} | CE {a[0]:.3f} | KD {a[1]:.3f} | dev F1@7 {f:.4f} (P {p:.4f} R {r:.4f}) | {(time.time()-t0)/60:.1f} min"); run = []
            if f > best:
                best = f; torch.save(dict(bb={n: q.detach().cpu() for n, q in student.named_parameters() if q.requires_grad}, proj=sproj.state_dict()), path)
    ck = torch.load(path); student.load_state_dict(ck["bb"], strict=False); sproj.load_state_dict(ck["proj"])
    t1 = time.time(); eval_dev(student, sproj); s_inf = time.time() - t1
    R7[name] = dict(dev_f1=best, teacher_f1=ft, trainable_M=ntr / 1e6, peak_gb=torch.cuda.max_memory_allocated() / 1e9, minutes=(time.time() - t0) / 60, student_embed_s=s_inf, teacher_embed_s=t_inf)
    print(f"\n✅ [{name}] student dev F1@7 {best:.4f} vs teacher {ft:.4f} | dev embedding time: student {s_inf:.0f}s vs teacher {t_inf:.0f}s")
    return student, sproj
print("✅ KD toolkit ready")

✅ KD toolkit ready


**Code cell 86. What this cell does.** The knowledge-distillation toolkit for retrieval.

**`enc_chunks(bb, proj, chunks)`**: embeds a list of token windows (one vector per window, token-mean feature, projection, unit length), with gradients.

**`load_pcr(name, kind)`**: rebuilds a backbone, loads its saved trainable parameters and projection, freezes everything and sets eval mode; this is how the trained teacher is restored.

**`train_kd_pcr(student, teacher, tproj, name, ...)`**
- First the teacher is evaluated on the dev set (printing its F1@7 and the time it takes to embed the dev pool, for the cost comparison).
- The student has its own projection and the trainable backbone parameters, trained with AdamW (backbone 5e-5, projection 1e-3).
- Each step uses the same sampling as `train_pcr`, with one random window per document. **The same windows go to both models.** The student computes its similarity matrix (queries against positives and hard negatives, divided by the temperature 0.05) and the frozen teacher computes its own on the same windows; gold-for-this-query columns are masked in both.
- The loss is `0.5 * CE + 0.5 * KL`. `CE` is the usual InfoNCE of the student against the true answers; `KL` is the divergence between the student's softmax over the candidates and the teacher's softmax. So the student learns not only which candidate is right but how similar the teacher thinks the others are.
- Evaluation and checkpointing follow `train_pcr`. At the end it times the student's dev embedding, and stores in `R7` the student F1, the teacher F1, trainable millions, peak memory, training minutes and both embedding times, which give the speed-up of the smaller model.

In [ ]:
teacher, tproj = load_pcr("gqa_full"); st = make_backbone("student")
train_kd_pcr(st, teacher, tproj, "dbg_kd", debug=False); del st; torch.cuda.empty_cache()

teacher dev F1@7 0.3224 | teacher dev embedding time 60s | student trainable 43.5M
step 4/4 | CE 2.453 | KD 0.663 | dev F1@7 0.2991 (P 0.3257 R 0.2765) | 0.7 min

✅ [dbg_kd] student dev F1@7 0.2991 vs teacher 0.3224 | dev embedding time: student 40s vs teacher 60s


**Code cell 87. What this cell does.** Loads the trained GQA retrieval model as teacher and runs a smoke test of the distillation loop with the student. The name `dbg_kd` marks a test run (note that `debug=False` here means it ran the full loop, so a result file with that name may exist and can be ignored).

In [ ]:
st = make_backbone("student"); st, sproj = train_kd_pcr(st, teacher, tproj, "student_kd")
write_pred_pcr(st, sproj, "student_kd")

**Code cell 88. What this cell does.** The real distillation run for retrieval: the 43M student learns from the fine-tuned GQA teacher for 1,500 steps, and its test file `prediction_pcr_student_kd.json` is written.

In [ ]:
bb = make_backbone("student"); bb, p = train_pcr(bb, "student_ce", lr=5e-5, steps=1500, B=16, eval_every=250); del bb; torch.cuda.empty_cache()

**Code cell 89. What this cell does.** The no-teacher baseline for retrieval: the same student trained only with the contrastive loss (1,500 steps, 16 queries per step, learning rate 5e-5). Comparing it with the distilled student isolates the benefit of distillation.